<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.fr/cap05/cap05_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

# 5 Transformées et Compression

Dans les chapitres précédents, toutes les opérations ont été réalisées **dans le domaine spatial**, où les algorithmes agissent directement sur les valeurs d'intensité des pixels.

Ce chapitre présente une approche complémentaire : le **domaine fréquentiel**, dans lequel l'image est représentée par les variations spatiales d'intensité, et non seulement par les valeurs individuelles des pixels.

Le concept de **fréquence spatiale** décrit la rapidité avec laquelle l'intensité varie le long de l'image. Les variations lentes correspondent aux **basses fréquences**, tandis que les contours, les détails fins et les bruits correspondent aux **hautes fréquences**.

Cette représentation repose sur le fait que toute image numérique discrète peut être décomposée en une combinaison de **fonctions orthogonales**. La **Transformée de Fourier** utilise une **base d'exponentielles complexes bidimensionnelles** (équivalentes à des sinusoïdes avec une orientation et une fréquence spécifiques). D'autres transformées, comme la **Transformée en Cosinus (DCT)** et la **Transformée *Wavelet* (DWT)**, utilisent différentes familles de fonctions de base — des cosinus bidimensionnels dans le cas de la DCT, et des fonctions à support compact dans le cas des *wavelets*.

Parmi les principales applications de cette représentation, on distingue :

1. **Le filtrage dans le domaine fréquentiel**, pour atténuer ou rehausser certaines bandes de fréquences ;
2. **L'analyse multirésolution au moyen de transformées *wavelet***, qui représente les structures à différentes échelles ;
3. **La compression d'images**, par la réduction du nombre de coefficients nécessaires pour représenter l'image.

## 5.1 Objectifs

À la fin de ce chapitre, vous serez capable de :

- **Interpréter le spectre de Fourier** d’une image, en distinguant l’amplitude, la phase et les composantes de fréquence ;
- **Appliquer le théorème de convolution** pour effectuer un filtrage dans le domaine fréquentiel à l’aide de la transformée de Fourier rapide (FFT) ;
- **Concevoir et analyser des filtres dans le domaine fréquentiel**, en comprenant le fonctionnement des filtres passe-bas, passe-haut et *coupe-bande* ;
- **Comprendre l’analyse multirésolution par transformées en *ondelettes*** et son application à la représentation hiérarchique des images ;
- **Décrire le processus de compression d’images**, y compris la transformée en cosinus discrète (DCT) et la quantification des coefficients ;
- **Choisir des formats de stockage d’images**, tels que JPEG, PNG et WebP, en fonction des exigences de l’application.

## 5.2 Configuration de l'environnement

In [ ]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # artefacts de build du parcours C++

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# Noyau Python même dans le parcours C++. cpp=True télécharge morph.hpp + stb ; les
# cellules %%writefile *.cpp de ce chapitre compilent AVEC OpenCV
# (-DMM_USE_OPENCV + pkg-config opencv4).
import config
config.setup(cpp=True)
from morph import mm
import numpy as np

## 5.3 Transformada de Fourier Discreta 2D

L'analyse de Fourier repose sur le principe selon lequel tout signal périodique peut être représenté comme une somme de fonctions sinusoïdales de différentes fréquences, amplitudes et phases. Ce concept s'applique également aux images numériques, permettant de les représenter dans le **domaine fréquentiel** plutôt que dans le domaine spatial.

La [Figure 5.1](#fig-decomposicao-1d) illustre cette décomposition pour un signal unidimensionnel. Dans le cas d'une image, la Transformée de Fourier Discrète (TFD) convertit la matrice d'intensités $f(x,y)$ en un ensemble de coefficients décrivant la contribution des différentes fréquences spatiales présentes dans l'image.

In [ ]:
%%writefile tmp/fig_decomposicao_1d.cpp
#define MM_OUT "tmp/fig_decomposicao_1d.png"
//| label: fig-decomposicao-1d
//| fig-cap: "Decomposição de Fourier 1D: uma onda quadrada (linha tracejada) é aproximada pela soma das primeiras senoides (linhas coloridas). Quanto mais termos, melhor a aproximação."
//| echo: false
//| output: true

#include <cmath>
#include <vector>
#include <string>
#include <opencv2/opencv.hpp>
#include "morph.hpp"
#include <filesystem>

int main() {
    int n_points = 400;
    std::vector<double> x(n_points);
    for (int i = 0; i < n_points; ++i) {
        x[i] = (2.0 * M_PI) * i / (n_points - 1);
    }

    std::vector<double> square(n_points);
    for (int i = 0; i < n_points; ++i) {
        square[i] = (x[i] < M_PI) ? 1.0 : -1.0;
    }

    std::vector<double> soma(n_points, 0.0);
    std::vector<std::vector<double>> harms(3, std::vector<double>(n_points));
    for (int n = 1; n <= 3; ++n) {
        std::vector<double> h(n_points);
        for (int i = 0; i < n_points; ++i) {
            h[i] = (4.0 / M_PI) * (1.0 / (2 * n - 1)) * std::sin((2 * n - 1) * x[i]);
            soma[i] += h[i];
        }
        harms[n - 1] = h; // index 0-based
    }

    std::vector<std::vector<double>> ys = {square, harms[0], harms[1], harms[2], soma};
    std::vector<std::string> labels = {"Onda quadrada ideal", "1a harmonica", "3a harmonica", "5a harmonica", "Soma (3 primeiras)"};
    std::vector<cv::Scalar> colors = {
        cv::Scalar(40, 40, 40),
        cv::Scalar(60, 160, 80),
        cv::Scalar(180, 120, 60),
        cv::Scalar(150, 80, 160),
        cv::Scalar(60, 60, 220)
    };

    mm::Image chart = mm::lineChart(
        x, ys, colors, labels,
        "Sintese de Fourier: de senos a uma onda quadrada",
        "Posicao", "Intensidade"
    );

    mm::show(std::vector<mm::Image>{chart}, MM_OUT, std::vector<std::string>{"Decomposicao de Fourier 1D"}, 1);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(chart, "tmp/fig_decomposicao_1d_0.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_decomposicao_1d.cpp -o tmp/fig_decomposicao_1d -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_decomposicao_1d \
  && test -f "tmp/fig_decomposicao_1d.png" \
  || echo "⚠ mm::show não gravou tmp/fig_decomposicao_1d.png"

In [ ]:
mm.show(
    [
        mm.read("tmp/fig_decomposicao_1d_0.png"),
    ],
    titles=[
        'Decomposicao de Fourier 1D',
    ],
    cols=1,
)

**Figure 5.1:** Decomposição de Fourier 1D: uma onda quadrada (linha tracejada) é aproximada pela soma das primeiras senoides (linhas coloridas). Quanto mais termos, melhor a aproximação.


### 5.3.1 Simulateur : Reconstruire des signaux avec des sinusoïdes

Avant d'étudier les images bidimensionnelles, le simulateur de la [Figure 5.2](#fig-05-sim-05-freq) illustre le principe de l'analyse de Fourier pour des signaux unidimensionnels : **une forme d'onde peut être approchée par la somme de sinusoïdes de différentes fréquences et amplitudes**.

À mesure que de nouveaux termes sont ajoutés, la somme des sinusoïdes (courbe noire) se rapproche de la forme d'onde de référence (en pointillés). Le graphique inférieur présente le spectre d'amplitudes, indiquant la contribution de chaque fréquence à la reconstruction du signal.

> ### 💡 Activité
>
> Explorez le simulateur et répondez :
>
> 1. Combien de termes sont nécessaires pour obtenir une bonne approximation de l'onde carrée ?
> 2. Laquelle des trois formes d'onde converge le plus rapidement ? Justifiez votre réponse.
> 3. Comment le spectre d'amplitudes se modifie-t-il en remplaçant l'onde carrée par l'onde triangulaire ?

> ### 📝 Réponses
>
> **1. Combien de termes sont nécessaires pour une bonne approximation de l'onde carrée ?**
>
> Avec environ 15 à 20 termes, la forme de l'onde se rapproche déjà bien de la référence. Cependant, à proximité des discontinuités subsiste une petite oscillation, connue sous le nom de **phénomène de Gibbs**, qui ne disparaît pas même avec l'ajout de termes supplémentaires.
>
> **2. Quelle forme converge le plus rapidement ? Pourquoi ?**
>
> L'**onde triangulaire** converge plus rapidement, car les amplitudes de ses harmoniques décroissent plus vite que celles de l'onde carrée et de l'onde en dents de scie. Par conséquent, peu de termes suffisent déjà à produire une bonne approximation.
>
> **3. Comment le spectre change-t-il entre l'onde carrée et l'onde triangulaire ?**
>
> Toutes deux ne possèdent que des **harmoniques impairs**, mais, dans l'onde triangulaire, les amplitudes diminuent beaucoup plus rapidement. Ainsi, peu d'harmoniques suffisent pour reconstruire le signal avec une bonne précision.

In [ ]:
from IPython.display import HTML

HTML("""
<div id="sim-05-freq" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">
<style>
  #sim-05-freq * { box-sizing: border-box; }
  #sim-05-freq canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; }
  #sim-05-freq button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; }
  #sim-05-freq button:hover { background: #e8dfcf; }
  #sim-05-freq button.sim05fft_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim05fft_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim05fft_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim05fft_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; flex: 1; min-width: 90px; }
  .sim05fft_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim05fft_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">∿ Simulateur : Décomposition de Fourier 1D</span>
  <span class="sim05fft_pill">somme de sinusoïdes</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div style="display:flex; gap:10px; margin-bottom:14px; flex-wrap:wrap;">
    <div class="sim05fft_stat_box"><div class="sim05fft_stat_label">Termes</div><div id="sim05fft_nTerms" class="sim05fft_stat_value" style="color:#2980b9;">1</div></div>
    <div class="sim05fft_stat_box"><div class="sim05fft_stat_label">Erreur RMS</div><div id="sim05fft_rms" class="sim05fft_stat_value" style="color:#c0392b;">–</div></div>
    <div class="sim05fft_stat_box"><div class="sim05fft_stat_label">Forme cible</div><div id="sim05fft_target" class="sim05fft_stat_value" style="color:#27ae60; font-size:13px;">carrée</div></div>
  </div>

  <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:12px;text-align:center;">
    <canvas id="sim05fft_Canvas" width="660" height="200" style="margin:0 auto;"></canvas>
    <canvas id="sim05fft_SpecCanvas" width="660" height="80" style="margin:8px auto 0 auto;"></canvas>
  </div>

  <div style="display:flex; gap:12px; margin-top:12px; flex-wrap:wrap;">
    <div class="sim05fft_panel" style="flex:1; min-width:200px;">
      <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">Forme cible</div>
      <div style="display:flex; gap:6px; flex-wrap:wrap;">
        <button id="sim05fft_sq" class="sim05fft_active" onclick="sim05fft_setTarget('square')" style="flex:1; justify-content:center;">Carrée</button>
        <button id="sim05fft_tr" onclick="sim05fft_setTarget('triangle')" style="flex:1; justify-content:center;">Triangulaire</button>
        <button id="sim05fft_sw" onclick="sim05fft_setTarget('sawtooth')" style="flex:1; justify-content:center;">Dent de scie</button>
      </div>
    </div>
    
    <div class="sim05fft_panel" style="flex:1; min-width:180px;">
      <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">Nombre de termes</div>
      <div style="display:flex; align-items:center; gap:8px;">
        <input type="range" id="sim05fft_slider" min="1" max="25" value="1" style="flex:1; cursor:pointer; height:4px;">
        <span id="sim05fft_slVal" style="font-size:12px; font-family:monospace; font-weight:700; min-width:22px; color:#26241d;">1</span>
      </div>
    </div>

    <div class="sim05fft_panel" style="flex:1; min-width:140px;">
      <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">Affichage</div>
      <div style="display:flex; gap:6px;">
        <button id="sim05fft_chk_comp" class="sim05fft_active" onclick="sim05fft_toggleComp()" style="flex:1; justify-content:center;">Composantes</button>
        <button id="sim05fft_chk_sum" class="sim05fft_active" onclick="sim05fft_toggleSum()" style="flex:1; justify-content:center;">Somme</button>
      </div>
    </div>
  </div>

  <div id="sim05fft_termList" style="margin-top:12px; display:flex; gap:6px; flex-wrap:wrap; justify-content:center;"></div>

</div>
</div>

<script>
(function(){
  function initSim05FFT(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const C = root.querySelector('#sim05fft_Canvas');
    const S = root.querySelector('#sim05fft_SpecCanvas');
    const ctx = C.getContext('2d');
    const sctx = S.getContext('2d');
    const sim05fft_N = 512;
    let sim05fft_nTerms = 1, sim05fft_showComp = true, sim05fft_showSum = true, sim05fft_targetType = 'square';

    const sim05fft_PALETTE = ['#2980b9','#27ae60','#b9770e','#c0392b','#8e44ad','#16a085','#d35400','#2c3e50'];

    function sim05fft_getTerms(type, n) {
      const terms = [];
      for (let k = 1; k <= n; k++) {
        let freq, amp, phase = 0;
        if (type === 'square') {
          const m = 2*k - 1;
          freq = m; amp = (4/Math.PI) * (1/m);
        } else if (type === 'triangle') {
          const m = 2*k - 1;
          freq = m; amp = (8/Math.PI**2) * (1/m**2);
          phase = -Math.PI/2;
        } else {
          freq = k; amp = (2/Math.PI) * (1/k);
          phase = Math.PI;
        }
        terms.push({freq, amp, phase});
      }
      return terms;
    }

    function sim05fft_getTarget(type) {
      const t = new Float32Array(sim05fft_N);
      for (let i = 0; i < sim05fft_N; i++) {
        const x = i / sim05fft_N;
        if (type === 'square') t[i] = x < 0.5 ? 1 : -1;
        else if (type === 'triangle') t[i] = x < 0.5 ? (4*x - 1) : (3 - 4*x);
        else t[i] = 2*x - 1;
      }
      return t;
    }

    function sim05fft_evalTerms(terms) {
      const sig = new Float32Array(sim05fft_N);
      for (const {freq, amp, phase} of terms) {
        for (let i = 0; i < sim05fft_N; i++) {
          sig[i] += amp * Math.sin(2*Math.PI*freq*i/sim05fft_N + phase);
        }
      }
      return sig;
    }

    function sim05fft_draw() {
      const W = C.width, H = C.height;
      ctx.clearRect(0, 0, W, H);
      ctx.fillStyle = '#ffffff'; ctx.fillRect(0, 0, W, H);

      const terms = sim05fft_getTerms(sim05fft_targetType, sim05fft_nTerms);
      const sum = sim05fft_evalTerms(terms);
      const target = sim05fft_getTarget(sim05fft_targetType);
      let rms = 0;
      for (let i = 0; i < sim05fft_N; i++) rms += (sum[i]-target[i])**2;
      rms = Math.sqrt(rms/sim05fft_N);

      // Grid
      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth = 0.5; ctx.setLineDash([3,3]);
      ctx.beginPath(); ctx.moveTo(0,H/2); ctx.lineTo(W,H/2); ctx.stroke();
      ctx.setLineDash([]);

      // Componentes individuais
      if (sim05fft_showComp) {
        for (let k = 0; k < terms.length; k++) {
          const {freq, amp, phase} = terms[k];
          ctx.strokeStyle = sim05fft_PALETTE[k % sim05fft_PALETTE.length] + '55';
          ctx.lineWidth = 1;
          ctx.beginPath();
          for (let i = 0; i < sim05fft_N; i++) {
            const x = i * W / sim05fft_N;
            const y = H/2 - amp * Math.sin(2*Math.PI*freq*i/sim05fft_N + phase) * H/3;
            i === 0 ? ctx.moveTo(x,y) : ctx.lineTo(x,y);
          }
          ctx.stroke();
        }
      }

      // Sinal alvo
      ctx.strokeStyle = '#8a8371'; ctx.lineWidth = 1.5; ctx.setLineDash([4,4]);
      ctx.beginPath();
      for (let i = 0; i < sim05fft_N; i++) {
        const x = i * W / sim05fft_N;
        const y = H/2 - target[i] * H/3;
        i === 0 ? ctx.moveTo(x,y) : ctx.lineTo(x,y);
      }
      ctx.stroke(); ctx.setLineDash([]);

      // Soma
      if (sim05fft_showSum) {
        ctx.strokeStyle = '#26241d'; ctx.lineWidth = 2.5;
        ctx.beginPath();
        for (let i = 0; i < sim05fft_N; i++) {
          const x = i * W / sim05fft_N;
          const y = H/2 - sum[i] * H/3;
          i === 0 ? ctx.moveTo(x,y) : ctx.lineTo(x,y);
        }
        ctx.stroke();
      }

      // Espectro
      const SW = S.width, SH = S.height;
      sctx.clearRect(0, 0, SW, SH);
      sctx.fillStyle = '#fafaf7'; sctx.fillRect(0,0,SW,SH);
      const maxFreq = sim05fft_getTerms(sim05fft_targetType, 25)[24].freq;
      for (let k = 0; k < terms.length; k++) {
        const {freq, amp} = terms[k];
        const x = freq/maxFreq * SW;
        const h2 = amp / 2 * SH * 0.8;
        sctx.fillStyle = sim05fft_PALETTE[k % sim05fft_PALETTE.length];
        sctx.fillRect(x-2, SH-h2, 4, h2);
      }
      sctx.strokeStyle = '#e4dcc8'; sctx.lineWidth = 0.5;
      sctx.strokeRect(0,0,SW,SH);
      sctx.fillStyle = '#8a8371'; sctx.font = '9.5px monospace'; sctx.textAlign='left';
      sctx.fillText('Espectro de Amplitude (frequência →)', 6, 14);

      // Stats
      root.querySelector('#sim05fft_nTerms').textContent = sim05fft_nTerms;
      root.querySelector('#sim05fft_rms').textContent = rms.toFixed(3);

      // Term list
      const tl = root.querySelector('#sim05fft_termList');
      tl.innerHTML = '';
      for (let k = 0; k < Math.min(terms.length, 8); k++) {
        const {freq, amp} = terms[k];
        const d = document.createElement('span');
        d.style.cssText = 'font-size:10px; display:flex; align-items:center; gap:4px; background:#fafaf7; border:1px solid #e9e3d3; padding:4px 8px; border-radius:6px;';
        d.innerHTML = '<span style="width:10px;height:10px;border-radius:3px;background:' + sim05fft_PALETTE[k%sim05fft_PALETTE.length] + ';display:inline-block;"></span><span style="font-weight:700; color:#5e5a4a;">k=' + freq + ' A=' + amp.toFixed(2) + '</span>';
        tl.appendChild(d);
      }
    }

    window.sim05fft_setTarget = function(t) {
      sim05fft_targetType = t;
      ['sq','tr','sw'].forEach(id => {
        const el = root.querySelector('#sim05fft_' + id);
        if (el) el.classList.remove('sim05fft_active');
      });
      const map = {square:'sq', triangle:'tr', sawtooth:'sw'};
      root.querySelector('#sim05fft_' + map[t]).classList.add('sim05fft_active');
      root.querySelector('#sim05fft_target').textContent = {square:'quadrada',triangle:'triangular',sawtooth:'dente-serra'}[t];
      sim05fft_draw();
    };

    window.sim05fft_toggleComp = function() {
      sim05fft_showComp = !sim05fft_showComp;
      root.querySelector('#sim05fft_chk_comp').classList.toggle('sim05fft_active', sim05fft_showComp);
      sim05fft_draw();
    };
    
    window.sim05fft_toggleSum = function() {
      sim05fft_showSum = !sim05fft_showSum;
      root.querySelector('#sim05fft_chk_sum').classList.toggle('sim05fft_active', sim05fft_showSum);
      sim05fft_draw();
    };

    root.querySelector('#sim05fft_slider').addEventListener('input', function(){
      sim05fft_nTerms = +this.value;
      root.querySelector('#sim05fft_slVal').textContent = sim05fft_nTerms;
      sim05fft_draw();
    });

    sim05fft_draw();
  }

  function tryInitSim05FFT(){
    var root = document.getElementById('sim-05-freq');
    if (root) initSim05FFT(root); else setTimeout(tryInitSim05FFT, 200);
  }
  tryInitSim05FFT();
})();
</script>
""")

**Figure 5.2:** Simulateur interactif de la décomposition de Fourier 1D : visualisation de la somme de sinusoïdes avec différentes fréquences, amplitudes et phases. Ajoutez des termes et observez la convergence vers des formes d


<figure id="fig-05-sim-05-freq">
  <img src="imagens/fig-05-sim-05-freq.png" alt=" Simulateur interactif de la décomposition de Fourier 1D : visualisation de la somme de sinusoïdes avec différentes fréquences, amplitudes et phases. Ajoutez des termes et observez la convergence vers des formes d'onde arbitraires. " style="max-width:80%" />
  <figcaption><strong>Figure 5.2:</strong>  Simulateur interactif de la décomposition de Fourier 1D : visualisation de la somme de sinusoïdes avec différentes fréquences, amplitudes et phases. Ajoutez des termes et observez la convergence vers des formes d'onde arbitraires. </figcaption>
</figure>

### 5.3.2 Interprétation du spectre de fréquences

En appliquant la Transformée de Fourier Discrète (TFD) à une image et en visualisant le module de ses coefficients (voir [Figure 5.5](#fig-05-espectro-conceitual)), on obtient le **spectre de magnitude**, qui montre la distribution des fréquences spatiales présentes dans l’image.

Le coefficient situé à l’origine de la TFD, appelé **composante continue** (*Direct Current*), correspond à la fréquence nulle et représente l’intensité moyenne de l’image. Par convention, ce coefficient est stocké dans le coin supérieur gauche du spectre. Pour faciliter son interprétation, on applique l’opération **FFT Shift**, qui déplace la composante continue vers le centre de l’image. Après ce décalage, les basses fréquences se concentrent dans la région centrale, tandis que les hautes fréquences se trouvent près des bords, comme le résume la [Tableau 5.1](#tbl-05-espectro-regioes).

<a id="tbl-05-espectro-regioes"></a>

**Tabela 5.1:** Correspondance entre les régions du spectre de magnitude après application du FFT Shift.

| Région du spectre | Composantes prédominantes | Exemples dans l’image |
|:---|:---|:---|
| **Centre** (basses fréquences) | Variations spatiales lentes | Éclairage, régions homogènes et formes globales |
| **Région intermédiaire** (fréquences moyennes) | Variations à échelle intermédiaire | Textures et motifs répétitifs |
| **Bords** (hautes fréquences) | Variations spatiales rapides | Contours, détails fins et bruit |


Cette organisation facilite l’interprétation du spectre et la conception de filtres. L’atténuation des basses fréquences réduit les variations globales d’intensité, tandis que l’atténuation des hautes fréquences lisse l’image en réduisant les détails fins et une partie du bruit.

### 5.3.3 L'Expérience de la Grille : Construire une Image à partir d'un Unique Coefficient

Avant de présenter la formulation mathématique de la Transformée de Fourier Discrète (TFD), il est utile d'analyser son inverse, appelée Transformée de Fourier Discrète Inverse (TFDI). Considérons un spectre où tous les coefficients sont nuls, à l'exception d'un seul. Un exemple de cette construction est présenté dans le code de la [Figure 5.3](#fig-05-grade-2d) et peut être exploré de manière interactive dans le simulateur de la [Figure 5.4](#fig-05-sim-05-grade-2d)..

L'image reconstruite est une sinusoïde bidimensionnelle. La position du coefficient dans le spectre détermine son **orientation** et sa **fréquence spatiale**, tandis que sa magnitude et sa phase définissent, respectivement, son amplitude et son déplacement spatial. Ainsi, chaque coefficient de la TFD représente une composante sinusoïdale, et l'image originale peut être reconstruite par la somme de toutes ces composantes.

In [ ]:
%%writefile tmp/fig_05_grade_2d.cpp
#define MM_OUT "tmp/fig_05_grade_2d.png"
//| label: fig-05-grade-2d
//| fig-cap: "Toute fréquence dans le spectre (point isolé) correspond à une onde sinusoïdale 2D rotatée dans le domaine spatial."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <cmath>
#include "morph.hpp"
#include <filesystem>

// Helper: inverse fftshift (swap quadrants)
cv::Mat ifftshift(const cv::Mat& input) {
    int cx = input.cols / 2;
    int cy = input.rows / 2;
    cv::Mat output = input.clone();

    // Top-left quadrant
    cv::Mat q0(input, cv::Rect(0, 0, cx, cy));
    // Top-right quadrant
    cv::Mat q1(input, cv::Rect(cx, 0, input.cols - cx, cy));
    // Bottom-left quadrant
    cv::Mat q2(input, cv::Rect(0, cy, cx, input.rows - cy));
    // Bottom-right quadrant
    cv::Mat q3(input, cv::Rect(cx, cy, input.cols - cx, input.rows - cy));

    // Swap quadrants: TL->BR, TR->BL, BL->TR, BR->TL
    cv::Mat tmp;
    q0.copyTo(tmp);
    q3.copyTo(q0);
    tmp.copyTo(q3);

    q1.copyTo(tmp);
    q2.copyTo(q1);
    tmp.copyTo(q2);

    return output;
}

int main() {
    int N_grid = 100;
    cv::Mat espectro_vazio = cv::Mat::zeros(N_grid, N_grid, CV_64FC2);

    // Allumant un seul point (fréquence) hors du centre
    int u0 = 10, v0 = 5;
    espectro_vazio.at<cv::Vec2d>(N_grid/2 - v0, N_grid/2 - u0) = cv::Vec2d(1000, 0);

    // Retour au domaine spatial (IDFT)
    cv::Mat espectro_shifted = ifftshift(espectro_vazio);
    cv::Mat onda_2d_complex;
    cv::idft(espectro_shifted, onda_2d_complex, cv::DFT_SCALE | cv::DFT_COMPLEX_OUTPUT);

    // Extraire la partie réelle
    cv::Mat onda_2d_parts[2];
    cv::split(onda_2d_complex, onda_2d_parts);
    cv::Mat onda_2d = onda_2d_parts[0];

    // Normalisation pour visualisation
    cv::Mat onda_vis;
    cv::normalize(onda_2d, onda_vis, 0, 255, cv::NORM_MINMAX, CV_8U);

    // Magnitude du spectre
    cv::Mat espectro_mag[2];
    cv::split(espectro_vazio, espectro_mag);
    cv::Mat espectro_abs;
    cv::magnitude(espectro_mag[0], espectro_mag[1], espectro_abs);

    cv::Mat espectro_vis;
    cv::normalize(espectro_abs, espectro_vis, 0, 255, cv::NORM_MINMAX, CV_8U);

    // Accent visuel du point
    cv::Mat espectro_color;
    cv::cvtColor(espectro_vis, espectro_color, cv::COLOR_GRAY2BGR);
    cv::circle(espectro_color, cv::Point(N_grid/2 - u0, N_grid/2 - v0), 2, cv::Scalar(0, 0, 255), -1);

    mm::show(std::vector<mm::Image>{espectro_color, onda_vis},
             MM_OUT,
             std::vector<std::string>{"Spectre (1 point actif)", "Onde 2D Résultante (IDFT)"},
             2);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(espectro_color, "tmp/fig_05_grade_2d_0.png");
mm::write(onda_vis, "tmp/fig_05_grade_2d_1.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_grade_2d.cpp -o tmp/fig_05_grade_2d -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_grade_2d \
  && test -f "tmp/fig_05_grade_2d.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_grade_2d.png"

In [ ]:
mm.show(
    [
        mm.read("tmp/fig_05_grade_2d_0.png"),
        mm.read("tmp/fig_05_grade_2d_1.png"),
    ],
    titles=[
        'Espectro (1 ponto ativo)',
        'Onda 2D Resultante (IDFT)',
    ],
    cols=2,
    figsize=(10, 4),
)

**Figure 5.3:** Toda frequência no espectro (ponto isolado) corresponde a uma onda senoidal 2D rotacionada no domínio espacial.


In [ ]:
from IPython.display import HTML

HTML("""
<div id="sim-05-grade-2d" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">
<style>
  #sim-05-grade-2d * { box-sizing: border-box; }
  #sim-05-grade-2d canvas { display: block; background: #ffffff; border: 1px solid #e4dcc8; border-radius: 8px; }
  #sim-05-grade-2d button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; }
  #sim-05-grade-2d button:hover { background: #e8dfcf; }
  .sim-05-grade-2d_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-05-grade-2d_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-05-grade-2d_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; flex: 1; min-width: 80px; }
  .sim-05-grade-2d_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim-05-grade-2d_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim-05-grade-2d_slider_container { display: flex; align-items: center; gap: 8px; margin-bottom: 8px; }
  .sim-05-grade-2d_slider_container label { font-size: 11px; font-weight: 700; min-width: 120px; display: inline-block; color: #5e5a4a; }
  .sim-05-grade-2d_slider_container input[type=range] { flex: 1; cursor: pointer; height: 4px; }
  .sim-05-grade-2d_slider_val { font-size: 12px; font-family: monospace; font-weight: 700; min-width: 25px; text-align: right; color: #26241d; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">∿ Simulateur : Synthèse de fréquence 2D (IDFT)</span>
  <span class="sim-05-grade-2d_pill">Espace de Fourier</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div style="display:flex; gap:10px; margin-bottom:14px; flex-wrap:wrap;">
    <div class="sim-05-grade-2d_stat_box"><div class="sim-05-grade-2d_stat_label">Fréquence u</div><div id="sim-05-grade-2d_valU" class="sim-05-grade-2d_stat_value" style="color:#2980b9;">10</div></div>
    <div class="sim-05-grade-2d_stat_box"><div class="sim-05-grade-2d_stat_label">Fréquence v</div><div id="sim-05-grade-2d_valV" class="sim-05-grade-2d_stat_value" style="color:#27ae60;">5</div></div>
    <div class="sim-05-grade-2d_stat_box"><div class="sim-05-grade-2d_stat_label">Distance R</div><div id="sim-05-grade-2d_valR" class="sim-05-grade-2d_stat_value" style="color:#b9770e;">11.18</div></div>
    <div class="sim-05-grade-2d_stat_box"><div class="sim-05-grade-2d_stat_label">Angle θ</div><div id="sim-05-grade-2d_valAng" class="sim-05-grade-2d_stat_value" style="color:#c0392b;">26.6°</div></div>
  </div>

  <!-- Exibição Central (Espectro e Espaço) -->
  <div style="display: flex; gap: 16px; justify-content: center; align-items: center; margin-bottom: 14px; flex-wrap: wrap;">
    <div style="text-align: center; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px;">
      <div style="font-size: 10.5px; font-weight: 700; color: #5e5a4a; margin-bottom: 8px;">Spectre (Cliquez pour déplacer le point)</div>
      <canvas id="sim-05-grade-2d_CanvasSpec" width="220" height="220" style="margin:0 auto;"></canvas>
    </div>
    <div style="font-size: 20px; color: #8a8371; font-weight: bold;">➔</div>
    <div style="text-align: center; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px;">
      <div style="font-size: 10.5px; font-weight: 700; color: #5e5a4a; margin-bottom: 8px;">Onde 2D résultante (domaine spatial)</div>
      <canvas id="sim-05-grade-2d_CanvasSpace" width="220" height="220" style="margin:0 auto;"></canvas>
    </div>
  </div>

  <!-- Controles Deslizantes -->
  <div class="sim-05-grade-2d_panel">
    <div class="sim-05-grade-2d_slider_container">
      <label style="color:#2980b9;">Déplacement u (X) :</label>
      <input type="range" id="sim-05-grade-2d_sliderU" min="-30" max="30" value="10">
      <span id="sim-05-grade-2d_slValU" class="sim-05-grade-2d_slider_val">10</span>
    </div>
    <div class="sim-05-grade-2d_slider_container" style="margin-bottom:0;">
      <label style="color:#27ae60;">Déplacement v (Y) :</label>
      <input type="range" id="sim-05-grade-2d_sliderV" min="-30" max="30" value="5">
      <span id="sim-05-grade-2d_slValV" class="sim-05-grade-2d_slider_val">5</span>
    </div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Exp2D(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const specC = root.querySelector('#sim-05-grade-2d_CanvasSpec');
    const spaceC = root.querySelector('#sim-05-grade-2d_CanvasSpace');
    const sctx = specC.getContext('2d');
    const spctx = spaceC.getContext('2d');

    let u = 10;
    let v = 5;
    const N = 220;

    function updateMetrics() {
      const r = Math.sqrt(u*u + v*v);
      let angle = Math.atan2(v, u) * (180 / Math.PI);
      if (angle < 0) angle += 360;

      root.querySelector('#sim-05-grade-2d_valU').textContent = u;
      root.querySelector('#sim-05-grade-2d_valV').textContent = v;
      root.querySelector('#sim-05-grade-2d_valR').textContent = r.toFixed(2);
      root.querySelector('#sim-05-grade-2d_valAng').textContent = angle.toFixed(1) + '°';

      root.querySelector('#sim-05-grade-2d_sliderU').value = u;
      root.querySelector('#sim-05-grade-2d_sliderV').value = v;
      root.querySelector('#sim-05-grade-2d_slValU').textContent = u;
      root.querySelector('#sim-05-grade-2d_slValV').textContent = v;
    }

    function render() {
      updateMetrics();

      // 1. Desenhar Espectro
      sctx.fillStyle = '#fafaf7';
      sctx.fillRect(0, 0, N, N);

      sctx.strokeStyle = '#e4dcc8';
      sctx.lineWidth = 1;
      sctx.beginPath();
      sctx.moveTo(N/2, 0); sctx.lineTo(N/2, N);
      sctx.moveTo(0, N/2); sctx.lineTo(N, N/2);
      sctx.stroke();

      sctx.fillStyle = '#8a8371';
      sctx.beginPath();
      sctx.arc(N/2, N/2, 2.5, 0, 2*Math.PI);
      sctx.fill();

      let ptX = N/2 + u;
      let ptY = N/2 - v;

      sctx.strokeStyle = '#b9770e88';
      sctx.lineWidth = 1.5;
      sctx.beginPath();
      sctx.moveTo(N/2, N/2);
      sctx.lineTo(ptX, ptY);
      sctx.stroke();

      let symX = N/2 - u;
      let symY = N/2 + v;
      sctx.fillStyle = '#c0392b88';
      sctx.beginPath();
      sctx.arc(symX, symY, 4, 0, 2*Math.PI);
      sctx.fill();

      sctx.fillStyle = '#2980b9';
      sctx.strokeStyle = '#ffffff';
      sctx.lineWidth = 1.5;
      sctx.beginPath();
      sctx.arc(ptX, ptY, 5.5, 0, 2*Math.PI);
      sctx.fill();
      sctx.stroke();

      // 2. Desenhar Onda Espacial 2D Resultante
      const imgData = spctx.createImageData(N, N);
      const data = imgData.data;
      const freqScale = 2 * Math.PI / N;

      for (let y = 0; y < N; y++) {
        const ny = y - N/2;
        for (let x = 0; x < N; x++) {
          const nx = x - N/2;
          const val = Math.cos(freqScale * (u * nx + v * (-ny)));
          const intensity = Math.floor((val + 1) * 127.5);

          const idx = (y * N + x) * 4;
          data[idx]     = intensity;
          data[idx + 1] = intensity;
          data[idx + 2] = intensity;
          data[idx + 3] = 255;
        }
      }
      spctx.putImageData(imgData, 0, 0);
    }

    root.querySelector('#sim-05-grade-2d_sliderU').addEventListener('input', function() {
      u = parseInt(this.value, 10);
      render();
    });

    root.querySelector('#sim-05-grade-2d_sliderV').addEventListener('input', function() {
      v = parseInt(this.value, 10);
      render();
    });

    specC.addEventListener('mousedown', function(e) {
      const rect = specC.getBoundingClientRect();
      const clickX = e.clientX - rect.left;
      const clickY = e.clientY - rect.top;

      let newU = Math.round(clickX - N/2);
      let newV = Math.round(N/2 - clickY);

      u = Math.max(-30, Math.min(30, newU));
      v = Math.max(-30, Math.min(30, newV));

      render();
    });

    render();
  }

  function tryInitSim05Exp2D(){
    var root = document.getElementById('sim-05-grade-2d');
    if (root) initSim05Exp2D(root); else setTimeout(tryInitSim05Exp2D, 200);
  }
  tryInitSim05Exp2D();
})();
</script>
""")

**Figure 5.4:** Simulateur interactif de la synthèse de Fourier 2D. Modifiez la position horizontale ($u$) et verticale ($v$) du coefficient dans le spectre de fréquences centré et observez comment la distance par rapport au centre dicte la fréquence spatiale (épaisseur) et l


<figure id="fig-05-sim-05-grade-2d">
  <img src="imagens/fig-05-sim-05-grade-2d.png" alt=" Simulateur interactif de la synthèse de Fourier 2D. Modifiez la position horizontale ($u$) et verticale ($v$) du coefficient dans le spectre de fréquences centré et observez comment la distance par rapport au centre dicte la fréquence spatiale (épaisseur) et l'angle dicte l'orientation de l'onde sinusoïdale générée. " style="max-width:80%" />
  <figcaption><strong>Figure 5.4:</strong>  Simulateur interactif de la synthèse de Fourier 2D. Modifiez la position horizontale ($u$) et verticale ($v$) du coefficient dans le spectre de fréquences centré et observez comment la distance par rapport au centre dicte la fréquence spatiale (épaisseur) et l'angle dicte l'orientation de l'onde sinusoïdale générée. </figcaption>
</figure>

### 5.3.4 Définition mathématique

Considérons une image $f(x,y)$ de dimensions $M \times N$. Sa **Transformée de Fourier discrète 2D** (TFD) est définie par :

<a id="eq-05-dft"></a>
$$
F(u,v) = \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} f(x,y)\, e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)} \tag{5.1}
$$


où $u = 0, 1, \ldots, M-1$ et $v = 0, 1, \ldots, N-1$ représentent les fréquences discrètes dans les directions horizontale et verticale, respectivement. Le terme exponentiel correspond à une sinusoïde bidimensionnelle, dont la fréquence et l’orientation sont déterminées par les indices $(u,v)$.

La **Transformée de Fourier discrète inverse 2D** (TFDI) reconstruit l’image originale à partir de ses coefficients :

<a id="eq-05-idft"></a>
$$
f(x,y) = \frac{1}{MN} \sum_{u=0}^{M-1} \sum_{v=0}^{N-1} F(u,v)\, e^{j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)} \tag{5.2}
$$


Les équations [Équation 5.1](#eq-05-dft) et [Équation 5.2](#eq-05-idft) montrent que la TFD et la TFDI forment une paire de transformations : la première convertit l’image dans le domaine fréquentiel, tandis que la seconde reconstruit exactement l’image originale à partir de ses coefficients.

> ### 📝 5.3.4.1 À propos du symbole $j$
>
> Le terme $j$ désigne l’**unité imaginaire**, définie par $j^2 = -1$. En ingénierie et en traitement du signal, on adopte $j$ plutôt que $i$ afin d’éviter toute confusion avec la notation du courant électrique. Son utilisation dans l’exponentielle complexe, régie par la formule d’Euler ($e^{j\theta} = \cos\theta + j\sin\theta$), permet de représenter de manière compacte l’amplitude et la phase de chaque fréquence spatiale présente dans l’image.

> ### 📝 Qu'est-ce que la composante DC ?
>
> Le coefficient $F(0,0)$, appelé **composante DC** (*courant continu*), est égal à la somme des intensités de tous les pixels de l'image (voir [Figure 5.5](#fig-05-espectro-conceitual)) :
>
> $$
> F(0,0)=MN\,\bar{f},
> $$
>
> où $\bar{f}$ est l'intensité moyenne de l'image. Ainsi, la composante DC représente le niveau moyen d'intensité et, dans la plupart des images naturelles, possède la plus grande amplitude du spectre.
>
> Les autres coefficients représentent les variations autour de cette moyenne. Après application du **décalage de FFT**, la composante DC est déplacée vers le centre du spectre, concentrant les basses fréquences dans la région centrale et les hautes fréquences sur les bords.

In [ ]:
from IPython.display import HTML

HTML("""
<div style="font-family:sans-serif; max-width:880px; margin:0 auto; padding:10px;">
<div style="text-align:center; font-size:12px; font-weight:bold; color:#374151; margin-bottom:8px;">
  Anatomie du spectre de Fourier 2D (après fftshift)
</div>
<svg viewBox="0 0 640 320" xmlns="http://www.w3.org/2000/svg" style="width:100%;border:1px solid #e5e7eb;border-radius:8px;background:#f9fafb;">
  <!-- Fundo gradiente radial simulado -->
  <defs>
    <radialGradient id="specGrad" cx="50%" cy="50%" r="50%">
      <stop offset="0%" style="stop-color:#1e3a5f;stop-opacity:1"/>
      <stop offset="20%" style="stop-color:#1a5276;stop-opacity:1"/>
      <stop offset="50%" style="stop-color:#0d2137;stop-opacity:1"/>
      <stop offset="100%" style="stop-color:#050e1a;stop-opacity:1"/>
    </radialGradient>
    <radialGradient id="brightCenter" cx="50%" cy="50%" r="15%">
      <stop offset="0%" style="stop-color:#ffffff;stop-opacity:1"/>
      <stop offset="60%" style="stop-color:#f0c040;stop-opacity:0.9"/>
      <stop offset="100%" style="stop-color:#1a5276;stop-opacity:0"/>
    </radialGradient>
  </defs>
  <rect x="20" y="10" width="380" height="300" fill="url(#specGrad)" rx="6"/>
  <rect x="20" y="10" width="380" height="300" fill="url(#brightCenter)" rx="6"/>
  <!-- Cruzes de alta energia (bordas horizontais/verticais) -->
  <line x1="210" y1="10" x2="210" y2="310" stroke="#4a9eda" stroke-width="1.5" opacity="0.4"/>
  <line x1="20" y1="160" x2="400" y2="160" stroke="#4a9eda" stroke-width="1.5" opacity="0.4"/>
  <!-- Círculos de frequência -->
  <circle cx="210" cy="160" r="30" fill="none" stroke="#f0c040" stroke-width="1" stroke-dasharray="4,3" opacity="0.7"/>
  <circle cx="210" cy="160" r="70" fill="none" stroke="#7dd3fc" stroke-width="1" stroke-dasharray="4,3" opacity="0.5"/>
  <circle cx="210" cy="160" r="120" fill="none" stroke="#93c5fd" stroke-width="0.8" stroke-dasharray="4,3" opacity="0.3"/>
  <!-- Ponto DC -->
  <circle cx="210" cy="160" r="6" fill="#ffffff"/>
  <!-- Rótulos no espectro -->
  <text x="210" y="148" font-size="9" fill="#fff" text-anchor="middle" font-weight="bold">DC</text>
  <text x="210" y="205" font-size="8" fill="#f0c040" text-anchor="middle">basses fréquences</text>
  <text x="210" y="245" font-size="8" fill="#7dd3fc" text-anchor="middle">fréquences moyennes</text>
  <text x="330" y="110" font-size="8" fill="#93c5fd" text-anchor="middle">hautes fréquences</text>
  <text x="210" y="295" font-size="9" fill="#cbd5e1" text-anchor="middle" font-style="italic">Spectre de magnitude |F(u,v)| — échelle log</text>
  <!-- Painel direito: explicações -->
  <rect x="420" y="10" width="200" height="300" fill="#ffffff" rx="6" stroke="#e5e7eb"/>
  <text x="520" y="35" font-size="10" fill="#1e293b" text-anchor="middle" font-weight="bold">Régions du spectre</text>
  <!-- DC -->
  <circle cx="440" cy="65" r="7" fill="#ffffff" stroke="#f0c040" stroke-width="2"/>
  <text x="455" y="61" font-size="9" fill="#374151" font-weight="bold">DC (0,0)</text>
  <text x="455" y="73" font-size="8" fill="#6b7280">Moyenne globale des pixels</text>
  <!-- Baixas -->
  <rect x="433" y="95" width="14" height="14" rx="2" fill="#f0c040" opacity="0.7"/>
  <text x="455" y="105" font-size="9" fill="#374151" font-weight="bold">Basses fréquences</text>
  <text x="455" y="116" font-size="8" fill="#6b7280">Forme, fond, éclairage</text>
  <!-- Médias -->
  <rect x="433" y="135" width="14" height="14" rx="2" fill="#7dd3fc" opacity="0.7"/>
  <text x="455" y="145" font-size="9" fill="#374151" font-weight="bold">Fréquences moyennes</text>
  <text x="455" y="156" font-size="8" fill="#6b7280">Textures, motifs</text>
  <!-- Altas -->
  <rect x="433" y="175" width="14" height="14" rx="2" fill="#1e3a5f" stroke="#93c5fd" stroke-width="1"/>
  <text x="455" y="185" font-size="9" fill="#374151" font-weight="bold">Hautes fréquences</text>
  <text x="455" y="196" font-size="8" fill="#6b7280">Bords, bruit, détails</text>
  <!-- Seta de eixos -->
  <text x="440" y="235" font-size="8" fill="#6b7280">u → fréquence horizontale</text>
  <text x="440" y="248" font-size="8" fill="#6b7280">v → fréquence verticale</text>
  <line x1="440" y1="265" x2="600" y2="265" stroke="#d1d5db" stroke-width="0.8"/>
  <text x="520" y="280" font-size="8" fill="#9ca3af" text-anchor="middle">Visualisation en échelle log</text>
  <text x="520" y="292" font-size="8" fill="#9ca3af" text-anchor="middle">log(1 + |F|) compresse l'intervalle</text>
</svg>
</div>
""")

**Figure 5.5:** Diagramme conceptuel du spectre de Fourier 2D centré.


<figure id="fig-05-espectro-conceitual">
  <img src="imagens/fig-05-espectro-conceitual.png" alt=" Diagramme conceptuel du spectre de Fourier 2D centré. " style="max-width:80%" />
  <figcaption><strong>Figure 5.5:</strong>  Diagramme conceptuel du spectre de Fourier 2D centré. </figcaption>
</figure>

### 5.3.5 Magnitude et Phase

Chaque coefficient de la Transformée de Fourier Discrète (TFD) est un nombre complexe et peut s’écrire sous la forme

$$
F(u,v)=R(u,v)+j\,I(u,v),
$$

où $R(u,v)$ et $I(u,v)$ correspondent respectivement aux parties **réelle** et **imaginaire** du coefficient. À partir de l’équation [Équation 5.1](#eq-05-dft), on obtient

$$
R(u,v)=
\sum_{x=0}^{M-1}\sum_{y=0}^{N-1}
f(x,y)
\cos\!\left(
2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)
\right),
$$

et

$$
I(u,v)=
-
\sum_{x=0}^{M-1}\sum_{y=0}^{N-1}
f(x,y)
\sin\!\left(
2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)
\right).
$$

À partir de cette représentation, on définit deux grandeurs fondamentales :

- **Magnitude**, qui indique l’intensité de la composante fréquentielle,

$$
|F(u,v)|=\sqrt{R(u,v)^2+I(u,v)^2};
$$

- **Phase**, qui détermine l’alignement (ou le décalage) spatial de la composante,

$$
\phi(u,v)=\operatorname{atan2}\!\left(I(u,v),\,R(u,v)\right).
$$

Ainsi, chaque coefficient peut également être écrit sous sa forme polaire,

$$
F(u,v)=|F(u,v)|\,e^{j\phi(u,v)}.
$$

Le spectre de Fourier peut donc être visualisé au moyen de deux images distinctes : le **spectre de magnitude**, généralement utilisé pour analyser la distribution des fréquences, et le **spectre de phase**, qui décrit l’organisation spatiale des composantes sinusoïdales.

Bien que le spectre de magnitude soit le plus utilisé pour l’inspection visuelle, la phase contient une grande partie des informations structurelles de l’image. La combinaison de la magnitude et de la phase permet de reconstruire exactement l’image originale au moyen de la TFD inverse.

### 5.3.6 O que a Magnitude e a Fase carregam?

### 5.3.7 Ce que portent la magnitude et la phase ?

Une démonstration classique consiste à combiner la magnitude d'une image avec la phase d'une autre et à reconstruire le résultat. Cette expérience met en évidence que :

- **La phase** préserve la structure spatiale de l'image, y compris la position des objets, leurs contours et leur géométrie. De petites modifications de la phase peuvent provoquer de grands changements visuels.
- **La magnitude** contrôle la manière dont l'énergie est distribuée entre les fréquences spatiales, influençant principalement le contraste et la texture.

Lorsqu'une image est reconstruite avec la magnitude de A et la phase de B, le résultat tend à **ressembler davantage à B qu'à A**, ce qui montre que la phase est le principal composant responsable de l'organisation spatiale de la scène. Cependant, la magnitude reste importante, car elle module le contraste des structures reconstruites. Ainsi, une reconstruction fidèle dépend de la combinaison cohérente entre magnitude et phase.

Un exemple de ce comportement est présenté dans la [Figure 5.6](#fig-05-dft-intro).

> ### 📝 Analogie avec l'audio : limites et précautions
>
> La phase d'un signal joue des rôles distincts dans l'audio et les images :
>
> - **Audio stéréo ou multicanal :** la phase relative entre les canaux est fondamentale pour la perception de la position des sources sonores, via les différences interaurales de temps (ITD, *Interaural Time Differences*).
> - **Audio monaural :** la phase absolue exerce peu d'influence perceptuelle directe.
> - **Images (TFD) :** la phase est le principal facteur responsable de l'organisation spatiale de la scène, tandis que la magnitude module le contraste et la distribution de l'énergie entre les fréquences.
>
> Dans les deux domaines, la **magnitude** est liée à l'intensité des composantes de fréquence : en audio, elle influence le timbre et l'intensité perçue ; en images, elle influence le contraste et la texture.

In [ ]:
%%writefile tmp/fig_05_dft_intro.cpp
#define MM_OUT "tmp/fig_05_dft_intro.png"
//| label: fig-05-dft-intro
//| fig-cap: "Experimento de troca de fase: Imagem A (moedas) e Imagem B (padrão geométrico) reconstruídas com magnitudes e fases trocadas. O resultado mostra que a estrutura visual é **muito mais sensível à fase** do que à magnitude: quando a fase de B é mantida, a imagem resultante preserva a organização espacial de B, mesmo com a magnitude de A. A magnitude, por sua vez, influencia principalmente o contraste e a textura. Observe que a qualidade da reconstrução não é perfeita — há artefatos visíveis —, evidenciando a interdependência entre fase e magnitude para uma representação fiel da imagem."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <opencv2/core.hpp>
#include <opencv2/imgproc.hpp>
#include <opencv2/highgui.hpp>
#include <iostream>
#include <string>
#include <vector>
#include <cmath>
#include <complex>
#include <filesystem>
#include <algorithm>
#include "morph.hpp"

// Helper para trocar quadrantes (equivalente a np.fft.fftshift em 2D)
static cv::Mat fftShift(const cv::Mat& mat) {
    int cx = mat.cols / 2;
    int cy = mat.rows / 2;
    cv::Mat output;
    cv::Mat q0(mat, cv::Rect(0, 0, cx, cy));
    cv::Mat q1(mat, cv::Rect(cx, 0, mat.cols - cx, cy));
    cv::Mat q2(mat, cv::Rect(0, cy, cx, mat.rows - cy));
    cv::Mat q3(mat, cv::Rect(cx, cy, mat.cols - cx, mat.rows - cy));
    cv::Mat tmp;
    q0.copyTo(tmp);
    q3.copyTo(q0);
    tmp.copyTo(q3);
    q1.copyTo(tmp);
    q2.copyTo(q1);
    tmp.copyTo(q2);
    mat.copyTo(output);
    return output;
}

// Helper para construir imagem a partir de magnitude e fase
static cv::Mat buildFromMagPhase(const cv::Mat& mag, const cv::Mat& phase) {
    cv::Mat magF, phaseF;
    mag.convertTo(magF, CV_64F);
    phase.convertTo(phaseF, CV_64F);

    // Calcular cos e sin da fase
    cv::Mat cosPhase, sinPhase;
    cv::Mat phaseArr[] = {phaseF, phaseF};
    cv::Mat cosArr[] = {cosPhase, sinPhase};

    // Aplicar cos e sin elemento a elemento via loop
    cosPhase = cv::Mat(phaseF.size(), CV_64F);
    sinPhase = cv::Mat(phaseF.size(), CV_64F);
    for (int i = 0; i < phaseF.rows; i++) {
        for (int j = 0; j < phaseF.cols; j++) {
            double p = phaseF.at<double>(i, j);
            cosPhase.at<double>(i, j) = std::cos(p);
            sinPhase.at<double>(i, j) = std::sin(p);
        }
    }

    // real = mag * cos(phase), imag = mag * sin(phase)
    cv::Mat realPart = magF.mul(cosPhase);
    cv::Mat imagPart = magF.mul(sinPhase);

    // Construir complexa: real + i*imag
    cv::Mat complexParts[] = {realPart, imagPart};
    cv::Mat complexMat;
    cv::merge(complexParts, 2, complexMat);

    // IDFT
    cv::Mat result;
    cv::idft(complexMat, result, cv::DFT_SCALE | cv::DFT_REAL_OUTPUT);

    // Converter para 8-bit
    cv::Mat result8u;
    result.convertTo(result8u, CV_8U);

    return result8u;
}

int main() {
    // ── Experimento: A Importância da Fase ───────────────────────────────────────
    // ── Carregamento da imagem ────────────────────────────────────────────────────
    std::string url     = "https://upload.wikimedia.org/wikipedia/commons/2/25/GAZI.MD.AHAD_11.jpg";
    std::string caminho = "imagens/coins.jpg";

    mm::Image img_obj;
    if (!std::filesystem::exists(caminho)) {
        std::filesystem::create_directories("imagens");
        img_obj = mm::read(url);
        mm::write(img_obj, caminho);
    } else {
        img_obj = mm::read(caminho);
    }

    cv::Mat img_color = img_obj;
    if (img_color.channels() == 3) {
        cv::Mat temp;
        cv::cvtColor(img_color, temp, cv::COLOR_BGR2GRAY);
        img_color = temp;
    }
    mm::Image img_gray_mm = mm::gray(img_obj);
    cv::Mat img_gray = img_gray_mm;

    cv::Mat img_a;
    cv::resize(img_gray, img_a, cv::Size(400, 400));

    // Criar uma imagem B sintética (padrão geométrico)
    cv::Mat img_b = cv::Mat::zeros(400, 400, CV_8UC1);
    cv::rectangle(img_b, cv::Point(100, 100), cv::Point(300, 300), cv::Scalar(255), -1);
    cv::circle(img_b, cv::Point(200, 200), 150, cv::Scalar(128), 10);

    // FFT de A e B
    cv::Mat img_a_f;
    img_a.convertTo(img_a_f, CV_64F);
    cv::Mat img_b_f;
    img_b.convertTo(img_b_f, CV_64F);

    cv::Mat FA, FB;
    cv::dft(img_a_f, FA, cv::DFT_COMPLEX_OUTPUT);
    cv::dft(img_b_f, FB, cv::DFT_COMPLEX_OUTPUT);

    // Separar magnitude e fase de FA
    cv::Mat FA_parts[2];
    cv::split(FA, FA_parts);
    cv::Mat magA, phaseA;
    cv::magnitude(FA_parts[0], FA_parts[1], magA);
    cv::phase(FA_parts[0], FA_parts[1], phaseA);

    // Separar magnitude e fase de FB
    cv::Mat FB_parts[2];
    cv::split(FB, FB_parts);
    cv::Mat magB, phaseB;
    cv::magnitude(FB_parts[0], FB_parts[1], magB);
    cv::phase(FB_parts[0], FB_parts[1], phaseB);

    // Troca de Fase: rec_A_mag_B_fase = |FA| * exp(i * phase(FB))
    cv::Mat rec_A_mag_B_fase = buildFromMagPhase(magA, phaseB);

    // Troca de Fase: rec_B_mag_A_fase = |FB| * exp(i * phase(FA))
    cv::Mat rec_B_mag_A_fase = buildFromMagPhase(magB, phaseA);

    // Converter para mm::Image para exibição
    mm::Image img_a_mm = img_a;
    mm::Image img_b_mm = img_b;
    mm::Image rec_A_mm = rec_A_mag_B_fase;
    mm::Image rec_B_mm = rec_B_mag_A_fase;

    // Manter a variável img_gray como mm::Image no final (para persistência)
    img_gray = img_gray_mm;

    // Exibir resultados
    mm::show(
        std::vector<mm::Image>{img_a_mm, img_b_mm, rec_A_mm, rec_B_mm},
        MM_OUT,
        std::vector<std::string>{"Imagem A", "Imagem B", "Mag(A) + Fase(B)", "Mag(B) + Fase(A)"},
        4
    );

    // Exibir mensagem
    std::cout << "💡 A fase preserva bordas e contornos; a magnitude controla contraste e" << std::endl;
    std::cout << "textura. Em áudio estéreo, a fase afeta a localização espacial; em" << std::endl;
    std::cout << "imagens, determina a organização da cena." << std::endl;

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_gray, "tmp/state/img_gray_20.png");
// [pdi:state-io:end]

// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_a, "tmp/fig_05_dft_intro_0.png");
mm::write(img_b, "tmp/fig_05_dft_intro_1.png");
mm::write(rec_A_mag_B_fase, "tmp/fig_05_dft_intro_2.png");
mm::write(rec_B_mag_A_fase, "tmp/fig_05_dft_intro_3.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_dft_intro.cpp -o tmp/fig_05_dft_intro -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_dft_intro \
  && test -f "tmp/fig_05_dft_intro.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_dft_intro.png"

In [ ]:
mm.show(
    [
        mm.read("tmp/fig_05_dft_intro_0.png"),
        mm.read("tmp/fig_05_dft_intro_1.png"),
        mm.read("tmp/fig_05_dft_intro_2.png"),
        mm.read("tmp/fig_05_dft_intro_3.png"),
    ],
    titles=[
        'Imagem A',
        'Imagem B',
        'Mag(A) + Fase(B)',
        'Mag(B) + Fase(A)',
    ],
    cols=4,
    figsize=(16, 4),
)

**Figure 5.6:** Experimento de troca de fase: Imagem A (moedas) e Imagem B (padrão geométrico) reconstruídas com magnitudes e fases trocadas. O resultado mostra que a estrutura visual é **muito mais sensível à fase** do que à magnitude: quando a fase de B é mantida, a imagem resultante preserva a organização espacial de B, mesmo com a magnitude de A. A magnitude, por sua vez, influencia principalmente o contraste e a textura. Observe que a qualidade da reconstrução não é perfeita — há artefatos visíveis —, evidenciando a interdependência entre fase e magnitude para uma representação fiel da imagem.


## 5.4 Théorème de convolution et stratégies de filtrage

Le **théorème de convolution** établit une relation fondamentale entre les domaines spatial et fréquentiel :

<a id="eq-05-conv-teorema"></a>
$$
f(x,y) \circledast h(x,y) \;\overset{\mathcal{F}}{\longleftrightarrow}\; F(u,v)\,H(u,v) \tag{5.3}
$$


où $\circledast$ représente la **convolution circulaire discrète**. Ainsi, la convolution entre une image $f(x,y)$ et un filtre $h(x,y)$ peut être remplacée par la multiplication de leurs spectres.

En pratique, pour obtenir le même résultat que la convolution linéaire effectuée dans le domaine spatial, on applique un **remplissage par zéros** (*zero-padding*) avant la Transformée de Fourier rapide (FFT), afin d'éviter les artefacts sur les bords de l'image.

Cependant, le filtrage dans le domaine fréquentiel n'est pas toujours l'alternative la plus efficace. Pour des filtres tels que le filtre gaussien et le filtre moyenneur (*Box Filter*), la propriété de **séparabilité** permet de réduire considérablement le coût computationnel de la convolution dans le domaine spatial.

### 5.4.1 *Noyau* séparable vs. non séparable

Un ***noyau* séparable** peut être écrit comme le produit externe de deux vecteurs unidimensionnels,

$$
H = v\,h^T,
$$

ce qui permet de remplacer la convolution bidimensionnelle par deux convolutions unidimensionnelles successives : l'une dans la direction horizontale et l'autre dans la direction verticale.

En revanche, un ***noyau* non séparable** n'admet pas cette décomposition et, par conséquent, sa convolution doit être réalisée directement sur le voisinage bidimensionnel.

En pratique, pour un *noyau* de dimension $K \times K$, la convolution directe exige $K^2$ multiplications par pixel, tandis qu'un *noyau* séparable ne requiert que $2K$ multiplications, réduisant ainsi considérablement le coût computationnel.

### 5.4.2 Analyse de l’efficacité computationnelle

Considérons une image de dimensions $M \times N$ et un filtre carré de taille $K \times K$. La [Tableau 5.2](#tbl-05-fft-complexity-expanded) compare la complexité des principales stratégies de filtrage.

<a id="tbl-05-fft-complexity-expanded"></a>

**Tabela 5.2:** Comparaison de la complexité de la convolution directe, séparable et via la Transformée de Fourier rapide (FFT).

| Méthode de filtrage | Complexité asymptotique | Dépendance de $K$ | Application typique |
| --- | --- | --- | --- |
| **Spatiale non séparable** | $\mathcal{O}(MNK^2)$ | Quadratique | *Kernels* petits et non séparables |
| **Spatiale séparable** | $\mathcal{O}(MNK)$ | Linéaire | Filtres gaussien et de moyenne |
| **Via FFT** | $\mathcal{O}(MN\log(MN))$ | Indépendante de $K$ | *Kernels* grands |


Pour de petits *kernels*, la convolution spatiale, surtout lorsque le filtre est séparable, s’avère généralement plus efficace en raison du faible coût des opérations. À mesure que la taille du *kernel* augmente, le filtrage via FFT devient plus avantageux, car son coût ne dépend pratiquement pas de la dimension du filtre.

### 5.4.3 Discussion des résultats expérimentaux

Le graphique obtenu lors de l’essai avec l’image des pièces ($2560 \times 1920$), présenté à la [Figure 5.7](#fig-05-conv-eficiencia), confirme le comportement prédit par l’analyse de complexité computationnelle.

1. **Convolution non séparable ($\mathcal{O}(MNK^2)$)**  
La convolution directe présente une croissance quadratique avec la taille du *kernel*. Pour de petites valeurs de $K$, le coût est faible, mais il augmente rapidement à mesure que le *kernel* grandit, devenant irréalisable pour des applications en temps réel.

2. **Filtrage par FFT ($\mathcal{O}(MN \log(MN))$)**  
Le coût de la FFT ne dépend que de la taille de l’image, étant indépendant de $K$. Par conséquent, ses performances restent approximativement constantes lorsque le *kernel* varie, ce qui la rend avantageuse pour les filtres de grande taille ou non séparables.

3. **Convolution séparable ($\mathcal{O}(MNK)$)**  
La décomposition du *kernel* en deux filtres unidimensionnels réduit significativement le coût computationnel. En pratique, cette approche tend à être la plus efficace pour les filtres séparables, en particulier dans les implémentations optimisées.

En général, le choix de la méthode dépend de la taille et de la structure du *kernel*. Les filtres séparables sont plus efficaces dans le domaine spatial, tandis que la FFT devient plus avantageuse pour les *kernels* de grande taille ou pour de multiples convolutions dans le domaine fréquentiel.

<a id="eq-05-filter-comparison"></a>
$$
g = \mathcal{F}^{-1}\bigl[\mathcal{F}(f)\cdot \mathcal{F}(h)\bigr]
\quad \text{(FFT)}
\qquad
g = f \circledast h
\quad \text{(convolution directe)}
\qquad
g = (f \circledast v) \circledast h^T
\quad \text{(séparable)} \tag{5.4}
$$


où :

* $f(x,y)$ représente l’image d’entrée ;
* $h(x,y)$ est le *kernel* bidimensionnel du filtre ;
* $v$ et $h^T$ sont, respectivement, les vecteurs vertical et horizontal qui composent le *kernel* séparable.

In [ ]:
%%writefile tmp/fig_05_conv_eficiencia.cpp
#define MM_OUT "tmp/fig_05_conv_eficiencia.png"
//| label: fig-05-conv-eficiencia
//| fig-cap: "Comparação de eficiência: Convolução Não Separável (Espacial 2D), Separável (Espacial 1D) e via FFT."
//| echo: false
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <cmath>
#include "morph.hpp"
#include <filesystem>

int main() {
    // Trilha C++: curvas de CUSTO relativo (contagem de operações) — a forma das
    // curvas (K^2 vs 2K vs log N) e o que importa; a trilha py mede tempos reais.
    std::vector<double> K = {3, 7, 11, 15, 21, 31, 41, 51};
    double N2 = 256.0 * 256.0;

    std::vector<double> t_nao_sep, t_sep, t_fft;
    for (double k : K) {
        t_nao_sep.push_back(N2 * k * k / 1e6);
        t_sep.push_back(N2 * 2.0 * k / 1e6);
        t_fft.push_back(N2 * std::log2(N2) / 1e6);
    }

    std::vector<std::vector<double>> xs = {K, K, K};
    std::vector<std::vector<double>> ys = {t_nao_sep, t_sep, t_fft};
    std::vector<std::string> labels = {
        "Nao Separavel  O(N^2 K^2)",
        "Separavel  O(N^2 . 2K)",
        "Via FFT  O(N^2 log N)"
    };

    mm::Image chart = mm::lineChart(
        xs, ys, {}, labels,
        "Custo relativo: Espacial vs Frequencia",
        "Tamanho do kernel  K", "Operacoes  (x10^6)"
    );

    mm::show(std::vector<mm::Image>{chart}, MM_OUT, {"Comparacao de complexidade"}, 1);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(chart, "tmp/fig_05_conv_eficiencia_0.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_conv_eficiencia.cpp -o tmp/fig_05_conv_eficiencia -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_conv_eficiencia \
  && test -f "tmp/fig_05_conv_eficiencia.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_conv_eficiencia.png"

In [ ]:
mm.show(
    [
        mm.read("tmp/fig_05_conv_eficiencia_0.png"),
    ],
    titles=[
        'Comparacao de complexidade',
    ],
    cols=1,
)

**Figure 5.7:** Comparação de eficiência: Convolução Não Separável (Espacial 2D), Separável (Espacial 1D) e via FFT.


> ### ❗ 5.4.4 Le problème de la convolution circulaire (*wrap-around*)
>
> La Transformée de Fourier Discrète (TFD) suppose que l'image est **périodiquement étendue dans l'espace**, c'est-à-dire que ses bords se répètent indéfiniment.
>
> Dans cette condition, la multiplication dans le domaine fréquentiel correspond à une **convolution circulaire** dans le domaine spatial. Par conséquent, des régions opposées de l'image (haut et bas, gauche et droite) interagissent artificiellement, comme illustré dans la [Figure 5.8](#fig-05-padding-error).
>
> L'application d'un *zero-padding* avant la FFT réduit cet effet en étendant l'image avec des valeurs nulles sur les bords, rapprochant ainsi le résultat de la convolution linéaire. Ce comportement peut être interprété à la lumière du Théorème de Convolution, présenté dans la [Figure 5.9](#fig-05-conv-teorema).

In [ ]:
%%writefile tmp/fig_05_padding_error.cpp
#define MM_OUT "tmp/fig_05_padding_error.png"
//| label: fig-05-padding-error
//| fig-cap: "Sem *padding*, um deslocamento severo faz a imagem vazar para o lado oposto (convolução circular)."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <complex>
#include <vector>
#include <string>
#include <cmath>
#include "morph.hpp"
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_20.png");
// [pdi:state-io:end]

    // img_gray is provided (mm::Image)
    int M = img_gray.h;
    int N = img_gray.w;

    // Converter para CV_64F para cálculos complexos
    cv::Mat img_gray_mat(img_gray.h, img_gray.w, CV_8UC1, img_gray.data.data());
    cv::Mat img_gray_float;
    img_gray_mat.convertTo(img_gray_float, CV_64F);

    // Simulação de um filtro de deslocamento brutal
    cv::Mat H_shift(M, N, CV_64FC2, cv::Scalar(0, 0));
    for (int u = 0; u < M; ++u) {
        for (int v = 0; v < N; ++v) {
            double angle = -2.0 * M_PI * (u * 120.0 / M + v * 120.0 / N);
            H_shift.at<cv::Vec2d>(u, v) = cv::Vec2d(std::cos(angle), std::sin(angle));
        }
    }

    // Filtragem SEM padding (causa o wrap-around)
    cv::Mat F_img;
    cv::dft(img_gray_float, F_img, cv::DFT_COMPLEX_OUTPUT);

    // Multiplicação no domínio da frequência
    cv::Mat produto;
    cv::mulSpectrums(F_img, H_shift, produto, 0);

    cv::Mat img_vazada;
    cv::idft(produto, img_vazada, cv::DFT_SCALE | cv::DFT_REAL_OUTPUT);

    // Normalizar para visualização
    cv::Mat img_vazada_norm;
    cv::normalize(img_vazada, img_vazada_norm, 0, 255, cv::NORM_MINMAX, CV_8U);

    // Converter para mm::Image
    mm::Image img_vazada_vis(img_vazada_norm);

    // Mostrar resultados
    mm::show(std::vector<mm::Image>{img_gray, img_vazada_vis},
             MM_OUT,
             std::vector<std::string>{"Original", "Filtragem s/ Padding (Vazamento)"},
             2);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_05_padding_error_0.png");
mm::write(img_vazada_vis, "tmp/fig_05_padding_error_1.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_padding_error.cpp -o tmp/fig_05_padding_error -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_padding_error \
  && test -f "tmp/fig_05_padding_error.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_padding_error.png"

In [ ]:
mm.show(
    [
        mm.read("tmp/fig_05_padding_error_0.png"),
        mm.read("tmp/fig_05_padding_error_1.png"),
    ],
    titles=[
        'Original',
        'Filtragem s/ Padding (Vazamento)',
    ],
    cols=2,
    figsize=(10, 4),
)

**Figure 5.8:** Sem *padding*, um deslocamento severo faz a imagem vazar para o lado oposto (convolução circular).


In [ ]:
# [pdi:state-io] auto-gerado — não editar à mão
img_gray = mm.read("tmp/state/img_gray_20.png")
# [pdi:state-io:end]

# ── Noyau gaussien 11×11 
sigma  = 3.0
K      = 11
ks     = np.arange(K) - K // 2
gauss1d = np.exp(-ks**2 / (2 * sigma**2))
gauss1d /= gauss1d.sum()
kernel  = np.outer(gauss1d, gauss1d)    # noyau 2D séparable

# ── Méthode 1 : Convolution spatiale directe ─────────────────────────────────────
f_float  = img_gray.astype(np.float64)
conv_esp = cv2.filter2D(f_float, -1, kernel, borderType=cv2.BORDER_CONSTANT)

# ── Méthode 2 : Multiplication en fréquence (via FFT) ──────────────────────────
M, N     = f_float.shape
# Padding pour convolution linéaire (évite l'aliasing circulaire)
Mpad     = 2 ** int(np.ceil(np.log2(M + K - 1)))
Npad     = 2 ** int(np.ceil(np.log2(N + K - 1)))

# Positionne le noyau avec l'origine en (0,0) et padding avec des zéros
kernel_pad         = np.zeros((Mpad, Npad))
kh, kw             = kernel.shape
kernel_pad[:kh, :kw] = kernel

F_img   = np.fft.fft2(f_float,  (Mpad, Npad))
F_kern  = np.fft.fft2(kernel_pad)
conv_freq = np.real(np.fft.ifft2(F_img * F_kern))

# Recadrage pour compenser le décalage introduit par le positionnement du noyau
offset   = K // 2
conv_freq_crop = conv_freq[offset:offset+M, offset:offset+N]

# ── Vérification numérique ──────────────────────────────────────────────────────
diff = np.abs(conv_esp - conv_freq_crop)
print(f"Différence maximale  (|conv_esp - conv_freq|) : {diff.max():.2e}")
print(f"Différence moyenne   (|conv_esp - conv_freq|) : {diff.mean():.2e}")
print(f"→ Théorème de Convolution vérifié numériquement.")

conv_esp_vis  = cv2.normalize(conv_esp,       None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
conv_freq_vis = cv2.normalize(conv_freq_crop, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
diff_vis      = cv2.normalize(diff,           None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

mm.show(
    [img_gray, conv_esp_vis, conv_freq_vis, diff_vis],
    titles=[
        "Original",
        "Convolution spatiale",
        "Multiplication en fréquence",
        f"Différence (max={diff.max():.1e})"
    ],
    cols=4, figsize=(16, 5)
)

**Figure 5.9:** Vérification du Théorème de Convolution : la différence pixel à pixel entre la convolution spatiale (cv2.filter2D) et la multiplication en fréquence (FFT) est numériquement nulle — confirmant l


> ### 📝 5.5 À propos de la différence numérique
>
> La différence résiduelle de l'ordre de $10^{-13}$ ne viole pas le Théorème de Convolution, mais reflète des **limitations computationnelles** inhérentes à l'arithmétique en virgule flottante (double précision, ~$10^{-16}$) et à l'**ordre des opérations** entre les deux méthodes :
>
> - **Convolution spatiale :** somme pondérée des voisins avec des arrondis successifs.
> - **Convolution fréquentielle :** implique trois transformées FFT et une multiplication complexe, sujette à des erreurs de troncature et de quantification.
>
> Par conséquent, l'égalité théorique est exacte, mais l'implémentation numérique produit une différence pratiquement nulle (erreur relative < $10^{-12}$), confirmant le théorème dans la précision de la machine.

## 5.6 Filtres dans le Domaine Fréquentiel

Un filtre dans le domaine fréquentiel peut être interprété comme une **fonction de transfert appliquée au spectre de l'image**. Dans cette représentation, chaque coefficient de fréquence est multiplié par une valeur comprise entre 0 et 1, qui détermine son atténuation ou sa préservation. La forme de cette fonction définit l'effet visuel du filtre.

**Coupure brutale et *ringing*.** Les filtres idéaux avec une transition instantanée à une fréquence de coupure $D_0$ produisent des discontinuités dans le domaine fréquentiel. Cette discontinuité se reflète dans le domaine spatial sous forme d'oscillations près des bords, connues sous le nom de *ringing*. Cet effet est associé à la convolution avec des fonctions à support infini dans l'espace, comme la fonction *sinc*, comme illustré dans la [Figure 5.10](#fig-05-conv-teorema-zoom).

**Filtres à transition douce.** Des alternatives telles que les filtres gaussien et de Butterworth lissent la transition entre les régions de passage et de rejet, réduisant ainsi le *ringing*. En contrepartie, ce lissage implique une frontière de séparation moins nette entre les fréquences préservées et atténuées.

In [ ]:
%%writefile tmp/fig_05_conv_teorema_zoom.cpp
#define MM_OUT "tmp/fig_05_conv_teorema_zoom.png"
//| label: fig-05-conv-teorema-zoom
//| fig-cap: "A Dualidade Perigosa: o corte abrupto na Frequência (cilindro Ideal) vira obrigatoriamente uma *sinc* no espaço. Suas ondulações causam o *ringing* fantasma nas bordas da imagem."
//| echo: true
//| output: true
#include <opencv2/opencv.hpp>
#include "morph.hpp"
#include <vector>
#include <string>
#include <filesystem>

int main() {
    // Filtro Ideal na frequência (cilindro) e sua resposta espacial (sinc 2D).
    int N = 128;
    cv::Mat H_freq = mm::idealFilter(N, N, 20);  // 1 dentro do raio 20, 0 fora
    mm::Image h_space = mm::spatialKernel(H_freq); // ifft2(H) -> ondulações da sinc (ringing)

    mm::Image H_freq_img = H_freq; // convertendo para exibição
    mm::show(
        std::vector<mm::Image>{H_freq_img, h_space},
        MM_OUT,
        std::vector<std::string>{
            "Frequencia: filtro Ideal (cilindro)",
            "Espaco: ondulacoes da sinc (causa do ringing)",
        },
        2
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(H_freq, "tmp/fig_05_conv_teorema_zoom_0.png");
mm::write(h_space, "tmp/fig_05_conv_teorema_zoom_1.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_conv_teorema_zoom.cpp -o tmp/fig_05_conv_teorema_zoom -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_conv_teorema_zoom \
  && test -f "tmp/fig_05_conv_teorema_zoom.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_conv_teorema_zoom.png"

In [ ]:
mm.show(
    [
        mm.read("tmp/fig_05_conv_teorema_zoom_0.png"),
        mm.read("tmp/fig_05_conv_teorema_zoom_1.png"),
    ],
    titles=[
        'Frequencia: filtro Ideal (cilindro)',
        'Espaco: ondulacoes da sinc (causa do ringing)',
    ],
    cols=2,
)

**Figure 5.10:** A Dualidade Perigosa: o corte abrupto na Frequência (cilindro Ideal) vira obrigatoriamente uma *sinc* no espaço. Suas ondulações causam o *ringing* fantasma nas bordas da imagem.


### 5.6.1 Filtres Passe-Bas

Les filtres passe-bas atténuent les composantes haute fréquence, ce qui entraîne un lissage de l'image et une réduction du bruit. Après la centralisation du spectre (FFT Shift), la distance de chaque point au centre est donnée par :

<a id="eq-05-dist-centro"></a>
$$
D(u,v) = \sqrt{\left(u - \tfrac{M}{2}\right)^2 + \left(v - \tfrac{N}{2}\right)^2} \tag{5.5}
$$


**Filtre idéal (LPFI) :**
<a id="eq-05-lpf-ideal"></a>
$$
H_{\text{ideal}}(u,v) =
\begin{cases}
1, & D(u,v) \leq D_0 \\
0, & D(u,v) > D_0
\end{cases} \tag{5.6}
$$


La coupure brutale à $D_0$ introduit des discontinuités dans le domaine fréquentiel, entraînant des oscillations dans le domaine spatial, connues sous le nom de *ringing*. Cet effet est associé à la convolution avec des fonctions à support infini.

**Filtre gaussien (LPFG) :**
<a id="eq-05-lpf-gauss"></a>
$$
H_{\text{gauss}}(u,v) = e^{-D^2(u,v)/(2\sigma^2)} \tag{5.7}
$$


La régularité de la fonction gaussienne dans le domaine fréquentiel évite les discontinuités, ce qui élimine le *ringing* et produit une transition progressive entre les fréquences conservées et atténuées.

**Filtre de Butterworth (LPFB) d'ordre $n$ :**
<a id="eq-05-lpf-butterworth"></a>
$$
H_{\text{BW}}(u,v) = \frac{1}{1 + \left[D(u,v)/D_0\right]^{2n}} \tag{5.8}
$$


Le paramètre $n$ contrôle la régularité de la transition entre la transmission et la rejection des fréquences. Les petites valeurs produisent des transitions douces, tandis que les grandes valeurs rapprochent le comportement du filtre idéal, avec un risque accru de *ringing*. Un exemple comparatif est présenté à la [Figure 5.11](#fig-05-filtros-passa-baixa)..

In [ ]:
from IPython.display import HTML
HTML("""
<div style="font-family:sans-serif;max-width:680px;margin:0 auto;padding:10px;">
<div style="text-align:center;font-size:12px;font-weight:bold;color:#374151;margin-bottom:8px;">
  Profils des filtres passe-bas — comparaison visuelle (D₀ = 30)
</div>
<svg viewBox="0 0 640 200" xmlns="http://www.w3.org/2000/svg" style="width:100%;border:1px solid #e5e7eb;border-radius:8px;background:#fff;">
  <defs>
    <marker id="ah" markerWidth="6" markerHeight="6" refX="3" refY="3" orient="auto">
      <path d="M0,0 L6,3 L0,6 Z" fill="#9ca3af"/>
    </marker>
  </defs>
  <!-- Grid -->
  <line x1="60" y1="20" x2="60" y2="170" stroke="#e5e7eb" stroke-width="0.8"/>
  <line x1="60" y1="170" x2="610" y2="170" stroke="#e5e7eb" stroke-width="0.8"/>
  <!-- Eixos -->
  <line x1="60" y1="170" x2="605" y2="170" stroke="#9ca3af" stroke-width="1" marker-end="url(#ah)"/>
  <line x1="60" y1="175" x2="60" y2="15" stroke="#9ca3af" stroke-width="1" marker-end="url(#ah)"/>
  <text x="612" y="174" font-size="9" fill="#6b7280">D(u,v)</text>
  <text x="63" y="14" font-size="9" fill="#6b7280">H</text>
  <!-- Rótulos eixo Y -->
  <text x="52" y="35" font-size="8" fill="#6b7280" text-anchor="end">1.0</text>
  <text x="52" y="102" font-size="8" fill="#6b7280" text-anchor="end">0.5</text>
  <text x="52" y="173" font-size="8" fill="#6b7280" text-anchor="end">0.0</text>
  <line x1="57" y1="33" x2="63" y2="33" stroke="#9ca3af" stroke-width="0.8"/>
  <line x1="57" y1="100" x2="63" y2="100" stroke="#9ca3af" stroke-width="0.8"/>
  <!-- D0 marker -->
  <line x1="210" y1="30" x2="210" y2="175" stroke="#d1d5db" stroke-width="0.8" stroke-dasharray="3,3"/>
  <text x="210" y="184" font-size="8" fill="#9ca3af" text-anchor="middle">D₀</text>
  <!-- Filtro Ideal (vermelho) -->
  <polyline points="60,33 210,33 210,170 610,170" fill="none" stroke="#D85A30" stroke-width="2"/>
  <!-- Filtro Gaussiano (verde) -->
  <path d="M60,33 C100,33 130,40 160,60 S210,110 250,140 S320,168 610,170" fill="none" stroke="#1D9E75" stroke-width="2"/>
  <!-- Filtro Butterworth n=2 (azul) -->
  <path d="M60,33 C130,33 165,45 195,75 S225,130 250,148 S310,168 610,170" fill="none" stroke="#534AB7" stroke-width="2"/>
  <!-- Butterworth n=5 (roxo claro) -->
  <path d="M60,33 C170,33 195,40 208,70 S215,140 225,158 S260,170 610,170" fill="none" stroke="#9333ea" stroke-width="1.5" stroke-dasharray="5,3"/>
  <!-- Legenda -->
  <rect x="430" y="25" width="170" height="100" fill="#f9fafb" stroke="#e5e7eb" rx="4"/>
  <line x1="440" y1="45" x2="465" y2="45" stroke="#D85A30" stroke-width="2"/>
  <text x="470" y="49" font-size="9" fill="#374151">Idéal (coupure parfaite)</text>
  <text x="470" y="60" font-size="8" fill="#9ca3af">→ ringing sur les bords</text>
  <line x1="440" y1="78" x2="465" y2="78" stroke="#1D9E75" stroke-width="2"/>
  <text x="470" y="82" font-size="9" fill="#374151">Gaussien</text>
  <text x="470" y="93" font-size="8" fill="#9ca3af">→ sans ringing</text>
  <line x1="440" y1="106" x2="465" y2="106" stroke="#534AB7" stroke-width="2"/>
  <text x="470" y="110" font-size="9" fill="#374151">Butterworth n=2</text>
  <line x1="440" y1="118" x2="453" y2="118" stroke="#9333ea" stroke-width="1.5" stroke-dasharray="4,2"/>
  <text x="470" y="122" font-size="9" fill="#374151">Butterworth n=5</text>
  <!-- Zona de transição -->
  <text x="240" y="85" font-size="8" fill="#6b7280" font-style="italic">zone de</text>
  <text x="240" y="96" font-size="8" fill="#6b7280" font-style="italic">transition</text>
</svg>
<div style="font-size:10px;color:#6b7280;margin-top:6px;text-align:center;">
  À mesure que l’ordre du Butterworth augmente, le profil se rapproche du filtre idéal — et le ringing augmente.
</div>
</div>
""")

**Figure 5.11:** Filtros passe-bas.


<figure id="fig-05-filtros-passa-baixa">
  <img src="imagens/fig-05-filtros-passa-baixa.png" alt=" Filtros passe-bas. " style="max-width:80%" />
  <figcaption><strong>Figure 5.11:</strong>  Filtros passe-bas. </figcaption>
</figure>

### 5.6.2 Filtres Passe-Haut et Passe-Bande

**Les filtres passe-haut** peuvent être obtenus à partir d'un filtre passe-bas complémentaire, défini comme :

$$
H_{\text{HP}}(u,v) = 1 - H_{\text{LP}}(u,v)
$$

Ce type de filtre préserve les composantes haute fréquence, en mettant en évidence les contours et les détails, tout en atténuant les régions de variation lente.

**Les filtres passe-bande** préservent uniquement une bande intermédiaire de fréquences, limitée par deux rayons $D_L$ et $D_H$ :

$$
H_{\text{BP}}(u,v) =
H_{\text{LP}}^{(D_H)}(u,v)\cdot
\left[1 - H_{\text{LP}}^{(D_L)}(u,v)\right]
$$

Ce type de filtrage est utile lorsque l'on souhaite supprimer simultanément les composantes basse et haute fréquence, en ne préservant que les structures d'échelle intermédiaire.

Une application importante est la suppression du **bruit périodique**, dans lequel des motifs réguliers apparaissent comme des pics localisés dans le spectre de magnitude. Ces pics peuvent être atténués à l'aide de filtres *notch* (réjecteur de bande), positionnés spécifiquement sur les fréquences indésirables.

Des exemples de filtres dans le domaine fréquentiel sont présentés dans le simulateur de la [Figure 5.12](#fig-05-sim-05-filtros), [Figure 5.13](#fig-05-filtros-freq) et [Figure 5.14](#fig-05-filtros-passa-alta)..

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-05-filtros" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-05-filtros * { box-sizing: border-box; }
  #sim-05-filtros canvas { display: block; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; }
  #sim-05-filtros button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; }
  #sim-05-filtros button:hover { background: #e8dfcf; }
  #sim-05-filtros .sim05_sf_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  
  .sf_legend { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 8px; margin-bottom: 14px; }
  .sf_leg_item { display: flex; align-items: flex-start; gap: 10px; padding: 10px 12px; border-radius: 10px; border: 1px solid #e9e3d3; cursor: pointer; background: #fafaf7; transition: opacity .15s; }
  .sf_leg_item.sf_off { opacity: .35; }
  .sf_leg_swatch { width: 32px; min-width: 32px; height: 3px; margin-top: 8px; border-radius: 2px; }
  .sf_leg_name { font-size: 12.5px; font-weight: 700; }
  .sf_leg_desc { font-size: 10.5px; color: #8a8371; line-height: 1.4; margin-top: 2px; }
  
  .sf_controls { display: flex; align-items: center; gap: 12px; flex-wrap: wrap; margin-bottom: 14px; background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sf_ctrl_lbl { font-size: 11.5px; color: #5e5a4a; white-space: nowrap; font-weight: 600; }
  .sf_ctrl_val { font-size: 12px; font-weight: 700; min-width: 25px; color: #26241d; font-family: monospace; }
  .sf_radio_grp { display: flex; gap: 12px; }
  .sf_radio_grp label { display: flex; align-items: center; gap: 6px; font-size: 11.5px; color: #5e5a4a; cursor: pointer; font-weight: 600; }
  
  .sf_charts { display: grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap: 14px; }
  .sf_card { background: #fafaf7; border-radius: 12px; padding: 12px; border: 1px solid #e9e3d3; }
  .sf_card_lbl { font-size: 10.5px; color: #5e5a4a; margin-bottom: 8px; font-weight: 700; text-transform: uppercase; letter-spacing: .04em; }
  .sf_stats { display: flex; flex-wrap: wrap; gap: 8px; margin-top: 14px; }
  .sf_pill { font-size: 10.5px; padding: 4px 10px; border-radius: 8px; background: #fafaf7; color: #5e5a4a; border: 1px solid #e9e3d3; font-weight: 600; }
  .sf_pill b { color: #26241d; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎛️ Simulateur : Filtres dans le domaine fréquentiel</span>
  <span class="sim05_sf_pill">Passe-bas / Passe-haut</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <div class="sf_legend" id="sf_legend"></div>

  <div class="sf_controls">
    <span class="sf_ctrl_lbl">Fréquence de coupure D₀</span>
    <input type="range" id="sf_d0" min="5" max="100" value="30" step="1" style="flex:1;min-width:120px;max-width:240px;accent-color:#2980b9;cursor:pointer;">
    <span class="sf_ctrl_val" id="sf_d0v">30</span>
    <span class="sf_ctrl_lbl" style="margin-left:8px">Type de filtre</span>
    <div class="sf_radio_grp">
      <label><input type="radio" name="sf_ft" value="lp" checked style="cursor:pointer;">passe-bas</label>
      <label><input type="radio" name="sf_ft" value="hp" style="cursor:pointer;">passe-haut</label>
    </div>
  </div>

  <div class="sf_charts">
    <div class="sf_card">
      <div class="sf_card_lbl">Réponse H(D)</div>
      <canvas id="sf_c1" style="width:100%;display:block"></canvas>
    </div>
    <div class="sf_card">
      <div class="sf_card_lbl">Spectre filtré |F · H|</div>
      <canvas id="sf_c2" style="width:100%;display:block"></canvas>
    </div>
    <div class="sf_card">
      <div class="sf_card_lbl">Signal 1D — original vs filtré</div>
      <canvas id="sf_c3" style="width:100%;display:block"></canvas>
    </div>
    <div class="sf_card">
      <div class="sf_card_lbl">Énergie retenue par bande (%)</div>
      <canvas id="sf_c4" style="width:100%;display:block"></canvas>
    </div>
  </div>

  <div class="sf_stats" id="sf_stats"></div>

</div>
</div>

<script>
(function(){
  function initSim05Filtros(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    var SF = [
      {key:'ideal', name:'Ideal',          desc:'Corte perfeito em D₀ — causa ringing',       color:'#c0392b', dash:null },
      {key:'gauss', name:'Gaussiano',      desc:'Transição suave — sem ringing',                 color:'#27ae60', dash:[6,3]},
      {key:'bw2',   name:'Butterworth n=2', desc:'Compromisso: suave com banda controlável',       color:'#2980b9', dash:[4,2]},
      {key:'bw5',   name:'Butterworth n=5', desc:'Aproxima o ideal mantendo transição suave',    color:'#b9770e', dash:[2,2]},
    ];

    var sf_D0 = 30, sf_hp = false;
    var sf_on = {ideal:true, gauss:true, bw2:true, bw5:true};

    function sf_H(D, key, d0, hp){
      var h;
      if(key === 'ideal')      h = D <= d0 ? 1 : 0;
      else if(key === 'gauss') h = Math.exp(-D*D/(2*d0*d0));
      else if(key === 'bw2')   h = 1/(1+Math.pow(D/d0,4));
      else                     h = 1/(1+Math.pow(D/d0,10));
      return hp ? 1-h : h;
    }

    function sf_setup(id, h){
      var c = root.querySelector('#' + id);
      var w = c.parentElement.clientWidth - 24;
      if(w < 100) w = 280;
      c.width  = w;
      c.height = h || 180;
      return {c:c, ctx:c.getContext('2d'), w:c.width, h:c.height};
    }

    function sf_axes(ctx, w, h, pad, xmax, ymin, ymax, xlabel, ylabel){
      var l=pad.l, r=pad.r, t=pad.t, b=pad.b;
      ctx.clearRect(0,0,w,h);

      ctx.strokeStyle='rgba(0,0,0,.06)'; ctx.lineWidth=0.5;
      var nx=4, ny=4;
      for(var i=0; i<=nx; i++){
        var x = l + (w-l-r) * i / nx;
        ctx.beginPath(); ctx.moveTo(x, t); ctx.lineTo(x, h-b); ctx.stroke();
      }
      for(var j=0; j<=ny; j++){
        var y = t + (h-t-b) * j / ny;
        ctx.beginPath(); ctx.moveTo(l, y); ctx.lineTo(w-r, y); ctx.stroke();
      }

      ctx.strokeStyle='#e4dcc8'; ctx.lineWidth=1;
      ctx.beginPath(); ctx.moveTo(l, t); ctx.lineTo(l, h-b); ctx.lineTo(w-r, h-b); ctx.stroke();

      ctx.fillStyle='#8a8371'; ctx.font='10px monospace'; ctx.textAlign='center';
      for(var i=0; i<=nx; i++){
        var x = l + (w-l-r) * i / nx;
        var val = Math.round(xmax * i / nx);
        ctx.fillText(val, x, h-b+12);
      }
      ctx.textAlign='right';
      for(var j=0; j<=ny; j++){
        var y = t + (h-t-b) * j / ny;
        var val = ymax - (ymax-ymin) * j / ny;
        ctx.fillText(val.toFixed(2), l-4, y+3);
      }

      ctx.fillStyle='#5e5a4a'; ctx.font='10.5px Inter,sans-serif'; ctx.textAlign='center';
      ctx.fillText(xlabel, l+(w-l-r)/2, h-2);
      ctx.save(); ctx.translate(11, t+(h-t-b)/2); ctx.rotate(-Math.PI/2);
      ctx.fillText(ylabel, 0, 0); ctx.restore();

      var d0x = l + (sf_D0/xmax)*(w-l-r);
      if(d0x > l && d0x < w-r){
        ctx.strokeStyle='#b9770e'; ctx.lineWidth=1; ctx.setLineDash([4,3]);
        ctx.beginPath(); ctx.moveTo(d0x, t); ctx.lineTo(d0x, h-b); ctx.stroke();
        ctx.setLineDash([]);
        ctx.fillStyle='#b9770e'; ctx.font='10px monospace'; ctx.textAlign='center';
        ctx.fillText('D₀', d0x, t-2);
      }

      return {
        toX: function(v){ return l + (v/xmax)*(w-l-r); },
        toY: function(v){ return (h-b) - (v-ymin)/(ymax-ymin)*(h-t-b); }
      };
    }

    function sf_line(ctx, pts, color, dash, fill){
      if(!pts.length) return;
      ctx.strokeStyle = color; ctx.lineWidth = 2;
      ctx.setLineDash(dash || []);
      if(fill){
        ctx.fillStyle = color.replace(')', ', 0.12)').replace('rgb', 'rgba');
        ctx.beginPath();
        ctx.moveTo(pts[0].x, pts[0].baseY);
        pts.forEach(function(p){ ctx.lineTo(p.x, p.y); });
        ctx.lineTo(pts[pts.length-1].x, pts[pts.length-1].baseY);
        ctx.closePath(); ctx.fill();
      }
      ctx.beginPath();
      pts.forEach(function(p, i){ i===0 ? ctx.moveTo(p.x, p.y) : ctx.lineTo(p.x, p.y); });
      ctx.stroke();
      ctx.setLineDash([]);
    }

    function sf_drawProfile(){
      var s = sf_setup('sf_c1'); var ctx = s.ctx;
      var pad = {l:36, r:12, t:16, b:22};
      var ax = sf_axes(ctx, s.w, s.h, pad, 300, 0, 1, 'D(u,v)', 'H(D)');
      SF.forEach(function(f){
        if(!sf_on[f.key]) return;
        var pts = [];
        for(var d=0; d<=300; d+=2){
          pts.push({x: ax.toX(d), y: ax.toY(sf_H(d, f.key, sf_D0, sf_hp)), baseY: ax.toY(0)});
        }
        sf_line(ctx, pts, f.color, f.dash, false);
      });
    }

    function sf_drawSpectrum(){
      var s = sf_setup('sf_c2'); var ctx = s.ctx;
      var pad = {l:36, r:12, t:16, b:22};
      var ax = sf_axes(ctx, s.w, s.h, pad, 300, 0, 1, 'D(u,v)', '|F·H|');
      SF.forEach(function(f){
        if(!sf_on[f.key]) return;
        var pts = [];
        for(var d=0; d<=300; d+=2){
          var y = Math.max(0, Math.exp(-d*d/(2*80*80)) * sf_H(d, f.key, sf_D0, sf_hp));
          pts.push({x: ax.toX(d), y: ax.toY(y), baseY: ax.toY(0)});
        }
        sf_line(ctx, pts, f.color, f.dash, true);
      });
    }

    function sf_drawSignal(){
      var s = sf_setup('sf_c3'); var ctx = s.ctx;
      var pad = {l:36, r:12, t:16, b:22};
      var N = 128;
      var all = [];
      SF.forEach(function(f){
        if(!sf_on[f.key]) return;
        for(var t=0; t<N; t++){
          var v = 0;
          for(var fr=1; fr<80; fr++) v += sf_H(fr, f.key, sf_D0, sf_hp) * Math.exp(-fr*fr/(2*45*45)) * Math.cos(2*Math.PI*fr*t/N);
          all.push(v/7);
        }
      });
      for(var t=0; t<N; t++){
        var v = 0;
        for(var fr=1; fr<60; fr++) v += Math.exp(-fr*fr/(2*45*45)) * Math.cos(2*Math.PI*fr*t/N);
        all.push(v/7);
      }
      var ymin = Math.min.apply(null, all)*1.15, ymax = Math.max.apply(null, all)*1.15;
      if(ymax - ymin < 0.1){ymin = -0.5; ymax = 0.5;}
      var ax = sf_axes(ctx, s.w, s.h, pad, N, ymin, ymax, 'amostras', 'amp');

      var orig = [];
      for(var t=0; t<N; t++){
        var v = 0;
        for(var fr=1; fr<60; fr++) v += Math.exp(-fr*fr/(2*45*45)) * Math.cos(2*Math.PI*fr*t/N);
        orig.push({x: ax.toX(t), y: ax.toY(v/7), baseY: ax.toY(0)});
      }
      ctx.globalAlpha = 0.4;
      sf_line(ctx, orig, '#8a8371', [4,3], false);
      ctx.globalAlpha = 1;

      SF.forEach(function(f){
        if(!sf_on[f.key]) return;
        var pts = [];
        for(var t=0; t<N; t++){
          var v = 0;
          for(var fr=1; fr<80; fr++) v += sf_H(fr, f.key, sf_D0, sf_hp) * Math.exp(-fr*fr/(2*45*45)) * Math.cos(2*Math.PI*fr*t/N);
          pts.push({x: ax.toX(t), y: ax.toY(v/7), baseY: ax.toY(0)});
        }
        sf_line(ctx, pts, f.color, f.dash, false);
      });
    }

    function sf_drawEnergy(){
      var s = sf_setup('sf_c4'); var ctx = s.ctx;
      var pad = {l:36, r:12, t:16, b:32};
      var bands = [[0,20],[20,40],[40,60],[60,80],[80,100]];
      var labels = ['0–20','20–40','40–60','60–80','80–100'];
      var active = SF.filter(function(f){ return sf_on[f.key]; });
      if(active.length === 0) return;

      ctx.clearRect(0, 0, s.w, s.h);
      ctx.strokeStyle = 'rgba(0,0,0,.06)'; ctx.lineWidth = 0.5;
      for(var j=0; j<=4; j++){
        var y = pad.t + (s.h-pad.t-pad.b) * j / 4;
        ctx.beginPath(); ctx.moveTo(pad.l, y); ctx.lineTo(s.w-pad.r, y); ctx.stroke();
      }
      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth = 1;
      ctx.beginPath(); ctx.moveTo(pad.l, pad.t); ctx.lineTo(pad.l, s.h-pad.b);
      ctx.lineTo(s.w-pad.r, s.h-pad.b); ctx.stroke();

      ctx.fillStyle = '#8a8371'; ctx.font = '10px monospace'; ctx.textAlign = 'right';
      for(var j=0; j<=4; j++){
        var y = pad.t + (s.h-pad.t-pad.b) * j / 4;
        ctx.fillText((100 - j*25) + '%', pad.l-4, y+3);
      }

      var bw = (s.w - pad.l - pad.r) / bands.length;
      var gw = bw * 0.12, fw = (bw - gw*(active.length+1)) / active.length;
      if(fw < 2) fw = 2;

      bands.forEach(function(band, bi){
        var energy = function(key){
          var e = 0, n = 0;
          for(var d = band[0]; d < band[1]; d++){ e += Math.pow(sf_H(d, key, sf_D0, sf_hp), 2); n++; }
          return n > 0 ? Math.round(e/n * 100) : 0;
        };
        var bx = pad.l + bi * bw;
        active.forEach(function(f, fi){
          var val = energy(f.key);
          var x = bx + gw*(fi+1) + fw*fi;
          var barH = (val/100) * (s.h - pad.t - pad.b);
          var y = (s.h - pad.b) - barH;
          ctx.fillStyle = f.color + 'bb';
          ctx.fillRect(x, y, fw, barH);
          ctx.strokeStyle = f.color; ctx.lineWidth = 0.5;
          ctx.strokeRect(x, y, fw, barH);
        });
        ctx.fillStyle = '#8a8371'; ctx.font = '10px monospace'; ctx.textAlign = 'center';
        ctx.fillText(labels[bi], bx + bw/2, s.h - pad.b + 14);
      });

      ctx.fillStyle = '#5e5a4a'; ctx.font = '10.5px Inter,sans-serif'; ctx.textAlign = 'center';
      ctx.save(); ctx.translate(11, pad.t + (s.h-pad.t-pad.b)/2); ctx.rotate(-Math.PI/2);
      ctx.fillText('energia (%)', 0, 0); ctx.restore();
      ctx.fillText('banda de frequência', pad.l + (s.w-pad.l-pad.r)/2, s.h - 1);
    }

    function sf_buildLegend(){
      var el = root.querySelector('#sf_legend');
      el.innerHTML = '';
      SF.forEach(function(f){
        var d = document.createElement('div');
        d.className = 'sf_leg_item' + (sf_on[f.key] ? '' : ' sf_off');
        d.style.borderColor = sf_on[f.key] ? f.color : '#e9e3d3';
        var swatchStyle = 'background:' + f.color;
        if(f.dash){
          var seg = f.dash[0], gap = f.dash[1];
          swatchStyle = 'background:repeating-linear-gradient(90deg,' + f.color + ' 0 ' + seg + 'px,transparent ' + seg + 'px ' + (seg+gap) + 'px)';
        }
        d.innerHTML =
          '<div class="sf_leg_swatch" style="' + swatchStyle + '"></div>' +
          '<div><div class="sf_leg_name" style="color:' + f.color + '">' + f.name + '</div>' +
          '<div class="sf_leg_desc">' + f.desc + '</div></div>';
        d.addEventListener('click', function(){
          sf_on[f.key] = !sf_on[f.key]; sf_buildLegend(); sf_draw();
        });
        el.appendChild(d);
      });
    }

    function sf_stats(){
      var el = root.querySelector('#sf_stats');
      el.innerHTML = SF.filter(function(f){ return sf_on[f.key]; }).map(function(f){
        var h50 = sf_H(sf_D0, f.key, sf_D0, sf_hp).toFixed(2);
        var en = Math.round(function(){
          var s = 0;
          for(var i=0; i<200; i++) s += Math.pow(sf_H(i*0.5, f.key, sf_D0, sf_hp), 2);
          return s/200;
        }() * 100);
        return '<div class="sf_pill" style="border-color:' + f.color + '55">' +
          '<b style="color:' + f.color + '">' + f.name + '</b>' +
          ' H(D₀)=<b>' + h50 + '</b> &middot; energia=<b>' + en + '%</b></div>';
      }).join('');
    }

    function sf_draw(){
      sf_drawProfile();
      sf_drawSpectrum();
      sf_drawSignal();
      sf_drawEnergy();
      sf_stats();
    }

    root.querySelector('#sf_d0').addEventListener('input', function(){
      sf_D0 = +this.value;
      root.querySelector('#sf_d0v').textContent = sf_D0;
      sf_draw();
    });

    root.querySelectorAll('input[name="sf_ft"]').forEach(function(r){
      r.addEventListener('change', function(e){
        sf_hp = e.target.value === 'hp';
        sf_draw();
      });
    });

    sf_buildLegend();
    sf_draw();
    window.addEventListener('resize', sf_draw);
  }

  function tryInitSim05Filtros(){
    var root = document.getElementById('sim-05-filtros');
    if (root) initSim05Filtros(root); else setTimeout(tryInitSim05Filtros, 200);
  }
  tryInitSim05Filtros();
})();
</script>
""")

**Figure 5.12:** Simulateur interactif de filtres dans le domaine fréquentiel.


<figure id="fig-05-sim-05-filtros">
  <img src="imagens/fig-05-sim-05-filtros.png" alt=" Simulateur interactif de filtres dans le domaine fréquentiel. " style="max-width:80%" />
  <figcaption><strong>Figure 5.12:</strong>  Simulateur interactif de filtres dans le domaine fréquentiel. </figcaption>
</figure>

In [ ]:
%%writefile tmp/fig_05_filtros_freq.cpp
#define MM_OUT "tmp/fig_05_filtros_freq.png"
//| label: fig-05-filtros-freq
//| fig-cap: "Comparação entre filtros passa-baixa: Ideal (D₀=30), Gaussiano (D₀=30) e Butterworth (D₀=30, n=2). Perfis de H(u,v) ao longo de uma linha central e imagens filtradas correspondentes."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <cmath>
#include "morph.hpp"

// Fonction pour visualiser H (normalisation 0-255)
static cv::Mat H_vis(const cv::Mat& H) {
    cv::Mat h8;
    H.convertTo(h8, CV_8UC1, 255.0);
    cv::normalize(h8, h8, 0, 255, cv::NORM_MINMAX);
    return h8;
}

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_20.png");
// [pdi:state-io:end]

    int M = img_gray.h;
    int N = img_gray.w;
    int D0 = 30;
    int n_bw = 2;

    // ── Fonctions de transfert (H centré) via les constructeurs de morph.hpp ──
    //   Ideal : 1 si D<=D0, sinon 0
    //   Gaussien : exp(-D^2/(2 D0^2))
    //   Butterworth : 1 / (1 + (D/D0)^(2n))
    cv::Mat H_ideal = mm::idealFilter(M, N, D0);
    cv::Mat H_gauss = mm::gaussFilter(M, N, D0);
    cv::Mat H_bw    = mm::butterFilter(M, N, D0, n_bw);

    // ── Images filtrées via FFT ─────────────────────────────────────────────
    mm::Image img_ideal = mm::freqFilter(img_gray, H_ideal);
    mm::Image img_gauss = mm::freqFilter(img_gray, H_gauss);
    mm::Image img_bw    = mm::freqFilter(img_gray, H_bw);

    // ── Profils de H(u,v) sur la ligne centrale, dessinés avec cv::line ─────
    int linha = M / 2;
    int Wp = 512, Hp = 288;
    cv::Mat perfil(Hp, Wp, CV_8UC3, cv::Scalar(255, 255, 255));

    auto traca = [&](const cv::Mat& row, cv::Scalar cor) {
        cv::Point ant(-1, -1);
        bool first = true;
        for (int x = 0; x < N; ++x) {
            int px = (int)std::round(x * (Wp - 1.0) / (N - 1.0));
            int py = (int)std::round(10 + (1.0 - row.at<double>(0, x)) * (Hp - 20));
            if (!first) {
                cv::line(perfil, ant, cv::Point(px, py), cor, 2, cv::LINE_AA);
            }
            ant = cv::Point(px, py);
            first = false;
        }
    };

    // H_ideal[linha] est une ligne de la matrice CV_64F → extraire comme cv::Mat 1×N
    cv::Mat H_ideal_row = H_ideal.row(linha);
    cv::Mat H_gauss_row = H_gauss.row(linha);
    cv::Mat H_bw_row    = H_bw.row(linha);

    traca(H_ideal_row, cv::Scalar(216, 90, 48));
    traca(H_gauss_row, cv::Scalar(29, 158, 117));
    traca(H_bw_row,    cv::Scalar(83, 74, 183));

    mm::Image img_gray_c = img_gray;
    mm::Image img_ideal_c = img_ideal;
    mm::Image img_gauss_c = img_gauss;
    mm::Image img_bw_c = img_bw;
    mm::Image h_ideal_v = mm::Image(H_vis(H_ideal));
    mm::Image h_gauss_v = mm::Image(H_vis(H_gauss));
    mm::Image h_bw_v = mm::Image(H_vis(H_bw));
    mm::Image perfil_img = mm::Image(perfil);

    mm::show(
        std::vector<mm::Image>{img_gray_c, img_ideal_c, img_gauss_c, img_bw_c,
                                h_ideal_v, h_gauss_v, h_bw_v, perfil_img},
        MM_OUT,
        std::vector<std::string>{
            "Original", "LPF Ideal", "LPF Gaussien", "LPF Butterworth (n=2)",
            "H Ideal", "H Gaussien", "H Butterworth", "Profils H(u,v)"
        },
        4
    );

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_filtros_freq.cpp -o tmp/fig_05_filtros_freq -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_filtros_freq \
  && test -f "tmp/fig_05_filtros_freq.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_filtros_freq.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_filtros_freq.png"), figsize=(16, 9))

**Figure 5.13:** Comparação entre filtros passa-baixa: Ideal (D₀=30), Gaussiano (D₀=30) e Butterworth (D₀=30, n=2). Perfis de H(u,v) ao longo de uma linha central e imagens filtradas correspondentes.


In [ ]:
%%writefile tmp/fig_05_filtros_passa_alta.cpp
#define MM_OUT "tmp/fig_05_filtros_passa_alta.png"
#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include "morph.hpp"
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_20.png");
// [pdi:state-io:end]

    //| label: fig-05-filtros-passa-alta
    //| fig-cap: "Filtro passa-alta Gaussiano. (a) Original; (b) Filtro passa-alta (D₀=30) - as bordas das moedas e fundo texturizado são realçados."

    // Filtro passa-alta = complemento do passa-baixa Gaussiano (1 - H_gauss),
    // construído com mm.gaussFilter(..., highpass=true) e aplicado via FFT.
    int M = img_gray.h;
    int N = img_gray.w;
    double D0 = 30;
    cv::Mat H_alta = mm::gaussFilter(M, N, D0, true);
    mm::Image img_alta = mm::freqFilter(img_gray, H_alta);

    mm::show(
        std::vector<mm::Image>{img_gray, img_alta},
        MM_OUT,
        {"Original", "Passa-alta Gaussiano (D0=30)"},
        2
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_05_filtros_passa_alta_0.png");
mm::write(img_alta, "tmp/fig_05_filtros_passa_alta_1.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_filtros_passa_alta.cpp -o tmp/fig_05_filtros_passa_alta -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_filtros_passa_alta \
  && test -f "tmp/fig_05_filtros_passa_alta.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_filtros_passa_alta.png"

In [ ]:
mm.show(
    [
        mm.read("tmp/fig_05_filtros_passa_alta_0.png"),
        mm.read("tmp/fig_05_filtros_passa_alta_1.png"),
    ],
    titles=[
        'Original',
        'Passa-alta Gaussiano ($D_0=30$)',
    ],
    cols=2,
)

**Figure 5.14:** Filtro passa-alta Gaussiano. (a) Original; (b) Filtro passa-alta (D₀=30) - as bordas das moedas e fundo texturizado são realçados.


### 5.6.3 Suppression du bruit périodique

Le bruit périodique — associé à des interférences électriques, à des motifs réguliers de capteurs ou à des artefacts de numérisation — apparaît dans le spectre de Fourier sous forme de **pics ponctuels symétriques autour du centre**.

Le filtre **rejette-bande (*notch*)** atténue sélectivement ces fréquences, tout en préservant les autres composantes de l'image. Un exemple d'application est présenté dans la [Figure 5.15](#fig-05-ruido-periodico)..

In [ ]:
%%writefile tmp/fig_05_ruido_periodico.cpp
#define MM_OUT "tmp/fig_05_ruido_periodico.png"
//| label: fig-05-ruido-periodico
//| fig-cap: "Remoção de ruído periódico via filtro *notch* no domínio da frequência: (a) imagem com ruído senoidal, (b) espectro mostrando os picos do ruído, (c) máscara *notch* centrada nos picos, (d) imagem restaurada."
//| echo: true
//| output: true

// Ruído senoidal 2D -> espectro -> máscara notch nos 4 picos simétricos -> restauração.
#include <opencv2/opencv.hpp>
#include <cmath>
#include <iostream>
#include <vector>
#include <string>
#include "morph.hpp"

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_20.png");
// [pdi:state-io:end]

    int h_img = img_gray.h;
    int w_img = img_gray.w;

    // Criar meshgrid X, Y (equivalente a np.meshgrid)
    std::vector<std::vector<double>> X(h_img, std::vector<double>(w_img));
    std::vector<std::vector<double>> Y(h_img, std::vector<double>(w_img));
    for (int i = 0; i < h_img; ++i) {
        for (int j = 0; j < w_img; ++j) {
            X[i][j] = j;
            Y[i][j] = i;
        }
    }

    int u0 = 20, v0 = 20;  // frequências exatas do ruído

    // Converter img_gray para cv::Mat e depois para float64
    cv::Mat img_gray_mat(h_img, w_img, CV_8UC1, img_gray.data.data());
    cv::Mat img_gray_float;
    img_gray_mat.convertTo(img_gray_float, CV_64F);

    // Calcular ruído: ruido = 40 * sin(2 * pi * (u0*X/w_img + v0*Y/h_img))
    cv::Mat ruido(h_img, w_img, CV_64F);
    for (int i = 0; i < h_img; ++i) {
        for (int j = 0; j < w_img; ++j) {
            double arg = 2.0 * M_PI * (u0 * j / static_cast<double>(w_img) + v0 * i / static_cast<double>(h_img));
            ruido.at<double>(i, j) = 40.0 * std::sin(arg);
        }
    }

    // img_ruidosa = clip(img_gray + ruido, 0, 255).astype(uint8)
    cv::Mat img_ruidosa_float = img_gray_float + ruido;
    cv::Mat img_ruidosa_uint;
    img_ruidosa_float.convertTo(img_ruidosa_uint, CV_8UC1, 1.0, 0.0);
    cv::Mat img_ruidosa;
    cv::normalize(img_ruidosa_uint, img_ruidosa, 0, 255, cv::NORM_MINMAX, CV_8UC1);

    // Espectro de magnitude (log) da imagem ruidosa
    mm::Image img_ruidosa_mm(img_ruidosa);
    mm::Image mag_vis = mm::spectrumMag(img_ruidosa_mm);

    // Máscara notch: 1.0 em todo o plano, disco de 0.0 em cada um dos 4 picos
    cv::Mat mascara = cv::Mat::ones(h_img, w_img, CV_64F);
    int r_notch = 8;
    int cy = h_img / 2, cx = w_img / 2;

    // Coordenadas relativas para os 4 picos simétricos
    std::vector<std::pair<int, int>> picos = {{v0, u0}, {-v0, -u0}, {v0, -u0}, {-v0, u0}};

    for (const auto& pico : picos) {
        int dy = pico.first;
        int dx = pico.second;
        int py = cy + dy;
        int px = cx + dx;

        for (int i = 0; i < h_img; ++i) {
            for (int j = 0; j < w_img; ++j) {
                double dist = std::sqrt(std::pow(i - py, 2) + std::pow(j - px, 2));
                if (dist <= r_notch) {
                    mascara.at<double>(i, j) = 0.0;
                }
            }
        }
    }

    // Versão visual da máscara (0-255)
    cv::Mat mascara_vis_float;
    cv::normalize(mascara, mascara_vis_float, 0, 255, cv::NORM_MINMAX, CV_8UC1);
    mm::Image mascara_vis(mascara_vis_float);

    // Filtragem: aplica a máscara centrada como filtro no domínio da frequência
    mm::Image img_rest_vis = mm::freqFilter(img_ruidosa_mm, mascara);

    // Calcular PSNR
    cv::Mat img_rest_cv = img_rest_vis;
    double psnr = cv::PSNR(img_gray_mat, img_rest_cv);
    std::cout << "PSNR (original vs restaurada): " << psnr << " dB" << std::endl;

    // Mostrar resultados
    std::vector<mm::Image> imagens = {img_ruidosa_mm, mag_vis, mascara_vis, img_rest_vis};
    std::vector<std::string> titulos = {
        "Com ruído periódico",
        "Espectro (log)",
        "Máscara notch",
        "Restaurada (PSNR=" + std::to_string(psnr).substr(0, 4) + " dB)"
    };
    mm::show(imagens, MM_OUT, titulos, 4);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_ruido_periodico.cpp -o tmp/fig_05_ruido_periodico -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_ruido_periodico \
  && test -f "tmp/fig_05_ruido_periodico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_ruido_periodico.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_ruido_periodico.png"), figsize=(16, 4))

**Figure 5.15:** Remoção de ruído periódico via filtro *notch* no domínio da frequência: (a) imagem com ruído senoidal, (b) espectro mostrando os picos do ruído, (c) máscara *notch* centrada nos picos, (d) imagem restaurada.


### 📌 Synthèse — Filtres spectraux

| Filtre | Effet visuel | Artefact | Utilisation |
|:---|:---|:---|:---|
| Passe-bas idéal | Lissage intense | *Ring* | Illustratif |
| Passe-bas gaussien | Lissage doux | N’présente pas de *ring* | Lissage général |
| Passe-bas de Butterworth | Lissage contrôlé | *Ring* (ordres élevés) | Compromis entre lissage et sélectivité |
| Passe-haut | Renforcement des contours | Amplification du bruit | Détection des contours |
| *Notch* | Suppression sélective de fréquences | Distorsions locales possibles | Suppression du bruit périodique |

La conception de filtres dans le domaine fréquentiel consiste à définir des masques spectraux. Cependant, des effets dans le domaine spatial, tels que le *ring* et le flou, émergent directement de ces choix dans le spectre.

## 5.7 *Ondelettes* et Multirésolution

La Transformée de Fourier décompose le signal en fréquences **globales** : chaque coefficient $F(u,v)$ reçoit des contributions de toute l'image, sans information explicite sur la localisation spatiale de ces fréquences. Ainsi, les structures localisées, comme les bords, sont représentées de manière distribuée dans le spectre.

Les ***ondelettes* (ondelettes)** surmontent cette limitation en utilisant des fonctions de base **localisées dans l'espace**, qui peuvent être déplacées et mises à l'échelle. Ces fonctions possèdent un **support compact**, c'est-à-dire qu'elles sont différentes de zéro uniquement dans une région finie du domaine, permettant une représentation simultanée en termes de **fréquence et de localisation spatiale**.

### 5.7.1 La Limite de la Transformée de Fourier : localisation spatiale

La Transformée de Fourier décrit avec précision **quelles fréquences sont présentes** dans un signal, mais ne représente pas explicitement **où ces fréquences se produisent dans l’espace**.

Dans l’expérience présentée dans la [Figure 5.16](#fig-05-fracasso-fourier), deux images avec des structures localisées à des positions différentes produisent des spectres de magnitude pratiquement identiques. Cela est dû au fait que la représentation de Fourier est globale : chaque coefficient reçoit une contribution de l’ensemble de l’image.

Par conséquent, le spectre de magnitude ne représente pas explicitement la localisation des contours ou d’autres structures, mais uniquement la distribution des fréquences présentes. Cette limitation a motivé le développement de représentations multirésolution, telles que la Transformée *Wavelet* Discrète (DWT), capables de décrire simultanément la fréquence et la localisation spatiale des structures de l’image.

In [ ]:
%%writefile tmp/fig_05_fracasso_fourier.cpp
#define MM_OUT "tmp/fig_05_fracasso_fourier.png"
//| label: fig-05-fracasso-fourier
//| fig-cap: "Fourier global é cego para a posição. Os espectros não dizem onde as bordas estão."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include "morph.hpp"
#include <filesystem>

cv::Mat fftshift(const cv::Mat& input) {
    // Desloca o espectro para o centro
    int cx = input.cols / 2;
    int cy = input.rows / 2;
    cv::Mat output = input.clone();
    // Trocar os quadrantes
    cv::Mat q0(input, cv::Rect(0, 0, cx, cy));       // superior esquerdo
    cv::Mat q1(input, cv::Rect(cx, 0, input.cols - cx, cy)); // superior direito
    cv::Mat q2(input, cv::Rect(0, cy, cx, input.rows - cy)); // inferior esquerdo
    cv::Mat q3(input, cv::Rect(cx, cy, input.cols - cx, input.rows - cy)); // inferior direito

    cv::Mat temp;
    q0.copyTo(temp); q3.copyTo(q0); temp.copyTo(q3); // trocar q0 <-> q3
    q1.copyTo(temp); q2.copyTo(q1); temp.copyTo(q2); // trocar q1 <-> q2
    return output;
}

int main() {
    // Criar os sinais
    cv::Mat img_sinal1 = cv::Mat::zeros(128, 128, CV_64F);
    img_sinal1.colRange(20, 25) = 1.0;
    img_sinal1.rowRange(100, 105) = 1.0;

    cv::Mat img_sinal2 = cv::Mat::zeros(128, 128, CV_64F);
    img_sinal2.colRange(90, 95) = 1.0;
    img_sinal2.rowRange(30, 35) = 1.0;

    // Calcular os espectros com FFT e log1p
    cv::Mat fft_mat1, mag1, fft_mat2, mag2;
    cv::dft(img_sinal1, fft_mat1, cv::DFT_COMPLEX_OUTPUT);
    cv::dft(img_sinal2, fft_mat2, cv::DFT_COMPLEX_OUTPUT);

    // Dividir canais complexos
    std::vector<cv::Mat> planes1, planes2;
    cv::split(fft_mat1, planes1);
    cv::split(fft_mat2, planes2);

    // Magnitude
    cv::Mat mag1_shift, mag2_shift;
    cv::magnitude(planes1[0], planes1[1], mag1);
    cv::magnitude(planes2[0], planes2[1], mag2);

    // Aplicar fftshift e log1p (log(1+|F|))
    mag1 = fftshift(mag1);
    mag2 = fftshift(mag2);
    cv::log(1.0 + mag1, mag1_shift);
    cv::log(1.0 + mag2, mag2_shift);

    // Normalizar para visualização
    cv::normalize(mag1_shift, mag1_shift, 0, 255, cv::NORM_MINMAX, CV_8U);
    cv::normalize(mag2_shift, mag2_shift, 0, 255, cv::NORM_MINMAX, CV_8U);

    // Converter os sinais para 8-bit para exibição
    cv::Mat img1_8u, img2_8u;
    cv::normalize(img_sinal1, img1_8u, 0, 255, cv::NORM_MINMAX, CV_8U);
    cv::normalize(img_sinal2, img2_8u, 0, 255, cv::NORM_MINMAX, CV_8U);

    // Mostrar os resultados
    std::vector<mm::Image> images = {mm::Image(img1_8u), mm::Image(mag1_shift), 
                                     mm::Image(img2_8u), mm::Image(mag2_shift)};
    std::vector<std::string> titles = {"Sinal A", "Espectro A", "Sinal B (Deslocado)", "Espectro B"};
    mm::show(images, MM_OUT, titles, 4);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_sinal1, "tmp/fig_05_fracasso_fourier_0.png");
mm::write(mag1, "tmp/fig_05_fracasso_fourier_1.png");
mm::write(img_sinal2, "tmp/fig_05_fracasso_fourier_2.png");
mm::write(mag2, "tmp/fig_05_fracasso_fourier_3.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_fracasso_fourier.cpp -o tmp/fig_05_fracasso_fourier -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_fracasso_fourier \
  && test -f "tmp/fig_05_fracasso_fourier.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_fracasso_fourier.png"

In [ ]:
mm.show(
    [
        mm.read("tmp/fig_05_fracasso_fourier_0.png"),
        mm.read("tmp/fig_05_fracasso_fourier_1.png"),
        mm.read("tmp/fig_05_fracasso_fourier_2.png"),
        mm.read("tmp/fig_05_fracasso_fourier_3.png"),
    ],
    titles=[
        'Sinal A',
        'Espectro A',
        'Sinal B (Deslocado)',
        'Espectro B',
    ],
    cols=4,
    figsize=(14, 4),
)

**Figure 5.16:** Fourier global é cego para a posição. Os espectros não dizem onde as bordas estão.


### 5.7.2 Transformée *Wavelet* Discrète 2D

La Transformée *Wavelet* Discrète (DWT) applique, séparément dans les directions horizontale et verticale, deux filtres complémentaires : un **passe-bas** $h$ (approximation) et un **passe-haut** $g$ (détails), suivis d'un sous-échantillonnage par un facteur de 2 dans chaque dimension. Ce processus produit quatre sous-bandes, dont les noms indiquent la combinaison des filtres appliqués dans chaque direction (L = *Low-pass*, passe-bas ; H = *High-pass*, passe-haut). Les caractéristiques de chaque sous-bande sont résumées dans la [Tableau 5.3](#tbl-05-dwt-subbandas).

$$
\text{DWT}(f)=\{\underbrace{\text{LL}}_{\text{approx.}},\;
\underbrace{\text{LH}}_{\text{détails horizontaux}},\;
\underbrace{\text{HL}}_{\text{détails verticaux}},\;
\underbrace{\text{HH}}_{\text{détails diagonaux}}\}.
$$

<a id="tbl-05-dwt-subbandas"></a>

**Tabela 5.3:** Sous-bandes produites par la Transformée *Wavelet* Discrète 2D (DWT), indiquant les filtres appliqués dans chaque direction et le contenu prédominant de chaque composante.

| Sous-bande | Filtres appliqués | Contenu visuel |
|:---|:---:|:---|
| **LL** | bas × bas | Approximation de l'image (version lissée et réduite) |
| **LH** | bas × haut | Bords horizontaux et variations verticales |
| **HL** | haut × bas | Bords verticaux et variations horizontales |
| **HH** | haut × haut | Détails diagonaux et textures |


La décomposition peut être appliquée récursivement sur la sous-bande LL, générant une représentation multi-résolution. Après $J$ niveaux, on obtient une structure avec $3J+1$ sous-bandes, où chaque nouveau niveau réduit la résolution de la composante d'approximation.

> ### 📝 Lien avec les CNN
>
> La décomposition multi-résolution des *wavelets* possède une relation conceptuelle avec les représentations hiérarchiques utilisées dans les réseaux de neurones convolutifs (CNN). Dans les deux cas, des étapes successives de filtrage et de réduction de résolution produisent des descriptions de plus en plus abstraites de l'image. Cependant, les ***wavelets* utilisent des filtres mathématiquement définis et reconstructibles**, tandis que les **CNN apprennent leurs filtres pendant l'entraînement**.

### 5.7.3 Familles d'*ondelettes*

Différentes familles d'*ondelettes* présentent des compromis distincts entre **support spatial**, régularité et capacité de compression. Le **support** correspond à l'étendue de la fonction *ondelette* dans le domaine spatial : plus le support est petit, plus la fonction est localisée ; plus il est grand, plus sa représentation tend à être régulière, mais avec un coût de calcul plus élevé. La [Tableau 5.4](#tbl-05-wavelet-familias) compare certaines des familles les plus utilisées.

<a id="tbl-05-wavelet-familias"></a>

**Tabela 5.4:** Comparaison entre familles d'*ondelettes*, mettant en évidence la longueur du support, le nombre de moments nuls, la symétrie et les applications typiques.

| *Ondelette* | Longueur du support | Moments nuls | Symétrie | Usage typique |
|:---|:---:|:---:|:---:|:---|
| Haar | 2 | 1 | Asymétrique | Introduction et analyse de base |
| Daubechies db4 | 8 | 4 | Asymétrique | Compression et analyse générale |
| Symlet sym4 | 8 | 4 | Quasi symétrique | Reconstruction de signaux |
| Biorthogonale 5/3 | 5/3 | 2/2 | Symétrique | JPEG 2000 sans perte |
| Biorthogonale 9/7 | 9/7 | 4/4 | Symétrique | JPEG 2000 avec perte |


Les **moments nuls** mesurent la capacité de l'*ondelette* à représenter les régions lisses de l'image avec peu de coefficients non nuls. Une *ondelette* à $p$ moments nuls annule exactement les polynômes de degré jusqu'à $p-1$. Par conséquent, plus le nombre de moments nuls est élevé, plus l'efficacité de compression tend à être grande dans les régions homogènes, bien que cela implique généralement des fonctions à support plus long.

La [Figure 5.17](#fig-05-wavelet-functions) présente les fonctions de base (*ondelettes*) $\psi(t)$ dans le domaine spatial. Ces fonctions possèdent un **support compact**, c'est-à-dire qu'elles sont non nulles uniquement sur une région finie du domaine, contrairement aux sinusoïdes de la Transformée de Fourier, qui s'étendent sur tout le domaine.

In [ ]:
%%writefile tmp/fig_05_wavelet_functions.cpp
#define MM_OUT "tmp/fig_05_wavelet_functions.png"

#include <vector>
#include <string>
#include "morph.hpp"
#include <filesystem>

int main() {
    //| label: fig-05-wavelet-functions
    //| fig-cap: "Fonctions de la Wavelet (ψ). Notez comme elles décroissent rapidement vers zéro (support compact), contrairement aux sinusoïdes infinies de Fourier."
    //| echo: true
    //| output: true

    // psi via mm.wavefun (algorithme en cascade) — support compact : décroît vers zéro.
    std::vector<double> x_h, phi_h, psi_h;
    mm::wavefun("haar", 6, x_h, phi_h, psi_h);
    std::vector<double> x_d, phi_d, psi_d;
    mm::wavefun("db4", 6, x_d, phi_d, psi_d);

    mm::Image chart_h = mm::lineChart({x_h}, {psi_h}, {cv::Scalar(180, 60, 40)}, {"psi Haar"},
                                       "Ondaleta Haar (psi)", "t");
    mm::Image chart_d = mm::lineChart({x_d}, {psi_d}, {cv::Scalar(60, 140, 40)}, {"psi Daubechies 4"},
                                       "Ondaleta Daubechies 4 (psi)", "t");

    mm::show(std::vector<mm::Image>{chart_h, chart_d}, MM_OUT, {"Haar", "Daubechies 4"}, 2);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(chart_h, "tmp/fig_05_wavelet_functions_0.png");
mm::write(chart_d, "tmp/fig_05_wavelet_functions_1.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_wavelet_functions.cpp -o tmp/fig_05_wavelet_functions -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_wavelet_functions \
  && test -f "tmp/fig_05_wavelet_functions.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_wavelet_functions.png"

In [ ]:
mm.show(
    [
        mm.read("tmp/fig_05_wavelet_functions_0.png"),
        mm.read("tmp/fig_05_wavelet_functions_1.png"),
    ],
    titles=[
        'Haar',
        'Daubechies 4',
    ],
    cols=2,
)

**Figure 5.17:** Funções da *Wavelet* (ψ). Note como elas rapidamente decaem para zero (suporte compacto), ao contrário das senoides infinitas de Fourier.


Le diagramme de la [Figure 5.18](#fig-05-wavelet-diagrama) illustre l’analyse multirésolution effectuée par la DWT, dans laquelle la sous-bande d’approximation (LL) est successivement décomposée, formant une représentation hiérarchique à deux niveaux.

In [ ]:
from IPython.display import HTML
HTML("""
<div style="font-family:sans-serif;max-width:680px;margin:0 auto;padding:10px;">
<div style="text-align:center;font-size:12px;font-weight:bold;color:#374151;margin-bottom:8px;">
  Décomposition en ondelettes 2D — Structure multirésolution (2 niveaux)
</div>
<svg viewBox="0 0 640 260" xmlns="http://www.w3.org/2000/svg" style="width:100%;border:1px solid #e5e7eb;border-radius:8px;background:#f9fafb;">
  <!-- Imagem original -->
  <rect x="20" y="80" width="100" height="100" fill="#dbeafe" stroke="#3b82f6" stroke-width="1.5" rx="3"/>
  <text x="70" y="126" font-size="10" fill="#1e40af" text-anchor="middle" font-weight="bold">f(x,y)</text>
  <text x="70" y="140" font-size="9" fill="#1e40af" text-anchor="middle">M × N</text>
  <!-- Seta 1 -->
  <line x1="120" y1="130" x2="165" y2="130" stroke="#6b7280" stroke-width="1.5" marker-end="url(#arr)"/>
  <text x="142" y="124" font-size="8" fill="#6b7280" text-anchor="middle">TOD</text>
  <!-- Bloco Nível 1: 4 subbandas -->
  <rect x="165" y="55" width="80" height="75" fill="#fef3c7" stroke="#f59e0b" stroke-width="1.2" rx="2"/>
  <text x="205" y="87" font-size="8" fill="#92400e" text-anchor="middle" font-weight="bold">LL₁</text>
  <text x="205" y="98" font-size="7" fill="#92400e" text-anchor="middle">approx.</text>
  <text x="205" y="109" font-size="7" fill="#92400e" text-anchor="middle">M/2 × N/2</text>
  <rect x="245" y="55" width="80" height="75" fill="#dcfce7" stroke="#22c55e" stroke-width="1.2" rx="2"/>
  <text x="285" y="87" font-size="8" fill="#166534" text-anchor="middle" font-weight="bold">LH₁</text>
  <text x="285" y="98" font-size="7" fill="#166534" text-anchor="middle">horiz.</text>
  <rect x="165" y="130" width="80" height="75" fill="#fce7f3" stroke="#ec4899" stroke-width="1.2" rx="2"/>
  <text x="205" y="162" font-size="8" fill="#9d174d" text-anchor="middle" font-weight="bold">HL₁</text>
  <text x="205" y="173" font-size="7" fill="#9d174d" text-anchor="middle">vert.</text>
  <rect x="245" y="130" width="80" height="75" fill="#ede9fe" stroke="#8b5cf6" stroke-width="1.2" rx="2"/>
  <text x="285" y="162" font-size="8" fill="#5b21b6" text-anchor="middle" font-weight="bold">HH₁</text>
  <text x="285" y="173" font-size="7" fill="#5b21b6" text-anchor="middle">diag.</text>
  <!-- Rótulo nível 1 -->
  <text x="245" y="248" font-size="9" fill="#6b7280" text-anchor="middle">Niveau 1 — M/2 × N/2 chacun</text>
  <!-- Seta LL₁ → Nível 2 -->
  <line x1="205" y1="55" x2="205" y2="45" stroke="#6b7280" stroke-width="1" stroke-dasharray="3,2"/>
  <line x1="205" y1="45" x2="400" y2="45" stroke="#6b7280" stroke-width="1" stroke-dasharray="3,2"/>
  <line x1="400" y1="45" x2="400" y2="55" stroke="#6b7280" stroke-width="1" marker-end="url(#arr)" stroke-dasharray="3,2"/>
  <text x="302" y="40" font-size="8" fill="#6b7280" text-anchor="middle">TOD sur LL₁</text>
  <!-- Bloco Nível 2: subbandas de LL₁ -->
  <rect x="360" y="55" width="50" height="45" fill="#fef3c7" stroke="#f59e0b" stroke-width="1.2" rx="2"/>
  <text x="385" y="75" font-size="8" fill="#92400e" text-anchor="middle" font-weight="bold">LL₂</text>
  <text x="385" y="88" font-size="7" fill="#92400e" text-anchor="middle">M/4×N/4</text>
  <rect x="410" y="55" width="50" height="45" fill="#dcfce7" stroke="#22c55e" stroke-width="1.2" rx="2"/>
  <text x="435" y="80" font-size="8" fill="#166534" text-anchor="middle" font-weight="bold">LH₂</text>
  <rect x="360" y="100" width="50" height="45" fill="#fce7f3" stroke="#ec4899" stroke-width="1.2" rx="2"/>
  <text x="385" y="125" font-size="8" fill="#9d174d" text-anchor="middle" font-weight="bold">HL₂</text>
  <rect x="410" y="100" width="50" height="45" fill="#ede9fe" stroke="#8b5cf6" stroke-width="1.2" rx="2"/>
  <text x="435" y="125" font-size="8" fill="#5b21b6" text-anchor="middle" font-weight="bold">HH₂</text>
  <text x="435" y="168" font-size="8" fill="#6b7280" text-anchor="middle">Niveau 2</text>
  <!-- Legenda direita -->
  <rect x="490" y="55" width="140" height="130" fill="#fff" stroke="#e5e7eb" rx="4"/>
  <text x="560" y="73" font-size="9" fill="#374151" text-anchor="middle" font-weight="bold">Légende</text>
  <rect x="500" y="82" width="12" height="12" fill="#fef3c7" stroke="#f59e0b"/>
  <text x="518" y="92" font-size="8" fill="#374151">LL — Approximation</text>
  <rect x="500" y="100" width="12" height="12" fill="#dcfce7" stroke="#22c55e"/>
  <text x="518" y="110" font-size="8" fill="#374151">LH — Bords horiz.</text>
  <rect x="500" y="118" width="12" height="12" fill="#fce7f3" stroke="#ec4899"/>
  <text x="518" y="128" font-size="8" fill="#374151">HL — Bords vert.</text>
  <rect x="500" y="136" width="12" height="12" fill="#ede9fe" stroke="#8b5cf6"/>
  <text x="518" y="146" font-size="8" fill="#374151">HH — Détails diag.</text>
  <text x="560" y="168" font-size="8" fill="#6b7280" text-anchor="middle">Chaque niveau : ½ de la</text>
  <text x="560" y="178" font-size="8" fill="#6b7280" text-anchor="middle">résolution précédente</text>
  <defs>
    <marker id="arr" markerWidth="6" markerHeight="6" refX="3" refY="3" orient="auto">
      <path d="M0,0 L6,3 L0,6 Z" fill="#6b7280"/>
    </marker>
  </defs>
</svg>
</div>
""")

**Figure 5.18:** Schéma de la décomposition *wavelet* 2D en deux niveaux.


<figure id="fig-05-wavelet-diagrama">
  <img src="imagens/fig-05-wavelet-diagrama.png" alt=" Schéma de la décomposition *wavelet* 2D en deux niveaux. " style="max-width:80%" />
  <figcaption><strong>Figure 5.18:</strong>  Schéma de la décomposition *wavelet* 2D en deux niveaux. </figcaption>
</figure>

Le simulateur de la [Figure 5.19](#fig-05-sim-05-wavelet) permet d'explorer de manière interactive la Transformée *Wavelet* Discrète 2D (TWD) à l'aide de la *wavelet* de Haar. La décomposition en sous-bandes met en évidence la séparation entre la composante d'approximation et les composantes de détail de l'image.

Les différents motifs d'entrée permettent d'observer le comportement directionnel des filtres. Dans les images comportant des bords horizontaux et verticaux, les sous-bandes LH et HL mettent en évidence, respectivement, les variations verticales et horizontales de l'intensité. Dans les régions à variation douce, la majeure partie de l'énergie se concentre dans la sous-bande d'approximation LL, tandis que les sous-bandes de détail présentent des coefficients proches de zéro.

L'analyse multirésolution peut également être observée en augmentant le nombre de niveaux de décomposition. Dans ce cas, seule la sous-bande $\text{LL}_1$ est à nouveau décomposée, donnant naissance aux sous-bandes $\text{LL}_2$, $\text{LH}_2$, $\text{HL}_2$ et $\text{HH}_2$, qui forment le deuxième niveau de la représentation hiérarchique.

Dans les motifs constitués de régions homogènes de grande étendue, comme un dégradé doux ou un damier composé de grands blocs, l'énergie demeure principalement concentrée dans la sous-bande LL. Dans le dégradé, cela s'explique par le fait que les différences entre pixels voisins sont faibles. Dans le damier, en revanche, les pixels possèdent pratiquement la même intensité à l'intérieur de chaque bloc, de sorte que seules les frontières entre les blocs produisent des coefficients non nuls dans les sous-bandes de détail. Comme ces frontières n'occupent qu'une petite fraction de l'image, leur contribution à l'énergie totale reste réduite.

Pour permettre l'analyse visuelle de ces variations subtiles, le simulateur intègre un contrôle de gain de contraste des détails (variant de 1 à 8). Ce paramètre agit comme un facteur d'amplification linéaire appliqué exclusivement aux coefficients des sous-bandes de détail (LH, HL et HH) avant leur rendu à l'écran. Dans les scénarios de transition douce (comme le dégradé) ou d'uniformité locale (comme l'intérieur des blocs du damier), les différences numériques calculées par le filtre passe-haut de Haar donnent des coefficients très proches de zéro, ce qui rendrait les quadrants correspondants sombres et imperceptibles à l'œil nu. En multipliant ces valeurs par le gain, le simulateur fait ressortir visuellement les structures de haute fréquence cachées et met en évidence l'orientation des bords restants.

Le graphique de l'énergie par sous-bande quantifie cette répartition entre la composante d'approximation et les composantes de détail, démontrant que le gain visuel ne modifie pas la métrique originale de l'énergie. Dans les images naturelles, la majeure partie de l'énergie se concentre dans la sous-bande LL, tandis que les sous-bandes LH, HL et HH représentent principalement les bords, les textures et d'autres variations locales de l'intensité.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-05-wavelet" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-05-wavelet * { box-sizing: border-box; }
  #sim-05-wavelet canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; image-rendering: pixelated; }
  #sim-05-wavelet select { font-size: 11px; padding: 6px 10px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; font-weight: 600; cursor: pointer; outline: none; }
  #sim-05-wavelet input[type=range] { cursor: pointer; accent-color: #2980b9; }
  #sim-05-wavelet .sim04_w_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim04_w_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🌊 Simulateur : Décomposition Wavelet 2D</span>
  <span class="sim04_w_pill">Transformée de Haar</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Controles Superiores -->
  <div style="display:flex; flex-wrap:wrap; gap:14px; align-items:center; margin-bottom:14px; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px;">
    <div style="display:flex; flex-direction:column; gap:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Motif Synthétique</label>
      <select id="wsim_pattern">
        <option value="combined" selected>Combiné (formes + texture)</option>
        <option value="shapes">Formes (bords h/v)</option>
        <option value="texture">Texture (haute fréquence)</option>
        <option value="gradient">Dégradé doux</option>
      </select>
    </div>
    
    <div style="display:flex; flex-direction:column; gap:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Niveaux de Décomposition</label>
      <div style="display:flex; gap:12px; height:32px; align-items:center;">
        <label style="display:flex; align-items:center; gap:5px; font-size:11.5px; color:#5e5a4a; font-weight:600; cursor:pointer;"><input type="radio" name="wsim_lv" value="1" style="cursor:pointer;"> 1 niveau</label>
        <label style="display:flex; align-items:center; gap:5px; font-size:11.5px; color:#5e5a4a; font-weight:600; cursor:pointer;"><input type="radio" name="wsim_lv" value="2" checked style="cursor:pointer;"> 2 niveaux</label>
      </div>
    </div>
    
    <div style="display:flex; flex-direction:column; gap:4px; flex:1; min-width:160px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Gain de Contraste : <span id="wsim_gainv" style="font-weight:700; color:#2980b9;">3.0</span></label>
      <input type="range" id="wsim_gain" min="1" max="8" step="0.5" value="3" style="width:100%; height:4px;">
    </div>
  </div>

  <!-- Imagem Original vs Mosaico Wavelet -->
  <div style="display:grid; grid-template-columns:repeat(auto-fit, minmax(240px, 1fr)); gap:14px; margin-bottom:14px;">
    <div class="sim04_w_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; text-transform:uppercase; letter-spacing:.04em; margin-bottom:8px; font-weight:700;">Image Originale</div>
      <canvas id="wsim_orig" style="width:100%; display:block; margin:0 auto;"></canvas>
    </div>
    <div class="sim04_w_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; text-transform:uppercase; letter-spacing:.04em; margin-bottom:8px; font-weight:700;">Décomposition Wavelet (Mosaïque)</div>
      <canvas id="wsim_mosaic" style="width:100%; display:block; margin:0 auto; background:#ffffff;"></canvas>
    </div>
  </div>

  <!-- Energia por Subbanda -->
  <div class="sim04_w_panel" style="margin-bottom:14px; text-align:center;">
    <div style="font-size:10.5px; color:#5e5a4a; text-transform:uppercase; letter-spacing:.04em; margin-bottom:8px; font-weight:700; text-align:left;">Énergie par Sous-bande (%) — Somme Préservée (Parseval)</div>
    <canvas id="wsim_energy" style="width:100%; display:block; margin:0 auto;"></canvas>
  </div>

  <!-- Legendas Explicativas -->
  <div style="display:grid; grid-template-columns:repeat(auto-fit, minmax(220px, 1fr)); gap:8px;">
    <div style="display:flex; align-items:flex-start; gap:8px; padding:10px 12px; border-radius:10px; border:1px solid #e9e3d3; background:#fafaf7;">
      <div style="width:14px; height:14px; min-width:14px; margin-top:2px; border-radius:3px; background:linear-gradient(135deg,#26241d,#fafaf7);"></div>
      <div><div style="font-size:11.5px; font-weight:700; color:#26241d;">LL — Approximation</div><div style="font-size:10.5px; color:#8a8371;">Version lissée et réduite de l'image</div></div>
    </div>
    <div style="display:flex; align-items:flex-start; gap:8px; padding:10px 12px; border-radius:10px; border:1px solid #e9e3d3; background:#fafaf7;">
      <div style="width:14px; height:14px; min-width:14px; margin-top:2px; border-radius:3px; background:linear-gradient(135deg,#2980b9,#fafaf7,#c0392b);"></div>
      <div><div style="font-size:11.5px; font-weight:700; color:#26241d;">LH — Détail Horizontal</div><div style="font-size:10.5px; color:#8a8371;">Met en évidence les bords horizontaux (variation verticale)</div></div>
    </div>
    <div style="display:flex; align-items:flex-start; gap:8px; padding:10px 12px; border-radius:10px; border:1px solid #e9e3d3; background:#fafaf7;">
      <div style="width:14px; height:14px; min-width:14px; margin-top:2px; border-radius:3px; background:linear-gradient(135deg,#2980b9,#fafaf7,#c0392b);"></div>
      <div><div style="font-size:11.5px; font-weight:700; color:#26241d;">HL — Détail Vertical</div><div style="font-size:10.5px; color:#8a8371;">Met en évidence les bords verticaux (variation horizontale)</div></div>
    </div>
    <div style="display:flex; align-items:flex-start; gap:8px; padding:10px 12px; border-radius:10px; border:1px solid #e9e3d3; background:#fafaf7;">
      <div style="width:14px; height:14px; min-width:14px; margin-top:2px; border-radius:3px; background:linear-gradient(135deg,#2980b9,#fafaf7,#c0392b);"></div>
      <div><div style="font-size:11.5px; font-weight:700; color:#26241d;">HH — Détail Diagonal</div><div style="font-size:10.5px; color:#8a8371;">Textures et coins (variation dans les deux directions)</div></div>
    </div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim04Wavelet(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    var wsim_N = 128;

    function wsim_genImage(type){
      var img = [];
      for(var y=0; y<wsim_N; y++){
        var row = [];
        for(var x=0; x<wsim_N; x++){
          var v = 0;
          if(type==='gradient'){
            v = 255*(0.5*x/wsim_N + 0.5*y/wsim_N);
          } else if(type==='shapes'){
            v = 40 + 30*Math.sin(x/30);
            if(x>18&&x<58&&y>18&&y<58) v = 220;
            var cx=95, cy=95, r=24;
            if((x-cx)*(x-cx)+(y-cy)*(y-cy) < r*r) v = 195;
            if(x>70&&x<74) v = 235;
          } else if(type==='texture'){
            var period=8;
            v = ((Math.floor(x/period)+Math.floor(y/period))%2===0) ? 200 : 55;
          } else {
            v = 55 + 35*(x/wsim_N) + 15*Math.sin(y/12);
            if(x>12&&x<50&&y>12&&y<50) v = 225;
            var cx2=95, cy2=38, r2=17;
            if((x-cx2)*(x-cx2)+(y-cy2)*(y-cy2) < r2*r2) v = 205;
            if(y>82 && y<122){
              var p=6;
              v = ((Math.floor(x/p)+Math.floor(y/p))%2===0) ? 185 : 65;
            }
            if(Math.abs(x-y) < 2) v = 240;
          }
          row.push(Math.max(0,Math.min(255,v)));
        }
        img.push(row);
      }
      return img;
    }

    function wsim_dwt2(m){
      var h = m.length, w = m[0].length;
      var halfH = h / 2, halfW = w / 2;
      
      var LL = [], LH = [], HL = [], HH = [];
      for (var r = 0; r < halfH; r++) {
        LL.push(new Array(halfW));
        LH.push(new Array(halfW));
        HL.push(new Array(halfW));
        HH.push(new Array(halfW));
      }

      for(var r=0; r<halfH; r++){
        for(var c=0; c<halfW; c++){
          var a = m[2*r][2*c];
          var b = m[2*r][2*c+1];
          var g = m[2*r+1][2*c];
          var d = m[2*r+1][2*c+1];
          
          LL[r][c] = (a + b + g + d) / 2.0;
          LH[r][c] = (a - b + g - d) / 2.0;
          HL[r][c] = (a + b - g - d) / 2.0;
          HH[r][c] = (a - b - g + d) / 2.0;
        }
      }
      return {LL:LL, LH:LH, HL:HL, HH:HH};
    }

    function wsim_divCol(t){
      t = Math.max(-1, Math.min(1, t));
      if(t>=0) {
        // Interpola de branco (255,255,255) até azul forte (41,128,185)
        return [
          Math.round(255 + t*(41 - 255)),
          Math.round(255 + t*(128 - 255)),
          Math.round(255 + t*(185 - 255))
        ];
      }
      var s = -t;
      // Interpola de branco (255,255,255) até vermelho forte (192,57,43)
      return [
        Math.round(255 + s*(192 - 255)),
        Math.round(255 + s*(57 - 255)),
        Math.round(255 + s*(43 - 255))
      ];
    }

    function wsim_tileCanvas(mat, mode, gain){
      var d = mat.length;
      var cnv = document.createElement('canvas');
      cnv.width = d; cnv.height = d;
      var cctx = cnv.getContext('2d');
      var idata = cctx.createImageData(d,d);
      if(mode==='gray'){
        var mn=Infinity, mx=-Infinity;
        for(var r=0; r<d; r++) for(var c=0; c<d; c++){ var v=mat[r][c]; if(v<mn)mn=v; if(v>mx)mx=v; }
        var range=(mx-mn)||1;
        for(var r=0; r<d; r++) for(var c=0; c<d; c++){
          var v=(mat[r][c]-mn)/range*255;
          var idx=(r*d+c)*4;
          idata.data[idx]=v; idata.data[idx+1]=v; idata.data[idx+2]=v; idata.data[idx+3]=255;
        }
      } else {
        var maxAbs=0;
        for(var r=0; r<d; r++) for(var c=0; c<d; c++){ var av=Math.abs(mat[r][c]); if(av>maxAbs) maxAbs=av; }
        maxAbs = (maxAbs/gain) || 1;
        for(var r=0; r<d; r++) for(var c=0; c<d; c++){
          var t = mat[r][c]/maxAbs;
          var rgb = wsim_divCol(t);
          var idx=(r*d+c)*4;
          idata.data[idx]=rgb[0]; idata.data[idx+1]=rgb[1]; idata.data[idx+2]=rgb[2]; idata.data[idx+3]=255;
        }
      }
      cctx.putImageData(idata,0,0);
      return cnv;
    }

    function wsim_energySum(mat){
      var s=0;
      for(var r=0; r<mat.length; r++) for(var c=0; c<mat[0].length; c++) s += mat[r][c]*mat[r][c];
      return s;
    }

    var wsim_pattern='combined', wsim_level=2, wsim_gain=3;
    var wsim_currentImg = wsim_genImage(wsim_pattern);

    function wsim_setupSquare(id, cap){
      var c = root.querySelector('#' + id);
      var parentW = c.parentElement.clientWidth - 24;
      var w = Math.min(parentW, cap || 360);
      if(w<80) w = 240;
      c.width = w; c.height = w;
      return {c:c, ctx:c.getContext('2d'), size:w};
    }

    function wsim_drawOriginal(){
      var s = wsim_setupSquare('wsim_orig', 360);
      s.ctx.imageSmoothingEnabled = false;
      var tile = wsim_tileCanvas(wsim_currentImg, 'gray', wsim_gain);
      s.ctx.drawImage(tile, 0, 0, s.size, s.size);
    }

    function wsim_drawMosaic(){
      var s = wsim_setupSquare('wsim_mosaic', 360);
      var ctx = s.ctx, full = s.size, half = full/2;
      ctx.imageSmoothingEnabled = false;
      ctx.fillStyle = '#ffffff';
      ctx.fillRect(0, 0, full, full);

      var c1 = wsim_dwt2(wsim_currentImg);

      function place(mat, mode, x, y, w, h){
        var tile = wsim_tileCanvas(mat, mode, wsim_gain);
        ctx.drawImage(tile, x, y, w, h);
      }

      if(wsim_level===1){
        place(c1.LL, 'gray', 0, 0, half, half);
        place(c1.LH, 'div', half, 0, half, half);
        place(c1.HL, 'div', 0, half, half, half);
        place(c1.HH, 'div', half, half, half, half);
      } else {
        var c2 = wsim_dwt2(c1.LL);
        var q = half/2;
        place(c2.LL, 'gray', 0, 0, q, q);
        place(c2.LH, 'div', q, 0, q, q);
        place(c2.HL, 'div', 0, q, q, q);
        place(c2.HH, 'div', q, q, q, q);
        
        place(c1.LH, 'div', half, 0, half, half);
        place(c1.HL, 'div', 0, half, half, half);
        place(c1.HH, 'div', half, half, half, half);
      }

      ctx.strokeStyle = '#e4dcc8';
      ctx.lineWidth = 1.5;
      ctx.beginPath();
      ctx.moveTo(half,0); ctx.lineTo(half,full);
      ctx.moveTo(0,half); ctx.lineTo(full,half);
      ctx.stroke();
      if(wsim_level===2){
        ctx.lineWidth = 1;
        ctx.beginPath();
        ctx.moveTo(half/2,0); ctx.lineTo(half/2,half);
        ctx.moveTo(0,half/2); ctx.lineTo(half,half/2);
        ctx.stroke();
      }

      ctx.font = '700 10.5px monospace';
      function lbl(t,x,y){
        ctx.fillStyle = '#26241d';
        ctx.fillText(t, x+5, y+14);
      }
      if(wsim_level===1){
        lbl('LL', 0,0); lbl('LH', half,0); lbl('HL',0,half); lbl('HH', half,half);
      } else {
        lbl('LL₂', 0,0); lbl('LH₂', half/2,0); lbl('HL₂',0,half/2); lbl('HH₂', half/2, half/2);
        lbl('LH₁', half,0); lbl('HL₁',0,half); lbl('HH₁', half,half);
      }
    }

    function wsim_drawEnergy(){
      var s = root.querySelector('#wsim_energy');
      var w = s.parentElement.clientWidth - 24;
      if(w<100) w = 280;
      s.width = w; s.height = 160;
      var ctx = s.getContext('2d');
      ctx.clearRect(0,0,w,s.height);

      var c1 = wsim_dwt2(wsim_currentImg);
      var total = wsim_energySum(wsim_currentImg);
      var bars, labels;
      if(wsim_level===1){
        bars = [wsim_energySum(c1.LL), wsim_energySum(c1.LH), wsim_energySum(c1.HL), wsim_energySum(c1.HH)];
        labels = ['LL','LH','HL','HH'];
      } else {
        var c2 = wsim_dwt2(c1.LL);
        bars = [wsim_energySum(c2.LL), wsim_energySum(c2.LH), wsim_energySum(c2.HL), wsim_energySum(c2.HH),
                wsim_energySum(c1.LH), wsim_energySum(c1.HL), wsim_energySum(c1.HH)];
        labels = ['LL₂','LH₂','HL₂','HH₂','LH₁','HL₁','HH₁'];
      }
      var pcts = bars.map(function(b){ return b/total*100; });

      var pad = {l:34, r:10, t:12, b:24};
      var plotH = s.height - pad.t - pad.b;
      var plotW = w - pad.l - pad.r;

      ctx.strokeStyle = 'rgba(0,0,0,.06)'; ctx.lineWidth=0.5;
      for(var j=0;j<=4;j++){
        var y = pad.t + plotH*j/4;
        ctx.beginPath(); ctx.moveTo(pad.l,y); ctx.lineTo(w-pad.r,y); ctx.stroke();
      }
      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth=1;
      ctx.beginPath(); ctx.moveTo(pad.l,pad.t); ctx.lineTo(pad.l,s.height-pad.b); ctx.lineTo(w-pad.r,s.height-pad.b); ctx.stroke();

      ctx.fillStyle = '#8a8371'; ctx.font='9.5px monospace'; ctx.textAlign='right';
      for(var j=0;j<=4;j++){
        var y = pad.t + plotH*j/4;
        ctx.fillText(Math.round(100 - j*25) + '%', pad.l-4, y+3);
      }

      var bw = plotW/pcts.length;
      var barW = bw*0.6;
      ctx.textAlign='center';
      pcts.forEach(function(p, i){
        var x = pad.l + i*bw + (bw-barW)/2;
        var barH = (p/100)*plotH;
        var y = (s.height-pad.b) - barH;
        var isLL = labels[i].indexOf('LL') === 0;
        ctx.fillStyle = isLL ? '#2980b9' : '#c0392b';
        ctx.fillRect(x,y,barW,barH);
        ctx.strokeStyle = isLL ? '#1b4f72' : '#78281f';
        ctx.lineWidth=0.5;
        ctx.strokeRect(x,y,barW,barH);
        if(barH>16){
          ctx.fillStyle='#ffffff'; ctx.font='bold 9.5px monospace';
          ctx.fillText(Math.round(p) + '%', x+barW/2, y+13);
        }
        ctx.fillStyle='#8a8371'; ctx.font='10px monospace';
        ctx.fillText(labels[i], x+barW/2, s.height-pad.b+14);
      });
    }

    function wsim_redraw(){
      wsim_currentImg = wsim_genImage(wsim_pattern);
      wsim_drawOriginal();
      wsim_drawMosaic();
      wsim_drawEnergy();
    }

    root.querySelector('#wsim_pattern').addEventListener('change', function(e){
      wsim_pattern = e.target.value; wsim_redraw();
    });
    root.querySelectorAll('input[name="wsim_lv"]').forEach(function(r){
      r.addEventListener('change', function(e){ wsim_level = +e.target.value; wsim_drawMosaic(); wsim_drawEnergy(); });
    });
    root.querySelector('#wsim_gain').addEventListener('input', function(e){
      wsim_gain = +e.target.value;
      root.querySelector('#wsim_gainv').textContent = wsim_gain.toFixed(1);
      wsim_drawMosaic();
    });

    wsim_redraw();
    window.addEventListener('resize', wsim_redraw);
  }

  function tryInitSim04Wavelet(){
    var root = document.getElementById('sim-05-wavelet');
    if (root) initSim04Wavelet(root); else setTimeout(tryInitSim04Wavelet, 200);
  }
  tryInitSim04Wavelet();
})();
</script>
""")

**Figure 5.19:** Simulation de la décomposition *wavelet* 2D.


<figure id="fig-05-sim-05-wavelet">
  <img src="imagens/fig-05-sim-05-wavelet.png" alt=" Simulation de la décomposition *wavelet* 2D. " style="max-width:80%" />
  <figcaption><strong>Figure 5.19:</strong>  Simulation de la décomposition *wavelet* 2D. </figcaption>
</figure>

### 5.7.4 Analyse multirésolution avec la DWT 2D

La transformée *wavelet* discrète 2D (DWT) décompose une image en composantes d'approximation et de détail, organisées de manière hiérarchique à différentes échelles et orientations. Comme les sous-bandes de détail dans les images naturelles présentent souvent des coefficients de faible contraste, les exemples pratiques suivants utilisent un motif géométrique synthétique généré en Python. Cette approche reproduit le comportement du simulateur [Figure 5.19](#fig-05-sim-05-wavelet), rendant visuellement explicites les effets du filtrage spatial et de la décomposition multirésolution.

#### 5.7.4.1 Décomposition en Mosaïque à Multiples Niveaux

La [Figure 5.20](#fig-05-dwt-subbandas) illustre la structure hiérarchique de la DWT à deux niveaux en utilisant la *wavelet* de Haar. Le processus repose sur l'application combinée de filtres passe-bas et passe-haut dans les directions horizontale et verticale, suivis d'un sous-échantillonnage par un facteur de 2.

Au premier niveau, l'image originale donne naissance à la sous-bande d'approximation ($LL_1$) ainsi qu'aux composantes de détail horizontale ($LH_1$), verticale ($HL_1$) et diagonale ($HH_1$). Dans l'analyse multirésolution, la sous-bande $LL_1$ est à nouveau filtrée et sous-échantillonnée, générant le deuxième niveau de décomposition ($LL_2$, $LH_2$, $HL_2$ et $HH_2$).

Pour faciliter l'interprétation visuelle des composantes de détail, le code extrait la valeur absolue de leurs coefficients et applique une normalisation linéaire (*min-max*) afin d'occuper toute la plage dynamique des niveaux de gris [0, 255]. Cette opération transforme les régions homogènes (coefficients nuls) en noir et met en évidence en blanc les contours et textures extraits à chaque échelle et orientation.

In [ ]:
%%writefile tmp/fig_05_dwt_subbandas.cpp
#define MM_OUT "tmp/fig_05_dwt_subbandas.png"
//| label: fig-05-dwt-subbandas
//| fig-cap: "Decomposição *wavelet* 2D de 2 níveis com *wavelet* Haar: subbandas LL, LH, HL, HH em cada nível. As subbandas de detalhe revelam estruturas orientadas em diferentes escalas utilizando um padrão sintético."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <cmath>

// Geração da Imagem Sintética (Mesmo padrão 'combined' do simulador)
cv::Mat gerar_imagem_sintetica(int N = 256) {
    cv::Mat img = cv::Mat::zeros(N, N, CV_64F);
    for (int y = 0; y < N; y++) {
        for (int x = 0; x < N; x++) {
            double v = 55 + 35 * ((double)x / N) + 15 * std::sin(y / 24.0);
            // Quadrado
            if (24 < x && x < 100 && 24 < y && y < 100) {
                v = 225;
            }
            // Círculo
            int cx = 190, cy = 76, r = 34;
            if ((x - cx) * (x - cx) + (y - cy) * (y - cy) < r * r) {
                v = 205;
            }
            // Textura periódica (inferior)
            if (y > 164 && y < 244) {
                int p = 12;
                v = (((x / p + y / p) % 2) == 0) ? 185 : 65;
            }
            // Linha diagonal
            if (std::abs(x - y) < 4) {
                v = 240;
            }
            img.at<double>(y, x) = std::max(0.0, std::min(v, 255.0));
        }
    }
    cv::Mat out;
    img.convertTo(out, CV_8U);
    return out;
}

// Função para visualizar subbanda
cv::Mat sb_vis(const cv::Mat& sb) {
    cv::Mat abs_sb;
    cv::absdiff(sb, cv::Scalar(0), abs_sb);
    cv::Mat normalized;
    cv::normalize(abs_sb, normalized, 0, 255, cv::NORM_MINMAX, CV_8U);
    return normalized;
}

int main() {
    // Substitui a imagem escura de moedas pelo padrão sintético claro
    cv::Mat img_gray = gerar_imagem_sintetica(256);

    // Decomposição wavelet 2 níveis
    std::string wavelet = "haar";
    cv::Mat img_float;
    img_gray.convertTo(img_float, CV_64F);

    // Nível 1
    mm::Subbands coefs1 = mm::dwt2(img_float, wavelet);
    cv::Mat LL1 = coefs1.LL;
    cv::Mat LH1 = coefs1.LH;
    cv::Mat HL1 = coefs1.HL;
    cv::Mat HH1 = coefs1.HH;

    // Nível 2 (aplicado sobre LL1)
    mm::Subbands coefs2 = mm::dwt2(LL1, wavelet);
    cv::Mat LL2 = coefs2.LL;
    cv::Mat LH2 = coefs2.LH;
    cv::Mat HL2 = coefs2.HL;
    cv::Mat HH2 = coefs2.HH;

    std::cout << "Forma original     : " << img_gray.rows << "x" << img_gray.cols << std::endl;
    std::cout << "LL1 (nível 1)      : " << LL1.rows << "x" << LL1.cols << "  |  LH1/HL1/HH1: " << LH1.rows << "x" << LH1.cols << std::endl;
    std::cout << "LL2 (nível 2)      : " << LL2.rows << "x" << LL2.cols << "    |  LH2/HL2/HH2: " << LH2.rows << "x" << LH2.cols << std::endl;

    std::vector<mm::Image> imgs_dwt = {mm::Image(img_gray), mm::Image(sb_vis(LL1)), mm::Image(sb_vis(LH1)), mm::Image(sb_vis(HL1)), mm::Image(sb_vis(HH1)),
                                       mm::Image(sb_vis(LL2)), mm::Image(sb_vis(LH2)), mm::Image(sb_vis(HL2)), mm::Image(sb_vis(HH2))};
    std::vector<std::string> titles_dwt = {"Original",
                                           "LL₁ (aprox.)", "LH₁ (horiz.)", "HL₁ (vert.)", "HH₁ (diag.)",
                                           "LL₂ (aprox.)", "LH₂ (horiz.)", "HL₂ (vert.)", "HH₂ (diag.)"};

    mm::show(imgs_dwt, MM_OUT, titles_dwt, 5);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_dwt_subbandas.cpp -o tmp/fig_05_dwt_subbandas -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_dwt_subbandas \
  && test -f "tmp/fig_05_dwt_subbandas.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_dwt_subbandas.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_dwt_subbandas.png"), figsize=(16, 7))

**Figure 5.20:** Decomposição *wavelet* 2D de 2 níveis com *wavelet* Haar: subbandas LL, LH, HL, HH em cada nível. As subbandas de detalhe revelam estruturas orientadas em diferentes escalas utilizando um padrão sintético.


#### 5.7.4.2 Le Compromis entre Localisation et Régularité

Le choix de la fonction de base (*ondelette*) influence directement la manière dont les caractéristiques de l’image sont distribuées et codées par les coefficients de la DWT. La [Figure 5.21](#fig-05-dwt-wavelets) compare les résultats pratiques obtenus en appliquant quatre familles distinctes sur le motif géométrique synthétique : `haar`, `db4`, `sym4` et `bior2.2`.

En raison de son support court et de sa forme en fonction échelon, l’ondelette de Haar produit des coefficients hautement localisés au niveau des discontinuités spatiales, générant des bords fins et nets dans les sous-bandes de détail. En revanche, des familles comme Daubechies (`db4`) et Symlets (`sym4`), qui présentent un support plus étendu (filtres plus longs) et un plus grand nombre de moments nuls, génèrent des réponses plus lisses et plus distribuées autour des transitions, ce qui peut introduire de légères oscillations ou un adoucissement aux frontières abruptes.

Ce comportement met en évidence le compromis classique (*trade-off*) de l’analyse multirésolution : des supports plus petits favorisent la localisation spatiale exacte des bords, tandis que des supports plus grands et un nombre plus élevé de moments nuls tendent à produire des représentations plus parcimonieuses et plus régulières. Cette régularité et cette capacité d’atténuation des hautes fréquences garantissent une plus grande efficacité dans la compaction de l’énergie, des caractéristiques fondamentales pour les applications de compression de données et de débruitage (*denoising*).

In [ ]:
%%writefile tmp/fig_05_dwt_wavelets.cpp
#define MM_OUT "tmp/fig_05_dwt_wavelets.png"
//| label: fig-05-dwt-wavelets
//| fig-cap: "Comparação entre famílias de *wavelets*: Haar, db4, sym4 e bior2.2. Subbanda LL₁ (aproximação) e HH₁ (diagonal) para cada escolha, ilustrando o compromisso entre compactação e suavidade com base no padrão sintético."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <cmath>
#include "morph.hpp"

// Função auxiliar para visualizar subbandas (normaliza para 8 bits)
mm::Image sb_vis(const cv::Mat& subband) {
    cv::Mat normalized;
    cv::normalize(subband, normalized, 0, 255, cv::NORM_MINMAX, CV_8U);
    return mm::Image(normalized);
}

int main() {
    // Garante que img_gray e img_float utilizem o mesmo padrão sintético claro
    // (fallback: gera imagem sintética se não disponível de outra célula)
    cv::Mat img_gray;
    bool synthetic_available = false;

    // Nota: como não temos a função gerar_imagem_sintetica definida em outra célula,
    // geramos diretamente aqui.
    int N = 256;
    img_gray = cv::Mat(N, N, CV_8UC1);
    for (int y = 0; y < N; y++) {
        for (int x = 0; x < N; x++) {
            double v = 55 + 35 * (double(x) / N) + 15 * std::sin(y / 24.0);
            if (x > 24 && x < 100 && y > 24 && y < 100) v = 225;
            double cx = 190, cy = 76, r = 34;
            if (std::pow(x - cx, 2) + std::pow(y - cy, 2) < r*r) v = 205;
            if (y > 164 && y < 244) {
                int p = 12;
                v = (((x / p + y / p) % 2) == 0) ? 185 : 65;
            }
            if (std::abs(x - y) < 4) v = 240;
            v = std::max(0.0, std::min(v, 255.0));
            img_gray.at<uchar>(y, x) = static_cast<uchar>(std::round(v));
        }
    }

    cv::Mat img_float;
    img_gray.convertTo(img_float, CV_64F);

    std::vector<std::string> wavelets_comp = {"haar", "db4", "sym4", "bior2.2"};
    std::vector<mm::Image> imgs_comp;
    std::vector<std::string> titles_comp;

    for (const auto& wname : wavelets_comp) {
        mm::Subbands s = mm::dwt2(img_float, wname);
        imgs_comp.push_back(sb_vis(s.LL));
        imgs_comp.push_back(sb_vis(s.HH));
        titles_comp.push_back(wname + " — LL₁");
        titles_comp.push_back(wname + " — HH₁");
    }

    mm::show(imgs_comp, MM_OUT, titles_comp, 4);
    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_dwt_wavelets.cpp -o tmp/fig_05_dwt_wavelets -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_dwt_wavelets \
  && test -f "tmp/fig_05_dwt_wavelets.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_dwt_wavelets.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_dwt_wavelets.png"), figsize=(14, 8))

**Figure 5.21:** Comparação entre famílias de *wavelets*: Haar, db4, sym4 e bior2.2. Subbanda LL₁ (aproximação) e HH₁ (diagonal) para cada escolha, ilustrando o compromisso entre compactação e suavidade com base no padrão sintético.


#### 5.7.4.3 Limiarização de Coeficientes e Compressão

Uma das principais aplicações da Transformada *Wavelet* Discreta (DWT) é a compressão de dados, impulsionada pela capacidade de representação **esparsa** dos coeficientes. A [Figure 5.22](#fig-05-dwt-reconstrucao) ilustra o efeito da limiarização abrupta (*hard thresholding*), técnica na qual coeficientes de detalhe com magnitude inferior a um limiar $T$ são integralmente anulados antes do processo de síntese realizado pela Transformada *Wavelet* Discreta Inversa (IDWT).

À medida que o limiar $T$ é elevado, um volume crescente de coeficientes de alta frequência é zerado. Por concentrarem menor energia, a remoção dessas componentes reduz consideravelmente a quantidade de informação necessária para representar a imagem, mantendo a componente de aproximação global (a subbanda $LL$ mais profunda) intacta para preservar a estrutura macro. Visualmente, esse descarte de coeficientes manifesta-se através do desaparecimento progressivo de texturas finas e da suavização de transições abruptas de intensidade.

A fidelidade da imagem reconstruída frente à original é quantificada pela métrica de **Pico da Relação Sinal-Ruído (PSNR, *Peak Signal-to-Noise Ratio*)**, expressa em decibéis (dB). Valores mais altos de PSNR indicam menor distorção e maior proximidade matemática com o sinal original. O experimento prático evidencia o decaimento gradual do PSNR conforme a agressividade da limiarização aumenta, permitindo avaliar numericamente o limiar ótimo para o balanço entre compressão e degradação visual.

#### 5.7.4.4 Seuil des Coefficients et Compression

L'une des principales applications de la Transformée *Wavelet* Discrète (DWT) est la compression de données, portée par la capacité de représentation **parcimonieuse** des coefficients. La [Figure 5.22](#fig-05-dwt-reconstrucao) illustre l'effet du seuillage abrupt (*hard thresholding*), technique dans laquelle les coefficients de détail dont la magnitude est inférieure à un seuil $T$ sont intégralement annulés avant le processus de synthèse réalisé par la Transformée *Wavelet* Discrète Inverse (IDWT).

À mesure que le seuil $T$ est augmenté, un volume croissant de coefficients haute fréquence est mis à zéro. Comme ils concentrent moins d'énergie, la suppression de ces composantes réduit considérablement la quantité d'information nécessaire pour représenter l'image, tout en maintenant intacte la composante d'approximation globale (la sous-bande $LL$ la plus profonde) afin de préserver la structure macroscopique. Visuellement, ce rejet de coefficients se manifeste par la disparition progressive des textures fines et par l'adoucissement des transitions abruptes d'intensité.

La fidélité de l'image reconstruite par rapport à l'originale est quantifiée par la métrique du **Pic du Rapport Signal sur Bruit (PSNR, *Peak Signal-to-Noise Ratio*)**, exprimée en décibels (dB). Des valeurs plus élevées de PSNR indiquent une distorsion moindre et une plus grande proximité mathématique avec le signal original. L'expérience pratique met en évidence la décroissance progressive du PSNR à mesure que l'agressivité du seuillage augmente, permettant d'évaluer numériquement le seuil optimal pour l'équilibre entre compression et dégradation visuelle.

In [ ]:
%%writefile tmp/fig_05_dwt_reconstrucao.cpp
#define MM_OUT "tmp/fig_05_dwt_reconstrucao.png"
//| label: fig-05-dwt-reconstrucao
//| fig-cap: "Reconstrução *wavelet* com limiarização de coeficientes (*hard thresholding*): à medida que o limiar aumenta, mais detalhes são zerados, produzindo imagens progressivamente mais suaves. Métrica PSNR quantifica a perda de qualidade sobre o padrão sintético."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <string>
#include <vector>
#include <cmath>
#include <algorithm>
#include "morph.hpp"

// Garante que img_gray utilize o mesmo padrão sintético claro
static cv::Mat gerar_imagem_sintetica(int N = 256) {
    cv::Mat img(N, N, CV_8UC1);
    for (int y = 0; y < N; ++y) {
        for (int x = 0; x < N; ++x) {
            double v = 55.0 + 35.0 * (static_cast<double>(x) / N) + 15.0 * std::sin(static_cast<double>(y) / 24.0);
            if (24 < x && x < 100 && 24 < y && y < 100) { v = 225.0; }
            int cx = 190, cy = 76, r = 34;
            if (std::pow(x - cx, 2) + std::pow(y - cy, 2) < r * r) { v = 205.0; }
            if (y > 164 && y < 244) {
                int p = 12;
                v = (((x / p + y / p) % 2) == 0) ? 185.0 : 65.0;
            }
            if (std::abs(x - y) < 4) { v = 240.0; }
            img.at<unsigned char>(y, x) = static_cast<unsigned char>(std::max(0.0, std::min(255.0, v)));
        }
    }
    return img;
}

static cv::Mat dwt_threshold_reconstruct(const cv::Mat& img, const std::string& wavelet = "db4", int nivel = 2, double threshold = 0.0) {
    // Decompõe, aplica limiar e reconstrói via IDWT
    cv::Mat img64f;
    img.convertTo(img64f, CV_64F);
    mm::WaveDec2 c = mm::wavedec2(img64f, wavelet, nivel);

    // Copia e aplica hard thresholding em todos os detalhes
    std::vector<std::vector<cv::Mat>> coefs_t;
    for (size_t j = 0; j < c.detail.size(); ++j) {
        std::vector<cv::Mat> sb(3);
        cv::Mat LH_t = mm::wave_threshold(c.detail[j][0], threshold, "hard");
        cv::Mat HL_t = mm::wave_threshold(c.detail[j][1], threshold, "hard");
        cv::Mat HH_t = mm::wave_threshold(c.detail[j][2], threshold, "hard");
        sb[0] = LH_t;
        sb[1] = HL_t;
        sb[2] = HH_t;
        coefs_t.push_back(sb);
    }

    // Reconstrução
    mm::WaveDec2 c_t;
    c_t.LL = c.LL;
    c_t.detail = coefs_t;
    cv::Mat rec = mm::waverec2(c_t, wavelet);

    // Recorte para dimensão original
    cv::Mat rec_cut = rec(cv::Rect(0, 0, img.cols, img.rows)).clone();
    cv::Mat rec_8u;
    cv::normalize(rec_cut, rec_8u, 0, 255, cv::NORM_MINMAX, CV_8U);
    return rec_8u;
}

int main() {
    // Cria imagem sintética de teste
    cv::Mat img_gray = gerar_imagem_sintetica(256);

    std::vector<int> thresholds = {0, 10, 30, 60, 100};
    std::vector<cv::Mat> imgs_thr;
    imgs_thr.push_back(img_gray);
    std::vector<std::string> titles_thr = {"Original"};

    for (double t : thresholds) {
        cv::Mat rec = dwt_threshold_reconstruct(img_gray, "db4", 2, t);
        double psnr = cv::PSNR(img_gray, rec);
        imgs_thr.push_back(rec);
        titles_thr.push_back("T=" + std::to_string(static_cast<int>(t)) + "  PSNR=" + 
                             std::to_string(psnr).substr(0, std::to_string(psnr).find(".") + 2) + " dB");
    }

    // Converte para mm::Image para exibição
    std::vector<mm::Image> imgs_mm;
    for (size_t i = 0; i < imgs_thr.size(); ++i) {
        imgs_mm.push_back(mm::Image(imgs_thr[i]));
    }
    mm::show(imgs_mm, MM_OUT, titles_thr, 3);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_dwt_reconstrucao.cpp -o tmp/fig_05_dwt_reconstrucao -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_dwt_reconstrucao \
  && test -f "tmp/fig_05_dwt_reconstrucao.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_dwt_reconstrucao.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_dwt_reconstrucao.png"), figsize=(14, 10))

**Figure 5.22:** Reconstrução *wavelet* com limiarização de coeficientes (*hard thresholding*): à medida que o limiar aumenta, mais detalhes são zerados, produzindo imagens progressivamente mais suaves. Métrica PSNR quantifica a perda de qualidade sobre o padrão sintético.


### Synthèse — Fourier vs. *wavelets* : quand utiliser chaque approche ?

La [Tableau 5.5](#tbl-05-fourier-wavelet) synthétise les principales différences structurelles et opérationnelles entre la transformée de Fourier discrète (TFD) et la transformée en *ondelettes* discrète (TOD).

<a id="tbl-05-fourier-wavelet"></a>

**Tabela 5.5:** Comparaison entre la transformée de Fourier discrète (TFD) et la transformée en *ondelettes* discrète (TOD), mettant en évidence leurs principales caractéristiques et applications.

| Critère | Fourier (TFD) | *Ondelettes* (TOD) |
|:---|:---|:---|
| **Fonctions de base** | Sinusoïdes à support infini | Fonctions à support compact |
| **Localisation spatiale** | Non explicite (globale) | Explicite (locale) |
| **Filtrage spectral** | Excellent pour un contrôle fin des fréquences | Basé sur des sous-bandes (échelles) |
| **Compression d'images** | Base de la TCD (JPEG traditionnel) | Base de la TOD (JPEG 2000) |
| **Analyse multi-échelle** | Non | Oui |
| **Suppression du bruit périodique** | Très efficace | Peu indiquée |
| **Signaux non stationnaires** | Limitée | Très efficace |


En termes pratiques, la TFD s'impose comme l'outil idéal pour l'analyse spectrale pure, la conception de filtres sélectifs dans le domaine fréquentiel et l'atténuation des bruits périodiques et harmoniques. En revanche, la TOD excelle dans les scénarios exigeant une préservation rigoureuse de la localisation spatiale des caractéristiques associée à leur contenu fréquentiel, se distinguant dans la compression de données, l'analyse multirésolution et le traitement des transitions abruptes. Ainsi, les deux transformées doivent être comprises comme des techniques parfaitement complémentaires, traçant des voies distinctes et spécifiques pour la résolution de problèmes en TNI-VC.

> ### 📝 Analogies avec l'audio : limites et précautions
>
> Lorsqu'on établit des analogies entre le traitement d'images et l'audio, il est important de prendre en compte les différences fondamentales :
>
> * Dans les systèmes audio stéréo/multicanaux, la phase entre les canaux est cruciale pour la perception de la localisation spatiale (différences interaurales de phase et de temps).
>
> * Dans les systèmes monauraux, la phase a une influence perceptuelle limitée — l'oreille humaine est relativement insensible à la phase absolue des composantes sinusoïdales isolées.
>
> * Dans les images, la phase de la TFD est toujours fondamentale pour la localisation spatiale des structures, qu'il s'agisse d'une image monochrome ou couleur.
>
> L'analogie entre la phase en audio et la phase en images doit être utilisée avec prudence, en soulignant que, bien que toutes deux portent des informations sur l'organisation spatiale/temporelle du signal, les mécanismes perceptuels sont fondamentalement différents.

## 5.8 Compression d'images

Alors que les *wavelets* établissent la fondation théorique du standard JPEG 2000, le standard JPEG traditionnel repose sur la **Transformée en Cosinus Discrète (DCT, *Discrete Cosine Transform*)**. Malgré les différences structurelles, les deux approches partagent le même principe fondamental : compacter l'énergie de l'image en un nombre réduit de coefficients et éliminer les composantes de moindre importance avec un impact visuel minimal.

L'objectif central de la compression est de réduire le volume de données nécessaire au stockage ou à la transmission d'une image. Ce processus est rendu possible par l'identification et l'élimination des **redondances** structurelles et perceptuelles.

### 5.8.1 Taxonomie des Redondances

Le développement des algorithmes de compression repose sur l'identification et l'élimination de trois catégories principales de redondance, synthétisées dans la [Tableau 5.6](#tbl-05-redundancias).

<a id="tbl-05-redundancias"></a>

**Tabela 5.6:** Catégories de redondance dans les images numériques et leurs mécanismes d'exploitation respectifs.

| Type | Définition | Approche d'exploitation |
|:---|:---|:---|
| **Spatiale (interpixel)** | Forte corrélation et dépendance statistique entre pixels voisins. | DCT, DWT et codage prédictif. |
| **Spectrale (intercanal)** | Corrélation statistique entre les canaux de couleur d'une même image. | Transformations d'espace colorimétrique (ex : RGB vers $YC_bC_r$). |
| **Psychovisuelle** | Insensibilité du système visuel humain (SVH) aux variations de haute fréquence et de faible contraste. | Processus de quantification sélective des coefficients. |


Selon la préservation de l'information originale après le processus de décodage, les méthodes de compression se divisent en deux classes fondamentales :

* **Sans perte (*lossless*) :** Garantit une reconstruction bit à bit identique à l'image originale. Elle est employée dans des scénarios où l'intégrité des données est strictement critique, comme dans l'imagerie médicale, les diagnostics par imagerie et le stockage de documents textuels.
* **Avec perte (*lossy*) :** Admet l'introduction d'une distorsion contrôlée du signal en échange de taux de compression substantiellement plus élevés. C'est l'approche standard pour les photographies grand public et le *streaming* vidéo, écosystèmes dans lesquels le SVH tolère de légères atténuations haute fréquence sans perception de dégradation de la qualité visuelle.

### 5.8.2 Transformée en Cosinus Discrète (DCT-II 2D)

La **Transformée en Cosinus Discrète** (DCT) constitue l'opération centrale du standard JPEG. Contrairement à la TFD, qui utilise une base complexe, la DCT repose sur des fonctions trigonométriques purement réelles. Pour un bloc d'image $f(x,y)$ de dimensions $N \times N$, la **DCT-II 2D** mappe le signal spatial vers le domaine des fréquences spatiales, générant la matrice de coefficients $C(u,v)$ au moyen de :

<a id="eq-05-dct"></a>
$$
C(u,v) = \alpha(u)\,\alpha(v) \sum_{x=0}^{N-1}\sum_{y=0}^{N-1} f(x,y)\,
\cos\!\left[\frac{\pi(2x+1)u}{2N}\right]
\cos\!\left[\frac{\pi(2y+1)v}{2N}\right] \tag{5.9}
$$


où les facteurs de normalisation orthogonale sont donnés par $\alpha(0) = \sqrt{1/N}$ et $\alpha(k) = \sqrt{2/N}$ pour $k > 0$.

Chaque coefficient $C(u,v)$ quantifie la contribution — ou le « poids » — d'une fréquence spatiale spécifique au sein de ce bloc. Le terme $C(0,0)$ est appelé **composante DC** et représente l'intensité moyenne du bloc (fréquence nulle). Les autres coefficients, désignés sous le nom de **composantes AC** (*Alternating Current*), correspondent aux fréquences spatiales progressivement plus élevées.

### 5.8.3 Les fonctions de base de la DCT

D'un point de vue géométrique, la [Équation 5.9](#eq-05-dct) réalise la projection du bloc de pixels sur un ensemble de fonctions orthogonales. Pour le cas standard du JPEG ($N=8$), le bloc spatial est décomposé en une combinaison linéaire de **64 fonctions de base** bidimensionnelles, notées $B_{u,v}(x,y)$ et générées par le produit de fonctions cosinusoïdales :

$$B_{u,v}(x,y) = \cos\left[ \frac{\pi (2x+1)u}{16} \right] \cos\left[ \frac{\pi (2y+1)v}{16} \right]$$

Ainsi, l'opération inverse peut être interprétée comme la reconstruction exacte du bloc original par la somme pondérée de ces 64 matrices de base, où chaque coefficient $C(u,v)$ agit comme le poids analytique de sa composante harmonique respective.

La **fréquence spatiale** indiquée par les indices $(u,v)$ détermine le nombre de cycles d'oscillation le long des dimensions horizontales et verticales du bloc. Comme illustré dans la [Figure 5.23](#fig-05-dct-basis) — dont le code isole chaque base en appliquant la transformation inverse sur des impulsions unitaires —, ces 64 fonctions sont organisées en une matrice $8 \times 8$. Le coin supérieur gauche ($u=0, v=0$) présente le motif uniforme de fréquence nulle (DC), tandis que la progression vers la droite (axe $u$) ou vers le bas (axe $v$) représente des variations harmoniques progressivement plus importantes, traduisant des transitions rapides, des contours et des textures dans les orientations horizontales, verticales et diagonales.

> ### 📝 DCT vs DFT : avantage de la compaction d'énergie
>
> La DCT et la DFT mappent toutes deux un bloc spatial $N \times N$ en une matrice de coefficients de même dimension. Cependant, pour les images naturelles, la DCT présente une plus grande efficacité en matière de **compaction d'énergie** dans les basses fréquences. Cela s'explique par le fait que la DCT suppose implicitement une symétrie paire du signal aux frontières du bloc, ce qui équivaut à une extension périodique continue, minimisant ainsi l'effet de diffusion spectrale (*ringing*). Par conséquent, la plupart des coefficients AC décroissent rapidement vers des valeurs proches de zéro, optimisant le *pipeline* de compression sans introduire de dégradation visuelle perceptible.

In [ ]:
%%writefile tmp/fig_05_dct_basis.cpp
#define MM_OUT "tmp/fig_05_dct_basis.png"
//| label: fig-05-dct-basis
//| fig-cap: "O Alfabeto Visual do JPEG: As 64 funções de base da DCT-II. O coeficiente DC fica no topo esquerdo (suave). Ao descer e avançar à direita, a oscilação espacial aumenta drasticamente."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include "morph.hpp"
#include <filesystem>

int main() {
    // Cada base é a IDCT de um único coeficiente unitário — montadas num mosaico 8×8.
    int tile = 32;
    cv::Mat montagem = cv::Mat::zeros(8 * tile, 8 * tile, CV_8U);
    for (int i = 0; i < 8; i++) {
        for (int j = 0; j < 8; j++) {
            cv::Mat coef = cv::Mat::zeros(8, 8, CV_64F);
            coef.at<double>(i, j) = 1.0;
            cv::Mat base = mm::idct2(coef);
            cv::Mat base_normalized;
            cv::normalize(base, base_normalized, 0, 255, cv::NORM_MINMAX, CV_8U);
            cv::Mat base_resized;
            cv::resize(base_normalized, base_resized, cv::Size(tile, tile), 0, 0, cv::INTER_NEAREST);
            base_resized.copyTo(montagem(cv::Rect(j * tile, i * tile, tile, tile)));
        }
    }

    std::string titulo = "As 64 bases da DCT-II 8×8 (DC no topo-esquerdo)";
    mm::show(std::vector<mm::Image>{mm::Image(montagem)}, MM_OUT, {titulo}, 1);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(montagem, "tmp/fig_05_dct_basis_0.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_dct_basis.cpp -o tmp/fig_05_dct_basis -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_dct_basis \
  && test -f "tmp/fig_05_dct_basis.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_dct_basis.png"

In [ ]:
mm.show(
    [
        mm.read("tmp/fig_05_dct_basis_0.png"),
    ],
    titles=[
        'As 64 bases da DCT-II 8×8 (DC no topo-esquerdo)',
    ],
    cols=1,
)

**Figure 5.23:** O Alfabeto Visual do JPEG: As 64 funções de base da DCT-II. O coeficiente DC fica no topo esquerdo (suave). Ao descer e avançar à direita, a oscilação espacial aumenta drasticamente.


### 5.8.4 Concentration d'Énergie et Reconstruction Progressive

Avant l'application de la DCT, les pixels du bloc d'intensité sont systématiquement translatés (en soustrayant $128$ pour les images 8 bits) afin de centrer le signal autour de zéro, éliminant ainsi les composantes continues superflues. Lors du calcul de la DCT sur le bloc résultant, la propriété de **compaction d'énergie** devient évidente : la quasi-totalité de la variance et de l'information de l'image originale se concentre dans le coefficient DC ($C(0,0)$) et dans les premiers harmoniques AC de basse fréquence.

La [Figure 5.24](#fig-05-dct-bloco) illustre ce phénomène par une reconstruction progressive par troncature abrupte. Au lieu d'utiliser l'ensemble des 64 coefficients, l'algorithme ne conserve que les $k$ premiers composants — sélectionnés sur la base d'un balayage qui privilégie les basses fréquences spatiales — et annule les autres.

La synthèse inverse (**IDCT**) réalisée avec seulement une fraction des coefficients (comme 15 % ou 30 %) est déjà capable de récupérer les structures et l'éclairage macro du bloc de pixels original. À mesure que les harmoniques de fréquences plus élevées sont progressivement réincorporés, les détails fins et les transitions rapides sont restaurés. Ce comportement valide le principe de la compression perceptuelle : les hautes fréquences écartées possèdent peu d'énergie et leur absence, en conditions normales, génère un impact visuel secondaire sur la perception de l'observateur.

In [ ]:
%%writefile tmp/fig_05_dct_bloco.cpp
#define MM_OUT "tmp/fig_05_dct_bloco.png"
//| label: fig-05-dct-bloco
//| fig-cap: "DCT 2D en bloc 8×8 : coefficients et reconstruction progressive."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <iostream>
#include <iomanip>
#include "morph.hpp"

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_20.png");
// [pdi:state-io:end]

    // img_gray est fourni automatiquement (mm::Image)

    // ── Bloc 8×8 centré de l'image ─────────────────────────────────────────
    int cy = img_gray.h / 2;
    int cx = img_gray.w / 2;

    // Extraction du bloc 8×8 et conversion en CV_64F avec soustraction de 128
    cv::Mat img_mat = img_gray;
    cv::Mat bloco_f;
    img_mat(cv::Rect(cx, cy, 8, 8)).convertTo(bloco_f, CV_64F);
    bloco_f -= 128.0;

    // DCT 2D du bloc
    cv::Mat C = mm::dct2(bloco_f);

    std::cout << "Coefficients DCT du bloc 8×8:" << std::endl;
    for(int i = 0; i < 8; i++) {
        for(int j = 0; j < 8; j++) {
            std::cout << std::setw(5) << std::round(C.at<double>(i,j)) << " ";
        }
        std::cout << std::endl;
    }

    // Calcul des énergies
    double energia_dc = C.at<double>(0,0) * C.at<double>(0,0);
    double energia_total = 0.0;
    for(int i = 0; i < 8; i++) {
        for(int j = 0; j < 8; j++) {
            energia_total += C.at<double>(i,j) * C.at<double>(i,j);
        }
    }

    std::cout << "\nÉnergie DC     : " << std::fixed << std::setprecision(1) << energia_dc << std::endl;
    std::cout << "Énergie totale : " << energia_total << std::endl;
    std::cout << "Fraction en DC : " << std::fixed << std::setprecision(0) 
              << (energia_dc / energia_total * 100.0) << "% ← concentration d'énergie" << std::endl;

    // ── Reconstruction progressive ──────────────────────────────────────────
    cv::Mat bloco_orig;
    bloco_f.copyTo(bloco_orig);
    bloco_orig += 128.0;
    cv::Mat bloco_orig_8u;
    bloco_orig.convertTo(bloco_orig_8u, CV_8U);

    std::vector<mm::Image> imgs_rec;
    imgs_rec.push_back(mm::Image(bloco_orig_8u));

    std::vector<std::string> titles_rec;
    titles_rec.push_back("Bloc original\n(8×8 pixels)");

    // Liste des paires (u,v) triées par somme u+v
    std::vector<std::pair<int,int>> indices;
    for(int u = 0; u < 8; u++) {
        for(int v = 0; v < 8; v++) {
            indices.push_back({u, v});
        }
    }
    std::sort(indices.begin(), indices.end(), 
              [](const std::pair<int,int>& a, const std::pair<int,int>& b) {
                  return (a.first + a.second) < (b.first + b.second);
              });

    std::vector<int> keeps = {1, 4, 10, 20, 40, 64};

    for(int keep : keeps) {
        // Création de la matrice tronquée
        cv::Mat C_trunc = cv::Mat::zeros(8, 8, CV_64F);
        for(int k = 0; k < keep; k++) {
            int u = indices[k].first;
            int v = indices[k].second;
            C_trunc.at<double>(u,v) = C.at<double>(u,v);
        }

        // Reconstruction IDCT
        cv::Mat rec_f = mm::idct2(C_trunc);
        rec_f += 128.0;

        cv::Mat rec_8u;
        rec_f.convertTo(rec_8u, CV_8U);
        cv::Mat rec_clipped;
        cv::threshold(rec_8u, rec_clipped, 255, 255, cv::THRESH_TRUNC);
        cv::threshold(rec_clipped, rec_clipped, 0, 0, cv::THRESH_TOZERO);

        imgs_rec.push_back(mm::Image(rec_clipped));

        double pct = (keep / 64.0) * 100.0;
        titles_rec.push_back(std::to_string(keep) + " coef.\n(" + 
                           std::to_string((int)pct) + "% du total)");
    }

    mm::show(imgs_rec, MM_OUT, titles_rec, 4);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_dct_bloco.cpp -o tmp/fig_05_dct_bloco -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_dct_bloco \
  && test -f "tmp/fig_05_dct_bloco.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_dct_bloco.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_dct_bloco.png"), figsize=(12, 7))

**Figure 5.24:** DCT 2D em bloco 8×8: coeficientes e reconstrução progressiva.


### 5.8.5 Le *Pipeline* de Compression JPEG

La norme JPEG opère en divisant l'image en blocs disjoints de $8 \times 8$ pixels, traités par une séquence de transformations spatiales, perceptuelles et statistiques. Le *pipeline* complet de codage est structuré en six étapes principales :

$$
\text{RGB} \xrightarrow{\text{(1) } YC_bC_r} \xrightarrow{\text{(2) Sous-échantillonnage}} \xrightarrow{\text{(3) Blocs } 8 \times 8} \xrightarrow{\text{(4) DCT}} \xrightarrow{\text{(5) Quantification}} \xrightarrow{\text{(6) Codage entropique}}
$$

La [Tableau 5.7](#tbl-05-pipeline-jpeg) détaille la fonction analytique et le fondement perceptuel qui justifient chacune de ces étapes.

<a id="tbl-05-pipeline-jpeg"></a>

**Tabela 5.7:** Étapes du *pipeline* de compression JPEG et leurs fondements de conception respectifs.

| Étape | Opération | Fondement perceptuel et statistique |
|:---:|:---|:---|
| **1** | Conversion $RGB \rightarrow YC_bC_r$ | Sépare la luminance ($Y$) de la chrominance ($C_b, C_r$). Le système visuel humain (SVH) présente une plus grande sensibilité aux variations de luminosité qu'aux variations de couleur. |
| **2** | Sous-échantillonnage de la chrominance (ex. : 4:2:0) | Réduit la résolution spatiale des canaux de couleur de moitié, en éliminant des données redondantes avec un impact visuel négligeable. |
| **3–4** | Centrage et application de la DCT $8 \times 8$ | Translates les pixels dans l'intervalle $[-128, 127]$ et compresse l'énergie spectrale du bloc dans les coefficients de basse fréquence. |
| **5** | Quantification linéaire sélective | Divise chaque coefficient $C(u,v)$ par l'élément correspondant de la matrice $Q(u,v)$, avec arrondi entier. Constitue la principale source de compression avec perte. |
| **6** | Balayage en zigzag et codage | Ordonne les coefficients quantifiés pour maximiser les séquences nulles consécutives, optimisant le codage par longueur de plage (RLE) et le codage de Huffman. |


La **matrice de quantification** $Q(u,v)$ est le mécanisme central de contrôle du compromis entre taux de compression et qualité visuelle. Dans l'algorithme pratique de la [Figure 5.25](#fig-05-jpeg-pipeline), le facteur de qualité spécifié par l'utilisateur (échelle de 1 à 100) est converti en un scalaire qui paramètre la sévérité de la matrice $Q$. Des valeurs de qualité réduites élargissent les diviseurs de $Q(u,v)$, forçant le tronquement en masse des coefficients AC à zéro. Lorsque cette élimination est excessive, la discontinuité aux frontières des blocs adjacents n'est pas atténuée lors de la reconstruction, générant ce que l'on appelle les **artefacts de bloc** (*blocking artifacts*).

#### La Logique du Balayage en Zigzag

L'efficacité du codeur entropique subséquent à la quantification dépend directement de l'ordonnancement des données. Comme la DCT concentre l'énergie vitale dans le coin supérieur gauche de la matrice (basses fréquences) et repousse les coefficients nuls vers les extrémités opposées, la lecture linéaire par lignes ou par colonnes fragmenterait les séquences de zéros.

L'ordonnancement en zigzag résout cette limitation en parcourant la matrice en diagonale, dans l'ordre croissant de la fréquence spatiale. Ce mappage regroupe les coefficients significatifs au début du vecteur et concentre les coefficients nuls en une seule séquence continue à la fin du tableau, permettant à l'algorithme RLE de coder de grands blocs de données de manière compacte et efficace.

> ### 📝 5.9 Qu'est-ce que le RLE ?
>
> **RLE** (*Run-Length Encoding*) est une technique de compression sans perte qui code des séquences consécutives de valeurs identiques — en particulier des **zéros** — comme une paire (compteur, valeur). Dans le JPEG, après le balayage en zigzag, les coefficients quantifiés sont organisés de sorte que les zéros se concentrent à la fin du vecteur. Le RLE compresse ensuite cette longue course de zéros avec une extrême efficacité, optimisant le stockage et la transmission de l'image compressée.

In [ ]:
%%writefile tmp/fig_05_jpeg_pipeline.cpp
#define MM_OUT "tmp/fig_05_jpeg_pipeline.png"
#include <opencv2/opencv.hpp>
#include "morph.hpp"
#include <string>
#include <vector>

int main() {
    //| label: fig-05-jpeg-pipeline
    //| fig-cap: "*Pipeline* JPEG simplifié appliqué à l'image classique du *Cameraman* : DCT en blocs 8×8, quantification avec différents facteurs de qualité et reconstruction via IDCT. Les artefacts de bloc (*blocking artifacts*) deviennent visuellement évidents pour des facteurs de qualité réduits ($Q=10$ et $Q=25$)."
    //| echo: true
    //| output: true

    // Pipeline JPEG (DCT 8x8 -> quantification -> IDCT) via mm.jpegCompress,
    // sur l'image classique du Cameraman (asset du chapitre) réduite à 256x256.
    mm::Image img_orig = mm::read("imagens/cameraman.png");
    cv::Mat img_resized;
    cv::resize(cv::Mat(img_orig), img_resized, cv::Size(256, 256));
    mm::Image img_src = mm::gray(mm::Image(img_resized));

    std::vector<mm::Image> imgs;
    std::vector<std::string> titles;
    imgs.push_back(img_src);
    titles.push_back("Original (Cameraman)");

    std::vector<int> qualities = {10, 25, 50, 75, 90};
    for (int q : qualities) {
        mm::Image rec = mm::jpegCompress(img_src, q);
        imgs.push_back(rec);
        double psnr_val = mm::psnr(img_src, rec);
        titles.push_back("Q=" + std::to_string(q) + " (PSNR=" + 
                         std::to_string(psnr_val).substr(0, std::to_string(psnr_val).find(".") + 2) + " dB)");
    }

    mm::show(imgs, MM_OUT, titles, 3);

    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_jpeg_pipeline.cpp -o tmp/fig_05_jpeg_pipeline -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_jpeg_pipeline \
  && test -f "tmp/fig_05_jpeg_pipeline.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_jpeg_pipeline.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_jpeg_pipeline.png"), figsize=(14, 10))

**Figure 5.25:** *Pipeline* JPEG simplificado aplicado à imagem clássica do *Cameraman*: DCT em blocos 8×8, quantização com diferentes fatores de qualidade e reconstrução via IDCT. Os artefatos de bloco (*blocking artifacts*) tornam-se visualmente evidentes em fatores de qualidade reduzidos ($Q=10$ e $Q=25$).


### 5.9.1 Simulateur interactif : Quantification DCT

Le simulateur de la [Figure 5.26](#fig-05-sim-05-dct) permet d'explorer l'impact du processus de quantification sur un bloc $8 \times 8$ extrait d'une image réelle, en synthétisant en temps réel les composantes suivantes :

* **Bloc original et reconstruit :** Représentation directe des pixels dans le domaine spatial en niveaux de gris [0, 255].
* **Coefficients DCT :** Distribution de l'énergie mappée de manière logarithmique sur un dégradé chromatique, mettant en évidence la concentration de l'intensité dans le coin supérieur gauche (basses fréquences).
* **Coefficients quantifiés :** Affichage des valeurs entières résultant de la division par la matrice $Q(u,v)$, rendant visuellement explicite l'apparition massive de coefficients nuls (en tons sombres) à mesure que le facteur de qualité est réduit.
* **Métriques de compression :** Panneau de surveillance qui quantifie l'Erreur Quadratique Moyenne (MSE), le nombre de coefficients préservés et le volume de zéros générés pour le codage entropique.

In [ ]:
from IPython.display import HTML

HTML("""
<div id="sim-05-dct" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">
<style>
  #sim-05-dct * { box-sizing: border-box; }
  #sim-05-dct canvas { display: block; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; }
  #sim-05-dct button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; }
  #sim-05-dct button:hover { background: #e8dfcf; }
  #sim-05-dct .sim04_dct_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim04_dct_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim04_dct_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; flex: 1; min-width: 90px; }
  .sim04_dct_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim04_dct_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim04_dct_slider_container { display: flex; align-items: center; gap: 8px; margin-bottom: 10px; }
  .sim04_dct_slider_container label { font-size: 11px; font-weight: 700; color: #5e5a4a; min-width: 110px; }
  .sim04_dct_slider_container input[type=range] { flex: 1; cursor: pointer; height: 4px; accent-color: #2980b9; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">⊞ Simulateur : Quantification DCT-JPEG (bloc 8×8)</span>
  <span class="sim04_dct_pill">blocs 8×8</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div style="display:flex; gap:10px; margin-bottom:14px; flex-wrap:wrap;">
    <div class="sim04_dct_stat_box"><div class="sim04_dct_stat_label">Qualité</div><div id="sim04_dct_qual" class="sim04_dct_stat_value" style="color:#2980b9;">50</div></div>
    <div class="sim04_dct_stat_box"><div class="sim04_dct_stat_label">Coef. ≠ 0</div><div id="sim04_dct_nonzero" class="sim04_dct_stat_value" style="color:#27ae60;">–</div></div>
    <div class="sim04_dct_stat_box"><div class="sim04_dct_stat_label">Zéros</div><div id="sim04_dct_zeros" class="sim04_dct_stat_value" style="color:#c0392b;">–</div></div>
    <div class="sim04_dct_stat_box"><div class="sim04_dct_stat_label">Erreur MSE</div><div id="sim04_dct_mse" class="sim04_dct_stat_value" style="color:#b9770e;">–</div></div>
  </div>

  <!-- Grid de Visualização dos Blocos -->
  <div style="display:flex; gap:12px; flex-wrap:wrap; align-items:flex-start; justify-content:center; margin-bottom:14px;">
    <div class="sim04_dct_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; margin-bottom:6px; font-weight:700;">Bloc original (8×8)</div>
      <canvas id="sim04_dct_cvOrig" width="160" height="160" style="margin:0 auto;"></canvas>
    </div>
    <div class="sim04_dct_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; margin-bottom:6px; font-weight:700;">Coef. DCT (abs, log)</div>
      <canvas id="sim04_dct_cvDCT" width="160" height="160" style="margin:0 auto;"></canvas>
    </div>
    <div class="sim04_dct_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; margin-bottom:6px; font-weight:700;">Coef. quantifiés</div>
      <canvas id="sim04_dct_cvQuant" width="160" height="160" style="margin:0 auto;"></canvas>
    </div>
    <div class="sim04_dct_panel" style="text-align:center;">
      <div style="font-size:10.5px; color:#5e5a4a; margin-bottom:6px; font-weight:700;">Bloc reconstruit</div>
      <canvas id="sim04_dct_cvRec" width="160" height="160" style="margin:0 auto;"></canvas>
    </div>
  </div>

  <!-- Controles -->
  <div class="sim04_dct_panel">
    <div class="sim04_dct_slider_container">
      <label>Qualité JPEG :</label>
      <input type="range" id="sim04_dct_slider" min="1" max="100" value="50">
      <span id="sim04_dct_slVal" style="font-size:12px; font-family:monospace; font-weight:700; min-width:30px; color:#26241d;">50</span>
    </div>
    <div style="display:flex; gap:6px; flex-wrap:wrap;">
      <button data-q="10" style="flex:1;">Q=10</button>
      <button data-q="25" style="flex:1;">Q=25</button>
      <button data-q="50" style="flex:1;">Q=50</button>
      <button data-q="75" style="flex:1;">Q=75</button>
      <button data-q="95" style="flex:1;">Q=95</button>
    </div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim04DCT(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const dct_block = [
      [52,55,61,66,70,61,64,73],
      [63,59,55,90,109,85,69,72],
      [62,59,68,113,144,104,66,73],
      [63,58,71,122,154,106,70,69],
      [67,61,68,104,126,88,68,70],
      [79,65,60,70,77,68,58,75],
      [85,71,64,59,55,61,65,83],
      [87,79,69,68,65,76,78,94]
    ];

    const Q_luma = [
      [16,11,10,16,24,40,51,61],[12,12,14,19,26,58,60,55],
      [14,13,16,24,40,57,69,56],[14,17,22,29,51,87,80,62],
      [18,22,37,56,68,109,103,77],[24,35,55,64,81,104,113,92],
      [49,64,78,87,103,121,120,101],[72,92,95,98,112,100,103,99]
    ];

    function dct1d(x) {
      const N = x.length, c = new Array(N).fill(0);
      for (let k = 0; k < N; k++) {
        let sum = 0;
        for (let n = 0; n < N; n++) sum += x[n] * Math.cos(Math.PI*(2*n+1)*k/(2*N));
        const alpha = k===0 ? Math.sqrt(1/N) : Math.sqrt(2/N);
        c[k] = alpha * sum;
      }
      return c;
    }

    function idct1d(c) {
      const N = c.length, x = new Array(N).fill(0);
      for (let n = 0; n < N; n++) {
        let sum = 0;
        for (let k = 0; k < N; k++) {
          const alpha = k===0 ? Math.sqrt(1/N) : Math.sqrt(2/N);
          sum += alpha * c[k] * Math.cos(Math.PI*(2*n+1)*k/(2*N));
        }
        x[n] = sum;
      }
      return x;
    }

    function dct2d(blk) {
      const N=8, rows=blk.map(r=>dct1d(r));
      const cols=[];
      for(let j=0;j<N;j++){const col=rows.map(r=>r[j]);cols.push(dct1d(col));}
      const out=Array.from({length:N},()=>new Array(N));
      for(let i=0;i<N;i++) for(let j=0;j<N;j++) out[i][j]=cols[j][i];
      return out;
    }

    function idct2d(C) {
      const N=8, cols=[];
      for(let j=0;j<N;j++){const col=C.map(r=>r[j]);cols.push(idct1d(col));}
      const rows=Array.from({length:N},()=>new Array(N));
      for(let i=0;i<N;i++) for(let j=0;j<N;j++) rows[i][j]=cols[j][i];
      return rows.map(r=>idct1d(r));
    }

    function getQ(quality) {
      const s = quality<50 ? 5000/quality : 200-2*quality;
      return Q_luma.map(row=>row.map(v=>Math.max(1,Math.min(255,Math.round(v*s/100)))));
    }

    function drawPixels(canvas, data, minV, maxV) {
      const ctx = canvas.getContext('2d');
      const sz = canvas.width/8;
      for(let i=0;i<8;i++) for(let j=0;j<8;j++){
        const v = (data[i][j]-minV)/(maxV-minV);
        const g = Math.round(v*255);
        ctx.fillStyle='rgb('+g+','+g+','+g+')';
        ctx.fillRect(j*sz,i*sz,sz,sz);
        ctx.strokeStyle='#e4dcc8'; ctx.lineWidth=0.5;
        ctx.strokeRect(j*sz,i*sz,sz,sz);
        ctx.fillStyle=g>128?'#26241d':'#fafaf7';
        ctx.font='bold ' + Math.round(sz*0.28) + 'px monospace';
        ctx.textAlign='center'; ctx.textBaseline='middle';
        const val = data[i][j];
        ctx.fillText(Math.abs(val)>999?'…':String(Math.round(val)), j*sz+sz/2, i*sz+sz/2);
      }
    }

    function drawHeatmap(canvas, data) {
      const ctx = canvas.getContext('2d');
      const sz = canvas.width/8;
      const flat=data.flat(); const mn=Math.min(...flat), mx=Math.max(...flat);
      for(let i=0;i<8;i++) for(let j=0;j<8;j++){
        const v=(data[i][j]-mn)/(mx-mn||1);
        const r=Math.round(v*220+35), gb=Math.round((1-v)*180+30);
        ctx.fillStyle='rgb('+r+','+gb+','+gb+')';
        ctx.fillRect(j*sz,i*sz,sz,sz);
        ctx.strokeStyle='#e4dcc8'; ctx.lineWidth=0.5;
        ctx.strokeRect(j*sz,i*sz,sz,sz);
        ctx.fillStyle='#ffffff'; ctx.font='bold ' + Math.round(sz*0.22) + 'px monospace';
        ctx.textAlign='center'; ctx.textBaseline='middle';
        const val = data[i][j];
        ctx.fillText(Math.abs(val)>999?'…':String(Math.round(val)), j*sz+sz/2, i*sz+sz/2);
      }
    }

    function update(quality) {
      const Q = getQ(quality);
      const centered = dct_block.map(r=>r.map(v=>v-128));
      const C = dct2d(centered);
      const Cq = C.map((r,i)=>r.map((v,j)=>Math.round(v/Q[i][j])));
      const Cdq = Cq.map((r,i)=>r.map((v,j)=>v*Q[i][j]));
      const rec = idct2d(Cdq).map(r=>r.map(v=>Math.max(0,Math.min(255,Math.round(v+128)))));

      const Clog = C.map(r=>r.map(v=>Math.log1p(Math.abs(v))*(v>=0?1:-1)));
      const Cqlog = Cq.map(r=>r.map(v=>v));

      let nz=0, mse=0;
      for(let i=0;i<8;i++) for(let j=0;j<8;j++){
        if(Cq[i][j]!==0) nz++;
        mse+=(dct_block[i][j]-rec[i][j])**2;
      }
      mse/=64;

      drawPixels(root.querySelector('#sim04_dct_cvOrig'), dct_block, 0, 255);
      drawHeatmap(root.querySelector('#sim04_dct_cvDCT'), Clog);
      drawHeatmap(root.querySelector('#sim04_dct_cvQuant'), Cqlog);
      drawPixels(root.querySelector('#sim04_dct_cvRec'), rec, 0, 255);

      root.querySelector('#sim04_dct_qual').textContent = quality;
      root.querySelector('#sim04_dct_nonzero').textContent = nz;
      root.querySelector('#sim04_dct_zeros').textContent = (64-nz);
      root.querySelector('#sim04_dct_mse').textContent = mse.toFixed(1);
    }

    window.dct_setQ = function(q){
      root.querySelector('#sim04_dct_slider').value = q;
      root.querySelector('#sim04_dct_slVal').textContent = q;
      update(q);
    };

    root.querySelector('#sim04_dct_slider').addEventListener('input', function(){
      root.querySelector('#sim04_dct_slVal').textContent = this.value;
      update(+this.value);
    });

    root.querySelectorAll('[data-q]').forEach(btn => {
      btn.addEventListener('click', function() {
        dct_setQ(parseInt(this.getAttribute('data-q'), 10));
      });
    });

    update(50);
  }

  function tryInitSim04DCT(){
    var root = document.getElementById('sim-05-dct');
    if (root) initSim04DCT(root); else setTimeout(tryInitSim04DCT, 200);
  }
  tryInitSim04DCT();
})();
</script>
""")

**Figure 5.26:** Simulateur interactif de compression DCT-JPEG : ajustez le facteur de qualité et visualisez en temps réel les coefficients nuls, le bloc reconstruit et l


<figure id="fig-05-sim-05-dct">
  <img src="imagens/fig-05-sim-05-dct.png" alt=" Simulateur interactif de compression DCT-JPEG : ajustez le facteur de qualité et visualisez en temps réel les coefficients nuls, le bloc reconstruit et l'erreur de quantification. " style="max-width:80%" />
  <figcaption><strong>Figure 5.26:</strong>  Simulateur interactif de compression DCT-JPEG : ajustez le facteur de qualité et visualisez en temps réel les coefficients nuls, le bloc reconstruit et l'erreur de quantification. </figcaption>
</figure>

## 5.10 Comparação de Formatos de Imagem

Le choix d'un format de stockage numérique impacte directement le compromis entre qualité visuelle, taille de fichier et coût computationnel de décodage. Les trois formats les plus pertinents pour les architectures *web* et les systèmes de calcul visuel sont le JPEG, le PNG et le WebP.

### 5.10.1 Caractéristiques des Formats

La [Tableau 5.8](#tbl-05-formatos) synthétise les propriétés structurelles des principaux formats d'image matriciels.

<a id="tbl-05-formatos"></a>

**Tabela 5.8:** Comparaison structurelle entre les principaux formats d'image matriciels.

| Caractéristique | JPEG | PNG | WebP |
|:---|:---:|:---:|:---:|
| **Compression** | Avec perte | Sans perte | Avec et sans perte. |
| **Transparence (canal alpha)** | Non | Oui | Oui. |
| **Prise en charge de l'animation** | Non | Limitée (APNG) | Oui. |
| **Algorithme de base** | DCT + Huffman | DEFLATE (LZ77 + Huffman) | VP8 / VP8L. |
| **Idéal pour** | Photographie | Graphiques, texte et icônes | Usage universel en environnement Web. |
| **Moins adapté pour** | Texte et contours nets | Images photographiques complexes | Compatibilité héritée. |


### 5.10.2 Métriques d'Évaluation de la Qualité

Deux métriques objectives sont largement adoptées pour quantifier la distorsion introduite par les processus de compression :

**Pic du Rapport Signal-à-Bruit (PSNR, *Peak Signal-to-Noise Ratio*) :**
<a id="eq-05-psnr"></a>
$$
\text{PSNR} = 10\,\log_{10}\!\left(\frac{L^2}{\text{MSE}}\right) \quad [\text{dB}] \tag{5.10}
$$


où $L = 255$ pour les images quantifiées sur 8 bits et $\text{MSE}$ représente l'**Erreur Quadratique Moyenne** (*Mean Squared Error*). Des valeurs de PSNR supérieures à 40 dB indiquent une excellente fidélité ; entre 30 dB et 40 dB, elles représentent une bonne qualité ; et des valeurs inférieures à 30 dB correspondent à des dégradations visuelles facilement perceptibles.

**Indice de Similarité Structurale (SSIM, *Structural Similarity Index*) :**
<a id="eq-05-ssim"></a>
$$
\text{SSIM}(f,g) = \frac{(2\mu_f\mu_g + c_1)(2\sigma_{fg} + c_2)}{(\mu_f^2+\mu_g^2+c_1)(\sigma_f^2+\sigma_g^2+c_2)} \tag{5.11}
$$


Le SSIM évalue des fenêtres locales de l'image sur la base de trois composantes complémentaires : la **luminance** ($\mu_f, \mu_g$), le **contraste** ($\sigma_f, \sigma_g$) et la **structure** ($\sigma_{fg}$), pondérées par des constantes de stabilité $c_1$ et $c_2$. L'indice varie dans l'intervalle $[-1, 1]$, où l'unité représente l'identité parfaite. Contrairement au PSNR, le SSIM prend en compte l'organisation spatiale des erreurs, s'alignant ainsi sur la perception du système visuel humain (SVH).

> ### 📝 PSNR vs SSIM : Application de Métriques Perceptuelles
>
> Le PSNR possède une formulation mathématique simple et un faible coût computationnel ; toutefois, il tend à surestimer la qualité des images présentant des distorsions localisées ou à la sous-estimer en cas de variations globales de luminosité tolérées par l'observateur. Le SSIM modélise plus fidèlement la perception biologique, mais exige un effort de traitement plus important. Pour des analyses rigoureuses des codecs, il est recommandé de rapporter les deux métriques statistiques à titre complémentaire.

### 5.10.3 Inspection Visuelle : Nature des Artefacts de Compression

La nature mathématique du codec détermine le type de dégradation introduit à des débits binaires réduits. Comme illustré dans la [Figure 5.27](#fig-05-zoom-artefatos), la compression agressive via DCT dans le standard JPEG segmente l'image en mailles rigides, générant les **artefacts de bloc** (*blocking artifacts*). En revanche, les algorithmes basés sur la codification prédictive ou les représentations soumises à des transformées spatiales avancées (comme le WebP et le JPEG 2000) éliminent les discontinuités de bloc, mais introduisent une perte de texture fine et des flous caractéristiques autour des contours à fort contraste.

In [ ]:
%%writefile tmp/fig_05_zoom_artefatos.cpp
#define MM_OUT "tmp/fig_05_zoom_artefatos.png"
//| label: fig-05-zoom-artefatos
//| fig-cap: "Análise comparativa de artefatos de compressão sob fator de qualidade reduzido ($Q=10$). À esquerda, observa-se o artefato de bloco característico da discretização por DCT no JPEG. À direita, evidencia-se o efeito de atenuação e suavização de bordas intrínseco ao padrão WebP."
//| echo: true
//| output: true
#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <filesystem>
#include "morph.hpp"

int main() {
    // Lê a imagem, converte para escala de cinza e redimensiona
    cv::Mat img_src_mat = cv::imread("imagens/cameraman.png", cv::IMREAD_GRAYSCALE);
    cv::Mat img_src_resized;
    cv::resize(img_src_mat, img_src_resized, cv::Size(256, 256));
    mm::Image img_src = img_src_resized;

    // Salva com qualidade Q=10 em JPEG e WebP
    cv::imwrite("tmp/zoom_q10.jpg", cv::Mat(img_src), {cv::IMWRITE_JPEG_QUALITY, 10});
    cv::imwrite("tmp/zoom_q10.webp", cv::Mat(img_src), {cv::IMWRITE_WEBP_QUALITY, 10});

    // Função de zoom com interpolação vizinho mais próximo
    auto zoom = [](const cv::Mat& img) {
        cv::Mat crop = img(cv::Rect(150, 120, 80, 80));
        cv::Mat enlarged;
        cv::resize(crop, enlarged, cv::Size(320, 320), 0, 0, cv::INTER_NEAREST);
        return enlarged;
    };

    // Aplica zoom nas três imagens
    cv::Mat z_orig = zoom(cv::Mat(img_src));
    cv::Mat z_jpeg = zoom(cv::imread("tmp/zoom_q10.jpg", cv::IMREAD_GRAYSCALE));
    cv::Mat z_webp = zoom(cv::imread("tmp/zoom_q10.webp", cv::IMREAD_GRAYSCALE));

    // Exibe as três imagens lado a lado
    std::vector<mm::Image> images = {
        mm::Image(z_orig),
        mm::Image(z_jpeg),
        mm::Image(z_webp)
    };
    std::vector<std::string> titles = {
        "Zoom Original",
        "JPEG Q=10 (Artefato de Bloco)",
        "WebP Q=10 (Suavizacao)"
    };
    mm::show(images, MM_OUT, titles, 3);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(z_orig, "tmp/fig_05_zoom_artefatos_0.png");
mm::write(z_jpeg, "tmp/fig_05_zoom_artefatos_1.png");
mm::write(z_webp, "tmp/fig_05_zoom_artefatos_2.png");
// [pdi:panel-io:end]
return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_zoom_artefatos.cpp -o tmp/fig_05_zoom_artefatos -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_zoom_artefatos \
  && test -f "tmp/fig_05_zoom_artefatos.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_zoom_artefatos.png"

In [ ]:
mm.show(
    [
        mm.read("tmp/fig_05_zoom_artefatos_0.png"),
        mm.read("tmp/fig_05_zoom_artefatos_1.png"),
        mm.read("tmp/fig_05_zoom_artefatos_2.png"),
    ],
    titles=[
        'Zoom Original',
        'JPEG Q=10 (Artefato de Bloco)',
        'WebP Q=10 (Suavizacao)',
    ],
    cols=3,
    figsize=(14, 5),
)

**Figure 5.27:** Análise comparativa de artefatos de compressão sob fator de qualidade reduzido ($Q=10$). À esquerda, observa-se o artefato de bloco característico da discretização por DCT no JPEG. À direita, evidencia-se o efeito de atenuação e suavização de bordas intrínseco ao padrão WebP.


### 5.10.4 Évaluation Quantitative et Spatiale de la Compression

La validation des algorithmes de compression avec perte exige une analyse qui corrèle le coût de stockage à la fidélité du signal reconstruit. Cette évaluation est réalisée de manière complémentaire à travers des courbes de performance globale et par la cartographie locale des distorsions induites par les codeurs.

#### 5.10.4.1 Courbes Débit-Distorsion

La [Figure 5.28](#fig-05-formatos-comparacao) présente l'évaluation empirique du *pipeline* JPEG et WebP au moyen de **courbes débit-distorsion**, qui surveillent le gain de compression (taille du fichier en Ko) en fonction du PSNR. Le format PNG sert de ligne de base idéale ($\text{PSNR} = \infty$), car sa nature *lossless* empêche toute dégradation, bien qu'il exige un volume de données substantiellement plus important.

L'analyse des courbes démontre la supériorité et l'efficacité du standard WebP par rapport au JPEG traditionnel : pour atteindre un même niveau de fidélité mathématique (comme la plage d'excellente qualité, où $\text{PSNR} > 40\text{ dB}$), le codeur WebP génère des fichiers significativement plus petits. Ce comportement traduit l'impact pratique de l'évolution des algorithmes sur l'optimisation des systèmes de transmission et de stockage numérique.

In [ ]:
import cv2
import os, cv2

src = mm.gray(cv2.resize(mm.read("imagens/cameraman.png"), (256, 256)))

jpeg_kb, jpeg_psnr = [], []
for q in [10, 20, 30, 40, 50, 60, 70, 80, 90, 95]:
    p = f"tmp/fmt_q{q}.jpg"
    cv2.imwrite(p, src, [cv2.IMWRITE_JPEG_QUALITY, q])
    rec = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    jpeg_kb.append(os.path.getsize(p) / 1024.0)
    jpeg_psnr.append(mm.psnr(src, rec))

webp_kb, webp_psnr = [], []
for q in [30, 50, 70, 85, 95]:
    p = f"tmp/fmt_w{q}.webp"
    cv2.imwrite(p, src, [cv2.IMWRITE_WEBP_QUALITY, q])
    rec = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    webp_kb.append(os.path.getsize(p) / 1024.0)
    webp_psnr.append(mm.psnr(src, rec))

p_png = "tmp/fmt.png"
cv2.imwrite(p_png, src, [cv2.IMWRITE_PNG_COMPRESSION, 9])
png_kb = os.path.getsize(p_png) / 1024.0

chart = mm.lineChart(
    [jpeg_kb, webp_kb], [jpeg_psnr, webp_psnr],
    labels=["JPEG", "WebP"],
    title=f"Curva Taxa-Distorcao (PNG sem perda: {png_kb:.1f} KB)",
    xlabel="Tamanho do arquivo (KB)", ylabel="PSNR (dB)"
)
mm.show([chart], titles=["JPEG x WebP x PNG"], cols=1)


**Figure 5.28:** Courbe taux-distorsion : PSNR vs taille de fichier pour JPEG, WebP et PNG appliquée à l’image du *Cameraman*.


> ### 📝 5.11 Taille originale de l'image
>
> L'image *Cameraman* ($256 \times 256$ pixels en niveaux de gris) occupe **64 Ko** en format brut (sans compression). À titre de référence, le PNG *lossless* compresse ce volume à **36,2 Ko** — mettant en évidence que la compression sans perte réduit déjà significativement le stockage pour les images comportant des régions homogènes. En revanche, les formats avec perte (JPEG et WebP) atteignent des tailles encore plus réduites : le JPEG avec une qualité de 95 occupe 22,3 Ko (PSNR ≈ 45 dB), tandis que le WebP avec une qualité de 90 atteint 12,5 Ko avec un PSNR équivalent, démontrant sa supériorité en matière d'efficacité de compression.

#### 5.11.0.1 Cartographie Spatiale des Erreurs et Corrélation Perceptuelle

Bien que le PSNR offre un indicateur numérique rapide, les métriques globales ne parviennent pas à distinguer comment la perte d'informations se répartit géométriquement sur l'image. La [Figure 5.29](#fig-05-ssim-artefatos) résout cette limitation en associant les reconstructions à différentes qualités à leurs cartes d'erreur absolue respectives et au SSIM.

Les cartes résiduelles — obtenues par la différence absolue normalisée entre l'image originale et l'image compressée — révèlent la signature spatiale intrinsèque de chaque architecture de codage :

* **À des qualités élevées ($Q=95$ à $Q=75$) :** Les distorsions se concentrent principalement autour des transitions abruptes d'intensité (bords), résultant du repliement spectral dû à l'élimination des hautes fréquences. L'indice SSIM reste proche de l'unité, attestant de l'intégrité des structures originales.
* **À des qualités agressives ($Q=50$ à $Q=25$) :** L'erreur adopte une structure de maillage orthogonal régularisé. Ce motif géométrique met en évidence l'apparition des **artefacts de bloc** (*blocking artifacts*), indiquant que la quantification sévère a corrompu la corrélation spatiale entre les blocs adjacents de $8 \times 8$ pixels.

Le SSIM capture cette dégradation morphologique de manière beaucoup plus sensible que le PSNR, pénalisant le score final à mesure que l'organisation structurelle et les textures fines — auxquelles le système visuel humain est hautement réactif — sont éliminées par le codeur.

In [ ]:
%%writefile tmp/fig_05_ssim_artefatos.cpp
#define MM_OUT "tmp/fig_05_ssim_artefatos.png"
//| label: fig-05-ssim-artefatos
//| fig-cap: "Analyse spatiale de la dégradation : images reconstruites et cartes d'erreur absolue normalisées pour différents facteurs de qualité JPEG."
//| echo: true
//| output: true

#include <opencv2/opencv.hpp>
#include <vector>
#include <string>
#include <cmath>
#include "morph.hpp"

// Fonction SSIM (Wang et al., 2004) — version fenêtrée avec filtre gaussien
// Retourne la valeur moyenne SSIM et la carte SSIM complète.
static double compute_ssim_full(const cv::Mat& a, const cv::Mat& b, cv::Mat& ssim_map) {
    // Convertir en flottant 64 bits
    cv::Mat A, B;
    a.convertTo(A, CV_64F);
    b.convertTo(B, CV_64F);

    const double C1 = 6.5025;  // (0.01 * 255)^2
    const double C2 = 58.5225; // (0.03 * 255)^2

    // Filtre gaussien 11x11, sigma = 1.5
    cv::Mat mu1, mu2, mu1_sq, mu2_sq, mu1_mu2, sigma1_sq, sigma2_sq, sigma12;
    cv::GaussianBlur(A, mu1, cv::Size(11, 11), 1.5);
    cv::GaussianBlur(B, mu2, cv::Size(11, 11), 1.5);
    cv::multiply(mu1, mu1, mu1_sq);
    cv::multiply(mu2, mu2, mu2_sq);
    cv::multiply(mu1, mu2, mu1_mu2);

    cv::Mat A_sq, B_sq, A_B;
    cv::multiply(A, A, A_sq);
    cv::multiply(B, B, B_sq);
    cv::multiply(A, B, A_B);

    cv::Mat sigma1_sq_tmp, sigma2_sq_tmp, sigma12_tmp;
    cv::GaussianBlur(A_sq, sigma1_sq_tmp, cv::Size(11, 11), 1.5);
    cv::GaussianBlur(B_sq, sigma2_sq_tmp, cv::Size(11, 11), 1.5);
    cv::GaussianBlur(A_B, sigma12_tmp, cv::Size(11, 11), 1.5);

    cv::subtract(sigma1_sq_tmp, mu1_sq, sigma1_sq);
    cv::subtract(sigma2_sq_tmp, mu2_sq, sigma2_sq);
    cv::subtract(sigma12_tmp, mu1_mu2, sigma12);

    cv::Mat cs_map = (2 * sigma12 + C2) / (sigma1_sq + sigma2_sq + C2);
    cv::Mat ssim_map_tmp = ((2 * mu1_mu2 + C1) / (mu1_sq + mu2_sq + C1));
    cv::multiply(ssim_map_tmp, cs_map, ssim_map);

    // Valeur moyenne
    cv::Scalar mssim = cv::mean(ssim_map);
    return mssim[0];
}

int main() {
    // Cameraman (asset du chapitre), 256x256 — même image que la piste py.
    std::string img_path = "imagens/cameraman.png";
    cv::Mat img_full = cv::imread(img_path, cv::IMREAD_GRAYSCALE);
    if (img_full.empty()) {
        // Fallback : lire via mm::read si le chemin direct échoue
        mm::Image tmp = mm::read(img_path);
        img_full = cv::Mat(tmp.h, tmp.w, CV_8UC1, tmp.data.data());
    }
    cv::Mat img_resized;
    cv::resize(img_full, img_resized, cv::Size(256, 256));
    mm::Image img_gray = img_resized;

    std::vector<mm::Image> imgs_ssim;
    std::vector<std::string> titles_ssim;
    imgs_ssim.push_back(img_gray);
    titles_ssim.push_back("Original");

    for (int q : {25, 50, 75, 95}) {
        // DCT 8x8 -> quantisation -> IDCT
        mm::Image rec = mm::jpegCompress(img_gray, q);
        double psnr_v = mm::psnr(img_gray, rec);

        // Convertir en cv::Mat pour le calcul SSIM
        cv::Mat img_cv = img_gray;   // conversion implicite
        cv::Mat rec_cv = rec;        // conversion implicite

        cv::Mat ssim_map;
        double ssim_v = compute_ssim_full(img_cv, rec_cv, ssim_map);

        // Carte d'erreur absolue normalisée (0..255)
        cv::Mat img_f, rec_f;
        img_cv.convertTo(img_f, CV_64F);
        rec_cv.convertTo(rec_f, CV_64F);
        cv::Mat diff = cv::abs(img_f - rec_f);
        cv::Mat diff_norm;
        cv::normalize(diff, diff_norm, 0, 255, cv::NORM_MINMAX);
        cv::Mat diff_vis;
        diff_norm.convertTo(diff_vis, CV_8U);

        imgs_ssim.push_back(rec);
        imgs_ssim.push_back(mm::Image(diff_vis));
        titles_ssim.push_back("Q=" + std::to_string(q) + " (PSNR=" + 
                             std::to_string(psnr_v).substr(0, 3) + "dB | SSIM=" + 
                             std::to_string(ssim_v).substr(0, 4) + ")");
        titles_ssim.push_back("Carte d'erreur (Q=" + std::to_string(q) + 
                             ") - bords et blocage");
    }

    mm::show(imgs_ssim, MM_OUT, titles_ssim, 3);
    return 0;
}

In [ ]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_05_ssim_artefatos.cpp -o tmp/fig_05_ssim_artefatos -lopencv_stitching -lopencv_alphamat -lopencv_aruco -lopencv_barcode -lopencv_bgsegm -lopencv_bioinspired -lopencv_ccalib -lopencv_dnn_objdetect -lopencv_dnn_superres -lopencv_dpm -lopencv_face -lopencv_freetype -lopencv_fuzzy -lopencv_hdf -lopencv_hfs -lopencv_img_hash -lopencv_intensity_transform -lopencv_line_descriptor -lopencv_mcc -lopencv_quality -lopencv_rapid -lopencv_reg -lopencv_rgbd -lopencv_saliency -lopencv_shape -lopencv_stereo -lopencv_structured_light -lopencv_phase_unwrapping -lopencv_superres -lopencv_optflow -lopencv_surface_matching -lopencv_tracking -lopencv_highgui -lopencv_datasets -lopencv_text -lopencv_plot -lopencv_ml -lopencv_videostab -lopencv_videoio -lopencv_viz -lopencv_wechat_qrcode -lopencv_ximgproc -lopencv_video -lopencv_xobjdetect -lopencv_objdetect -lopencv_calib3d -lopencv_imgcodecs -lopencv_features2d -lopencv_dnn -lopencv_flann -lopencv_xphoto -lopencv_photo -lopencv_imgproc -lopencv_core \
  && ./tmp/fig_05_ssim_artefatos \
  && test -f "tmp/fig_05_ssim_artefatos.png" \
  || echo "⚠ mm::show não gravou tmp/fig_05_ssim_artefatos.png"

In [ ]:
mm.show(mm.read("tmp/fig_05_ssim_artefatos.png"), figsize=(14, 14))

**Figure 5.29:** Análise espacial de degradação: imagens reconstruídas e respectivos mapas de erro absoluto normalizados para diferentes fatores de qualidade JPEG.


> ### 📝 5.12 Interprétation des cartes d'erreur
>
> Les cartes d'erreur présentées ont été **normalisées individuellement** (`cv2.NORM_MINMAX`) afin de maximiser le contraste visuel et de révéler la structure spatiale des distorsions. Cela signifie que :
>
> - Pour **Q=95**, l'erreur absolue est de l'ordre de **0,5 à 1,5 niveaux de gris** (imperceptible visuellement), mais la normalisation l'amplifie en noir et blanc pour mettre en évidence sa localisation au niveau des bords et des transitions.
> - Pour **Q=25**, l'erreur absolue est **10 à 20 fois plus grande** (5 à 15 niveaux de gris), mais la normalisation l'amène également dans la même plage [0, 255].
>
> Par conséquent, **l'intensité du blanc dans les cartes n'est PAS comparable entre différentes qualités** — les cartes servent uniquement à révéler la **signature spatiale** de l'erreur (bords vs blocs), et non son ampleur. L'ampleur correcte est fournie par les valeurs PSNR et SSIM, qui montrent clairement que Q=95 présente une erreur bien inférieure à celle de Q=25.

### Synthèse — Compression JPEG

Le processus de compression dans le standard JPEG repose sur l'application combinée de transformations spatiales, perceptuelles et statistiques pour réduire les redondances d'une image. La [Tableau 5.9](#tbl-05-sintese-jpeg) résume le rôle de chaque étape dans le *pipeline* et son impact respectif sur la réduction des données.

<a id="tbl-05-sintese-jpeg"></a>

**Tabela 5.9:** Synthèse des étapes du *pipeline* de compression JPEG et de leurs impacts respectifs.

| Étape | Opération Analytique | Mécanisme de Gain / Compression |
|:---|:---|:---|
| **Conversion $YC_bC_r$** | Isolation des canaux de luminance et de chrominance. | Modélise la perception du système visuel humain (SVH), permettant de traiter la couleur et la luminosité de manière indépendante. |
| **Sous-échantillonnage 4:2:0** | Réduction de la résolution spatiale des canaux de couleur ($C_b$ et $C_r$). | Élimine environ 50 % des données brutes avec un impact visuel minimal. |
| **DCT $8 \times 8$** | Mappage du domaine spatial vers le domaine des fréquences spatiales. | Compactage de l'énergie, concentrant l'information essentielle dans les premiers coefficients. |
| **Quantification Linéaire** | Division entière des coefficients par une matrice de pondération $Q(u,v)$. | Principale source de compression avec perte ; élimine les hautes fréquences imperceptibles. |
| **Codage Entropique** | Application d'algorithmes RLE et de codage de Huffman. | Compression statistique sans perte, optimisée par les longues séquences de coefficients nuls. |


#### Artefacts de dégradation caractéristiques

L'application de taux de compression excessivement agressifs (facteurs de qualité réduits) introduit des distorsions prévisibles dans l'image reconstruite, découlant des limitations mathématiques du modèle :

* **Artefacts de bloc (*blocking artifacts*) :** Discontinuités géométriques visibles aux frontières des blocs de $8 \times 8$ pixels, causées par la perte de corrélation spatiale après la quantification sévère des composantes AC.
* **Effet de sonnerie (*ringing*) :** Oscillations fantômes ou distorsions de « fumée » autour des bords nets et à fort contraste, provoquées par l'élimination brutale des harmoniques de haute fréquence nécessaires pour reconstruire les fonctions échelon.
* **Perte de texture fine :** Atténuation des détails à haute fréquence et à faible contraste (comme les pelouses, les tissus ou la porosité), donnant aux régions initialement texturées un aspect excessivement lisse ou homogénéisé.

## 5.13 Application Pratique : Débruitage par Filtrage Hybride

En réunissant les techniques consolidées tout au long de ce chapitre, on présente un *pipeline* complet de **restauration d'images** qui combine l'analyse spectrale dans le domaine fréquentiel avec le filtrage adaptatif dans le domaine spatial. L'objectif est d'atténuer un bruit mixte (composé d'une dégradation gaussienne et d'une interférence périodique) tout en préservant au maximum les détails structurels de l'image originale.

$$
\text{Image Bruitée} \xrightarrow{\text{FFT2}} \xrightarrow{\text{Filtre Notch Gaussien}} \xrightarrow{\text{IFFT2}} \xrightarrow{\text{Filtre Bilatéral}} \text{Image Restaurée}
$$

> ### 📝 Évaluation Complémentaire : PSNR vs. SSIM
>
> La paire de métriques statistiques PSNR et SSIM fournit une évaluation qualitative et morphologique complémentaire du processus de restauration :
>
> * **PSNR :** Pénalise uniformément l'écart quadratique moyen pixel par pixel.
> * **SSIM :** Évalue la préservation des structures locales perceptuellement pertinentes (luminance, contraste et contours).
>
> En pratique, il existe un compromis analytique (*trade-off*) entre **réduction du bruit** et **préservation des détails** : des filtres spatiaux excessivement agressifs atténuent bien le bruit haute fréquence, mais dégradent les textures fines et lissent les contours nets — ce qui **réduit simultanément** à la fois le PSNR et le SSIM par rapport à l'image originale. Le défi de la conception de filtres est de trouver le point d'équilibre qui maximise les deux métriques, garantissant une restauration fidèle et visuellement agréable.

### 5.13.1 Analyse des performances et conclusion du chapitre

Les résultats numériques et visuels générés par [Figure 5.30](#fig-05-pipeline-denoising) démontrent la pertinence pratique d’associer différents domaines de traitement. L’insertion simultanée de bruit périodique et stochastique corrompt les propriétés morphologiques du signal, réduisant sévèrement les indices de similarité et le rapport signal-bruit de l’image de référence.

L’isolation et la suppression des pics harmoniques dans le domaine fréquentiel au moyen du masque *notch* éliminent les franges d’interférence sinusoïdales réparties sur l’espace bidimensionnel. Comme le montrent les données imprimées de [Figure 5.30](#fig-05-pipeline-denoising), ce filtrage chirurgical induit un saut immédiat et substantiel de la métrique PSNR. Toutefois, le bruit gaussien haute fréquence reste actif de manière homogène dans le spectre, exigeant une approche complémentaire.

La restauration finale est consolidée dans le domaine spatial avec l’introduction du filtre bilatéral. Contrairement aux opérateurs passe-bas classiques (tels que le filtre gaussien ou le filtre moyenneur), qui lisseraient indifféremment le bruit et les contours structurels, le filtrage bilatéral calcule des poids pondérés par la proximité géométrique et par la différence d’intensité radiométrique. Ce comportement adaptatif atténue les fluctuations stochastiques résiduelles dans les régions de transition douce et préserve la netteté des bords spatiaux.

La convergence des deux approches aboutit à une **amélioration substantielle et simultanée** du PSNR et du SSIM par rapport à l’image bruitée — bien que les valeurs finales demeurent inférieures à celles de l’image originale (PSNR = $\infty$, SSIM = 1,0), en raison de la perte inévitable d’informations spectrales et texturales lors des processus de filtrage. L’atténuation douce (gaussienne) des pics dans le spectre évite les artefacts de *ringing*, tandis que le filtre bilatéral élimine le bruit stochastique résiduel sans compromettre la netteté des bords. Les résultats prouvent l’efficacité et la complémentarité pratique des outils d’analyse fréquentielle présentés dans ce chapitre, démontrant que le filtrage hybride (fréquence + spatial) est supérieur à toute approche isolée pour la restauration d’images dégradées par un bruit mixte.

In [ ]:
import numpy as np, cv2

# Cameraman (asset du chapitre), 256x256 — même image que la piste py.
img_gray = mm.gray(cv2.resize(mm.read("imagens/cameraman.png"), (256, 256)))
h_img, w_img = img_gray.shape

# ── 1. Bruit mixte : gaussien + périodique ───────────────────────────────────
np.random.seed(42)
X2, Y2 = np.meshgrid(np.arange(w_img), np.arange(h_img))
u0, v0       = 15, 10
ruido_gauss  = np.random.normal(0, 15, img_gray.shape)
ruido_period = 30 * np.sin(2 * np.pi * (u0 * X2 / w_img + v0 * Y2 / h_img))
img_noisy    = np.clip(img_gray.astype(float) + ruido_gauss + ruido_period, 0, 255).astype(np.uint8)

# ── 2. Spectre (log-magnitude) de l'image bruitée ──────────────────────────
mag_n = mm.spectrumMag(img_noisy)

# ── 3. Masque notch gaussien sur les 4 pics périodiques ──────────────────────
def suprimir_pico_gaussiano(mask, cy, cx, sigma=3.0):
    yy, xx = np.meshgrid(np.arange(mask.shape[0]), np.arange(mask.shape[1]), indexing="ij")
    notch = np.exp(-((yy - cy)**2 + (xx - cx)**2) / (2 * sigma**2))
    return mask * (1.0 - notch)

cy0, cx0 = h_img // 2, w_img // 2
mascara_notch = np.ones((h_img, w_img), dtype=np.float64)
for dy, dx in [(v0, u0), (-v0, -u0), (v0, -u0), (-v0, u0)]:
    mascara_notch = suprimir_pico_gaussiano(mascara_notch, cy0 + dy, cx0 + dx, 3.0)

# ── 4. Filtrage : notch dans le domaine fréquentiel + bilatéral ───────────────
img_notch = mm.freqFilter(img_noisy, mascara_notch)
img_den   = cv2.bilateralFilter(img_notch, 7, 25, 7)
mascara_vis = (mascara_notch * 255).astype(np.uint8)

psnr_n  = mm.psnr(img_gray, img_noisy)
psnr_no = mm.psnr(img_gray, img_notch)
psnr_d  = mm.psnr(img_gray, img_den)
print(f"Bruitée : PSNR={psnr_n:.2f} dB | Après notch : {psnr_no:.2f} dB | Notch+bilatéral : {psnr_d:.2f} dB")

mm.show(
    [img_gray, img_noisy, mag_n, mascara_vis, img_notch, img_den],
    titles=[
        "Originale",
        f"Bruitée (PSNR={psnr_n:.1f} dB)",
        "Spectre (pics visibles)",
        "Masque notch (gaussien)",
        f"Après notch (PSNR={psnr_no:.1f} dB)",
        f"Notch + bilatéral (PSNR={psnr_d:.1f} dB)",
    ],
    cols=6, figsize=(20, 4)
)


**Figure 5.30:** *Pipeline* complet de suppression de bruit mixte : (1) ajout de bruit gaussien et périodique ; (2) identification des pics d


## 5.14 Résumé du Chapitre

La transition du **domaine spatial** vers le **domaine fréquentiel** révèle la distribution spectrale de l'énergie de l'image, établissant ainsi la base analytique pour le filtrage avancé, la restauration et la compression des données. L'articulation structurelle de ces concepts est synthétisée dans la carte conceptuelle de la [Figure 5.31](#fig-05-mapa-conceitual).

<figure id="fig-05-mapa-conceitual" style="text-align:center; margin:1em 0;">
  <img src="imagens/fig-05-mapa-conceitual.png" alt="" style="max-width:60%; display:block; margin:auto;" />
  <figcaption><strong>Figure 5.31:</strong> Carte conceptuelle des transformations et propriétés dans le domaine fréquentiel.</figcaption>
</figure>

### Fundamentos Essenciais

* **TFD et Perception Visuelle :** Le spectre décompose l’image en composantes harmoniques. La **phase** conserve l’intelligibilité géométrique de la scène et la localisation des contours, tandis que la **magnitude** détermine la distribution du contraste et les amplitudes globales.
* **Efficacité Algorithmique :** Le Théorème de Convolution permet le traitement de masques à grande échelle dans le domaine fréquentiel via la FFT, réduisant la complexité computationnelle asymptotique de $O(N^2 K^2)$ dans l’espace à $O(N^2 \log N)$.
* **Phénomène de *Ringing* :** Les coupures abruptes dans le spectre (filtres idéaux) génèrent des oscillations spatiales indésirables (phénomène de Gibbs). L’atténuation douce par des filtres de **Butterworth** ou **gaussiens** élimine ces discontinuités.
* **Analyse Multirésolution via *Wavelets* :** Dépassant le caractère purement global de Fourier, la DWT capture simultanément la fréquence et la localisation spatiale, fondant le standard JPEG 2000 et appuyant des représentations hiérarchiques analogues aux extractions de caractéristiques dans les Réseaux de Neurones Convolutifs (CNN).
* **Compression Perceptuelle (DCT) :** Le *pipeline* JPEG exploite les limitations de contraste du système visuel humain aux hautes fréquences spatiales. La DCT isole l’énergie de blocs $8 \times 8$, permettant à la quantification d’éliminer les coefficients AC des détails fins sans préjudice perceptuel sévère.

**Prochaines Étapes :** Le **Chapitre 6** inaugure la Partie II de l’ouvrage, appliquant les outils de traitement d’images à la résolution de problèmes réels d’inspection industrielle. Seront explorées des techniques de **segmentation et d’analyse de formes** pour la détection automatique de défauts sur les lignes de production — depuis l’identification de défauts superficiels sur les pièces jusqu’à la lecture *QRCode* sur les épreuves, consolidant le pont entre la théorie présentée dans la Partie I et les exigences pratiques de la vision par ordinateur.

## 5.15 🤖 Utilisation de Gemini Notebook comme Tuteur Complémentaire

Dans cette édition, l'utilisation de la plateforme **Gemini Notebook** est encouragée comme outil d'apprentissage complémentaire — **et non comme substitut** de la lecture attentive, de la résolution d'exercices ou de l'expérimentation pratique. Fondé sur des architectures d'intelligence artificielle, le système utilise exclusivement le matériel pédagogique et les documents fournis par l'auteur comme base de connaissances, garantissant que les réponses générées soient conceptuellement alignées sur le contenu programmatique et l'approche pédagogique adoptée tout au long de cet ouvrage.

> ### ❗ Accès au Tuteur Intelligent
>
> [🚀 ACCÉDER À Gemini Notebook : CHAPITRE 05](https://notebooklm.google.com/notebook/b8b6cd26-ef65-4a10-b7e7-e072d4870ddb)
>
> #### 🌐 Langue et Langage de Programmation
>
> Le projet de ce chapitre dans Gemini Notebook a été construit uniquement avec le texte en **portugais** et les exemples de code en **Python**. Si vous étudiez à partir de l'édition en anglais ou en français, ou si vous suivez le parcours en C++, les réponses du tuteur peuvent ne pas correspondre exactement à la version que vous lisez.
>
> #### Directives concernant le Contenu Généré par Intelligence Artificielle
>
> Bien que les outils d'intelligence artificielle constituent des alliés efficaces dans le processus d'apprentissage et de révision, le contenu généré est sujet à des incohérences ou à des imprécisions techniques. Par conséquent, la consultation systématique de manuels, d'articles scientifiques et de sources académiques indexées est indispensable pour une validation rigoureuse des informations. Il est vivement recommandé d'exécuter et de modifier les exemples pratiques en Python fournis dans ce chapitre comme méthode principale de vérification expérimentale des résultats.

## 5.16 Liste d'exercices

1. **(10 %) Implémentation directe de la TFD 2D :** Implémentez analytiquement la Transformée de Fourier Discrète 2D (TFD) sans l'aide de fonctions natives de bibliothèques (comme `np.fft.fft2`), en utilisant strictement la formulation mathématique définie dans la [Équation 5.1](#eq-05-dft) pour une matrice de dimensions $16 \times 16$. Effectuez la validation numérique en comparant les coefficients générés avec les résultats de la fonction `np.fft.fft2`, en vous assurant que l'écart absolu maximal soit inférieur à $10^{-8}$. Mesurez les temps d'exécution des deux méthodes et présentez une justification théorique pour la disparité observée en termes de complexité asymptotique.

2. **(15 %) Suppression du bruit périodique :** Ajoutez des interférences sinusoïdales avec des fréquences spatiales $(u_0, v_0) \in \{(5,10), (20,5), (30,30)\}$ à l'image de test du *Cameraman*. Pour chaque scénario de dégradation, concevez un masque de filtrage *notch* spécifique dans le domaine fréquentiel afin d'isoler et d'atténuer les pics harmoniques indésirables. Évaluez quantitativement l'efficacité du processus de restauration par le calcul des métriques PSNR et SSIM. Discutez analytiquement du compromis entre l'atténuation du bruit sinusoïdal et l'atténuation indésirable des caractéristiques structurelles légitimes de l'image.

3. **(15 %) Analyse comparative des opérateurs passe-bas :** Réalisez une étude comparative entre les filtres passe-bas idéal, gaussien et de Butterworth (avec des ordres harmoniques $n = 1, 2, 4$), paramétrés avec des fréquences de coupure $D_0 = 20, 40, 60$ pixels. Pour chaque combinaison structurelle, calculez les indices PSNR et SSIM de l'image résultante par rapport au signal original de référence. Organisez les données quantitatives dans un tableau structuré et tracez les graphiques unidimensionnels des fonctions de transfert correspondantes le long du profil horizontal $H(u, 0)$.

4. **(15 %) Banque de filtres multirésolution de Haar :** Développez un script pour exécuter manuellement la décomposition *wavelet* discrète 2D de premier niveau en utilisant la famille de Haar. L'algorithme doit calculer les coefficients des filtres correspondants passe-bas ($h$) et passe-haut ($g$), en les appliquant de manière séparable sur les lignes et les colonnes de la matrice, suivis de l'opération de décimation (sous-échantillonnage spatial par un facteur de 2). Validez numériquement l'exactitude de votre implémentation en confrontant les sous-bandes obtenues avec la sortie de la fonction `pywt.dwt2(img, 'haar')`.

5. **(15 %) Compression éparse par seuillage wavelet :** Appliquez la technique de filtrage par seuillage abrupt (*hard thresholding*) sur les coefficients de détail de la décomposition *wavelet*, en adoptant les seuils numériques $T \in \{5, 10, 20, 40, 80\}$ pour les familles de Haar, Daubechies (`db4`) et Symlets (`sym4`). Après avoir réalisé le processus de synthèse au moyen de la transformée inverse (`pywt.waverec2`), calculez les valeurs de PSNR et SSIM de chaque image reconstruite. Identifiez et justifiez quelle combinaison de famille *wavelet* et de seuil $T$ maximise la similarité structurelle.

6. **(15 %) Construction d'un encodeur JPEG simplifié :** Implémentez le pipeline complet de compression de données simulant la norme JPEG. Le flux doit englober : la conversion spatiale $RGB \rightarrow YC_bC_r$, le sous-échantillonnage chromatique dans la proportion 4:2:0, la segmentation de la luminance en blocs disjoints de $8 \times 8$ pixels, l'application de la DCT-II 2D orthogonale, et la quantification linéaire basée sur la matrice normalisée de luminance mise à l'échelle par des facteurs de qualité souhaités. Réalisez le décodage inverse et comparez quantitativement les reconstructions avec les fichiers générés par la fonction `cv2.imencode` pour les facteurs de qualité de 20, 50 et 80.

7. **(15 %) Analyse perceptuelle sur des contenus hétérogènes :** Développez une image synthétique composée de trois régions distinctes et aux caractéristiques spectrales contrastées : une texture photographique complexe (représentant de hautes fréquences stochastiques), une zone de texte vectorisé avec des bords nets (représentant des transitions en échelon pures) et un gradient linéaire continu (représentant de basses fréquences homogènes). Soumettez cette image mixte aux processus de compression sous les formats JPEG, PNG et WebP. Évaluez et interprétez les résultats en corrélant la taille finale du fichier sur disque avec les métriques PSNR et SSIM obtenues, en justifiant quel format présente les meilleures performances pour des signaux de nature hétérogène et pourquoi cet avantage survient en termes de compactage d'énergie et de préservation perceptuelle.

## Références du chapitre

Le fondement théorique et le développement analytique des concepts abordés dans ce chapitre s'appuient sur les ouvrages de référence suivants :

* **Gonzalez (2018)** — Formulations classiques des Transformées de Fourier Discrètes 2D (TFD), conception de filtres analytiques dans le domaine fréquentiel, Transformée en Cosinus Discrète (TCD) et principes fondamentaux des systèmes de compression d'images.
* **Oppenheim (2010)** — Théorie formelle des signaux et des systèmes appliqués dans le domaine discret, couvrant les propriétés mathématiques de la TFD et la modélisation analytique du Théorème de Convolution.
* **Mallat (1999)** — Fondements mathématiques de la théorie des *ondelettes*, formalisation de l'analyse multirésolution (AMR) et architecture des bancs de filtres dyadiques.
* **Wallace (1991)** — Spécification originale et aspects techniques du standard de compression ISO/CEI JPEG, avec un accent sur les critères psychovisuels pour la conception des matrices de quantification TCD.
* **Szeliski (2022)** — Modélisation computationnelle et caractérisation des métriques modernes de fidélité et de qualité perceptuelle (PSNR et SSIM), ainsi que l'analyse comparative des formats d'images tramées à hautes performances.

## Références du Chapitre


GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

MALLAT, St{\'e}phane. **A wavelet tour of signal processing**. Elsevier, 1999.

OPPENHEIM, Alan V.; SCHAFER, Ronald W. **Discrete-Time Signal Processing**. Pearson, 2010.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Springer, 2022.

WALLACE, Gregory K. **The {JPEG} Still Picture Compression Standard**. 1991.